# Solver: baseline vs optimized schedule, step by step

This notebook runs the ThermaSched solver one step at a time, with the inputs and results
shown as tables and charts along the way:

1. load a scenario from `scenarios/`, or first build a new one from an existing one with
   some settings or constants changed;
2. build the **baseline** schedule (a simple placement rule at a fixed supply temperature);
3. build and solve the **optimized** MILP with HiGHS;
4. score both schedules with the same **evaluator** and run the checks;
5. compare the two and write the results to `results/`, where the dashboard can open them.

The notebook is self-contained: section 0 holds a copy of the model's code (the `scheduler`
package of the repository, and the parts of the dashboard it needs). It runs the same code
as the command line (`uv run python -m scheduler.run`), so with the same settings the
numbers are identical. `scheduler/README.md` in the repository explains the options, and
`docs/PROJECT_CONTEXT.md` has the full equations.

**What it needs.** A project folder with `scenarios/` in it (from the repository, or written
by `data_prep.ipynb`), and `data/clean/` to build new scenarios. The run is written to
`results/` in that folder, and a zip of it to `exports/`, ready for the dashboard.

**How to run.**

- **Google Colab:** upload this notebook, and put the project folder on Google Drive (by
  default `My Drive/thermasched/`, with `scenarios/` and `data/clean/` inside; set
  `COLAB_DRIVE_FOLDER` below if it is elsewhere). Press *Runtime › Run all*; Colab asks for
  permission to mount Drive, and HiGHS (the solver) is installed on the first run.
- **Locally:** after `uv sync`, open this notebook with the `.venv` kernel and press
  **Run All**. The repository is the project folder.

Run the cells from top to bottom. To try another run, change the Settings cell and run
everything again.

**Run time.** Loading, the baseline and the evaluation take a few seconds. The solve takes up
to `time_limit_s`. With `reassignment="none"`, every scenario solves in seconds. With `"free"`
or `"same_zone"`, the solver usually runs until the time limit.

## Settings

**Scenario.** `SCENARIO` is the folder under `scenarios/` to solve. With `BUILD_FROM = None` it
is solved as it is. With `BUILD_FROM` set to an existing scenario, `SCENARIO` is first built
from that one's settings, constants and cold-plate table with the values in the three
`..._CHANGES` dicts changed, and written to `scenarios/<SCENARIO>/` (section 1). Each dict
holds only the keys to change; any key of the matching block of `params.json` works
(`settings`, `constants_as_printed`, `cold_plate`), and an unknown key is an error. The
build reads only `data/clean/`, not the raw data. Running the notebook again rebuilds the
folder; the four shipped scenarios (`small`, `small_running`, `base`, `thesis10`) are never
overwritten, and `small_running` cannot be a starting point because it was edited by hand.
A built scenario is an ordinary scenario folder: the command line
(`--scenario <name>`) and the dashboard can use it too.

**Run.** Every switch of a run is a field of `RunSettings`, with the same meaning as the command-line
option of the same name (the table in [`scheduler/README.md`](../scheduler/README.md#3-run-options)).
Everything is set to the command-line defaults. With them, the solve on `small` runs for the
full 10-minute time limit. For a quick look, set `reassignment="none"` (a few seconds). Don't
just shorten `time_limit_s`: with `"free"`, 120 s on `small` ends still on the starting
schedule, so no workload moves.

The results go to `results/<scenario>/nb_<run name>/`. The `nb_` prefix stops a notebook run
from overwriting a command-line run with the same switches. A zip of the run goes to
`exports/` (section 7): drop it on the dashboard's *Upload results* page, or unzip it into
the repository's `results/` folder and commit it to keep it in the deployed dashboard.

In [ ]:
from pathlib import Path

# Project folder: the one that holds scenarios/ (and data/clean/ to build scenarios); results/
# and exports/ are written into it. On Colab it is a folder on Google Drive, under "My Drive".
# Locally it is the repository, found on its own when Jupyter starts in it or in notebooks/.
COLAB_DRIVE_FOLDER = "thermasched"
LOCAL_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Scenario to solve: a folder under scenarios/
SCENARIO = "small"
# None: solve SCENARIO as it is. A scenario name (e.g. "small"): build SCENARIO from it first,
# with the values below changed, and write it to scenarios/<SCENARIO>/.
BUILD_FROM = None
# Scenario construction. The values shown are small's; uncomment and edit the ones to change.
SCENARIO_CHANGES = {
    # "n_servers": 3,                 # servers, picked across the range of dynamic power
    # "freq_mhz": [2100.0, 2400.0, 3000.0],  # speed tiers, MHz
    # "freq_split": [1, 1, 1],        # servers per tier, slowest first; must add up to n_servers
    # "n_zones": 2,                   # servers are dealt to the zones in turn
    # "periods": 96,                  # horizon, in periods of 5 minutes
    # "start_hour_local": 12,         # clock hour of period 1
    # "target_peak_load": 0.60,       # workloads are added until the peak requested vCPUs reach
    #                                 #   this share of fleet cores (or the catalogue runs out)
    # "min_et_periods": 3,            # shortest baseline execution time admitted, periods
    # "max_et_periods": 36,           # longest baseline execution time admitted, periods
    # "rt_share": 0.30,               # share of workloads that are time-sensitive (RT_i = 1)
    # "rt_slack_frac": 0.25,          # their slack: this fraction of ET_base, at least 1 period
    # "dt_slack_mult": 2.0,           # delay-tolerant slack: mult x ET_base, clamped to
    # "dt_slack_min": 12,             #   [dt_slack_min,
    # "dt_slack_max": 36,             #    dt_slack_max] periods
    # "ci_mean_g_per_kwh": 621.73,    # carbon intensity: daily mean, g/kWh
    # "ci_amplitude": 0.25,           # its daily swing as a share of the mean; 0 is flat
    # "seed": 42,                     # workload draw and time-sensitive flags
}
# Model constants (params.json "constants_as_printed"). For H_min, the H_min_C run setting
# below also works without a new scenario.
CONSTANT_CHANGES = {
    # "T_max_server_C": 90.0,         # server temperature limit; the cap is T_max - H_min
    # "H_min_C": 10.0,                # minimum thermal headroom
    # "T_dvfs_C": 75.0,               # throttling starts here
    # "T_crac_min_C": 18.0,           # CRAC supply air range, whole degrees are used
    # "T_crac_max_C": 27.0,
    # "Q_crac_kW": 116.0,             # CRAC cooling capacity per zone
    # "T_max_zone_C": 32.0,           # zone air temperature limit
    # "Q_cp_kW": 0.3,                 # heat one cold plate can remove
}
# Cold-plate table (params.json "cold_plate"): flow settings and their thermal resistance.
COLD_PLATE_CHANGES = {
    # "flow_settings_Lmin": [0.10, 0.15, 0.20, 0.25, 0.30],
    # "R_C_per_W": [0.1543, 0.1143, 0.0943, 0.0843, 0.0793],
}

# The run: every field of RunSettings, the same switches as the command line.
RUN = dict(
    scenario=SCENARIO,
    # baseline
    baseline_rule="least_loaded",     # least_loaded, fastest_first or random
    seed=42,                          # for the random rule
    baseline_supply_temp_C=18.0,      # the baseline's fixed CRAC supply air temperature, degC
    # optimized model
    reassignment="free",              # free (any server), same_zone, or none (baseline server)
    supply_temp_mode="optimize",      # optimize (per zone and period) or fixed
    fixed_supply_temp_C=None,         # supply temperature when fixed; None: the baseline's
    H_min_C=None,                     # minimum thermal headroom, degC; None: the scenario's (10)
    time_limit_s=600.0,               # solver time limit, s
    mip_rel_gap=1e-3,                 # relative MIP gap at which the solver stops
    # cold plates
    cold_plates=False,                # model cold plates (plates per server from servers.csv)
    cold_plate_count=None,            # put this many plates on every server (needs cold_plates=True)
    baseline_cp_flow_Lmin=0.30,       # the baseline's fixed cold-plate flow, L/min
    baseline_cp_coolant_C=20.0,       # the baseline's fixed coolant temperature, degC
    cp_mode="optimize",               # optimize, or fixed (holds the settings below)
    fixed_cp_flow_Lmin=None,          # None: the baseline's
    fixed_cp_coolant_C=None,          # None: the baseline's
    # priority, Eq. (56)-(57)
    priority=True,
    # scenario overrides
    zone_map=None,                    # zone of each server in order, e.g. (1, 1, 1, 2, 2, 2)
    carbon_file=None,                 # CSV with columns t, CI_t_g_per_kWh
    ci_flat=False,                    # carbon intensity held at the scenario's mean
)

RUN_NAME = None                  # None: "nb_" + the command line's name for these switches
WARM_START = True                # offer the solver a starting schedule (command line: --no-warm-start)
DIAGNOSIS_TIME_LIMIT_S = 120.0   # seconds per relaxed re-solve if the model is infeasible
SOLVER_LOG = False               # HiGHS log; it prints in the terminal running Jupyter, not here
WRITE_RESULTS = True             # False: show everything here but write nothing to results/
DOWNLOAD_ZIP = True              # on Colab, download the run's zip at the end

## 0. Setup

Finds the project folder (on Colab, mounts Google Drive), installs HiGHS if it is missing,
and writes the model code in the cells below to a temporary folder `LIB`, from which the rest
of the notebook imports it. There is nothing to change here.

The code cells of this section are generated from the repository's `scheduler/` and
`dashboard/` packages by `tools/sync_notebooks.py`. To change the model, edit the package and
run that script; edits made here are overwritten. In Colab or Jupyter, collapse this section
to hide them.

In [ ]:
import importlib.util
import shutil
import subprocess
import sys
import tempfile

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
else:
    ROOT = LOCAL_ROOT
SCENARIOS_DIR = ROOT / "scenarios"
CLEAN_DIR = ROOT / "data" / "clean"            # the cleaned tables a scenario is built from
RESULTS_DIR = ROOT / "results"
EXPORTS_DIR = ROOT / "exports"                   # zips of runs, for the dashboard
if not SCENARIOS_DIR.is_dir():
    raise FileNotFoundError(f"no scenarios/ folder in {ROOT}; see 'What it needs' at the top")

# HiGHS solves the MILP. numpy, pandas and plotly come with Colab and with `uv sync`.
if importlib.util.find_spec("highspy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "highspy"], check=True)

# A fresh copy of the model code, imported from LIB ahead of any other copy.
LIB = Path(tempfile.gettempdir()) / "thermasched_notebook"
shutil.rmtree(LIB, ignore_errors=True)
for package in ("scheduler", "dashboard"):
    (LIB / package).mkdir(parents=True)
    (LIB / package / "__init__.py").write_text("")
    for name in [m for m in sys.modules if m == package or m.startswith(package + ".")]:
        del sys.modules[name]
if str(LIB) in sys.path:
    sys.path.remove(str(LIB))
sys.path.insert(0, str(LIB))
print(f"Project folder: {ROOT}\nModel code written to: {LIB}")

### Model code

Generated from `scheduler/` and `dashboard/`; see above.

In [ ]:
%%writefile {LIB}/scheduler/types.py
"""Shared data structures for the thermal-aware scheduling prototype.

This file is the contract between the loader, the baseline, the evaluator, the
optimizer, and the report writer. Equation numbers refer to docs/PROJECT_CONTEXT.md.

Conventions (all modules):
- Periods are numbered 1..T as in the scenario files. Arrays over periods have length T
  and are indexed by ``t - 1``.
- Servers are numbered 1..S, zones 1..Z, workloads 1..I, all contiguous. Arrays over
  servers are indexed by ``s - 1`` and over zones by ``z - 1``.
- A workload that starts in period ``start`` and runs ``n`` periods occupies periods
  ``start, ..., start + n - 1`` and has ``end = start + n`` (exclusive, the same as
  ``end`` in ``baseline_schedule.csv``). It is on time when ``end <= D_i``.
- ``T[s, t]`` is the server temperature at the end of period t, produced by the power
  drawn during period t. ``DPF[s, t]`` is taken from ``T[s, t]`` of the same period.
- Units: power in W, ``Q_crac`` and ``P_crac`` in kW, ``dt`` in seconds, energy in kWh,
  carbon intensity in g/kWh, carbon in g, work in MHz*s, temperature in degC.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, field

import numpy as np

BASELINE_RULES = ("least_loaded", "fastest_first", "random")
REASSIGNMENT_MODES = ("free", "same_zone", "none")
SUPPLY_TEMP_MODES = ("optimize", "fixed")
COLD_PLATE_MODES = ("optimize", "fixed")

# Constraint families that the infeasibility diagnosis relaxes one at a time.
RELAX_DEADLINES = "deadlines_and_waiting"   # (27), (28): windows widened to the horizon
RELAX_THERMAL = "thermal_cap"               # (31)-(33): T <= T_max - H_min dropped
RELAX_CAPACITY = "capacity"                 # (12), (13)
RELAX_COOLING = "cooling_capacity"          # (54): Q_crac <= Q_crac_kW
RELAX_PRIORITY = "priority"                 # (56), (57): start order dropped
RELAX_FAMILIES = (RELAX_DEADLINES, RELAX_THERMAL, RELAX_CAPACITY, RELAX_COOLING, RELAX_PRIORITY)

# Solver outcomes reported by the optimizer.
STATUS_OPTIMAL = "optimal"            # solved to the requested gap
STATUS_TIME_LIMIT = "time_limit"      # stopped on the time limit with a valid schedule
STATUS_INFEASIBLE = "infeasible"      # proven: no valid schedule exists
STATUS_NO_SOLUTION = "no_solution"    # stopped on the time limit with no schedule found


class NotDefinedYet(NotImplementedError):
    """Raised when a switch asks for a term the thesis does not define yet."""


@dataclass(frozen=True)
class Workload:
    i: int
    workload_id: str
    flavor: str
    A: int                # first period it may run
    D: int                # period by which it must be finished (end <= D)
    AD: int               # maximum allowable delay in periods, Eq. (22)
    ET_base: int          # baseline execution time in periods at f_ref
    RT: int               # 1 time-sensitive, 0 delay-tolerant
    B: int                # 1 already started: stays on its original server
    X0: int | None        # server in the scenario's own least-loaded baseline
    C_req: int            # cores
    M_req_MB: float
    W_MHz_s: float        # work required
    usage_imputed: int = 0


@dataclass(frozen=True)
class Server:
    s: int
    name: str
    zone: int
    f_MHz: float
    C_cores: int
    M_MB: float
    P_idle_W: float
    P_max_W: float
    T0_C: float
    CT_cold_plate: int    # CT_s: 1 if the server has cold-plate cooling
    th_a_C: float         # a_s
    th_R_C_per_W: float   # R_s
    th_alpha: float       # alpha_s per period
    n_cold_plates: int | None = None   # editable count; None: CT_cold_plate (0 or 1)

    @property
    def plates(self) -> int:
        """Cold plates installed on the server."""
        return self.CT_cold_plate if self.n_cold_plates is None else self.n_cold_plates


@dataclass(frozen=True)
class Constants:
    """Constants of docs/PROJECT_CONTEXT.md section 4.1 (params.json, constants_as_printed)."""
    R_dvfs: float
    F_throttle: float
    T_max_server_C: float
    H_min_C: float
    T_dvfs_C: float
    T_throttle_C: float
    Q_crac_kW: float
    COP_coeffs: tuple[float, float, float]   # COP(T) = c0*T^2 + c1*T + c2, T in degC
    T_crac_min_C: float
    T_crac_max_C: float
    T_max_zone_C: float
    ramp_short_lag: int = 3       # Eq. (35): 15 minutes
    ramp_short_C: float = 5.0
    ramp_long_lag: int = 12       # Eq. (36): one hour (decided, section 10)
    ramp_long_C: float = 20.0


@dataclass(frozen=True)
class ColdPlateSpec:
    """Cold-plate model agreed with the thesis group (docs/PROJECT_CONTEXT.md section 10).

    Every plate is identical and has its own pump. Per plate and period the heat removed
    is the least of: Q_max_per_plate_W, the server's heat P_IT, and
    G_k * (T[s,t-1] - coolant), where G_k = min(1/R_k, flow_k * density * c_pw) for the
    chosen flow setting k. Pump power is pump_ref_W * (flow_k / pump_ref_flow_Lmin)^3, Eq. (48).
    """
    # Wang et al. (2026), Fig. 10(a), CPU 1: R = (T_cpu,max - T_in) / P_cpu at 200 W
    flow_settings_Lmin: tuple[float, ...] = (0.10, 0.15, 0.20, 0.25, 0.30)
    R_C_per_W: tuple[float, ...] = (0.1543, 0.1143, 0.0943, 0.0843, 0.0793)
    flow_min_Lmin: float = 0.05          # Eq. (47); the table's printed 0.005 is a typo
    flow_max_Lmin: float = 0.35
    Q_max_per_plate_W: float = 300.0     # Eq. (55), 0.3 kW per cold plate
    coolant_min_C: float = 20.0          # Eq. (49), 293.15 K
    coolant_max_C: float = 39.95         # 313.1 K as printed
    pump_ref_W: float = 0.657e-3         # Wang and Tao (2025): 0.657 mW at 0.264 L/min
    pump_ref_flow_Lmin: float = 0.264
    c_pw_J_per_kgC: float = 4182.0       # printed with a "kJ" label, read as J
    water_density_kg_per_L: float = 1.0

    def validate(self) -> None:
        n = len(self.flow_settings_Lmin)
        if n == 0 or len(self.R_C_per_W) != n:
            raise ValueError("cold plate: flow_settings_Lmin and R_C_per_W must be non-empty "
                             "and of equal length")
        if any(r <= 0 for r in self.R_C_per_W):
            raise ValueError("cold plate: every R_C_per_W must be positive")
        if any(not self.flow_min_Lmin - 1e-9 <= f <= self.flow_max_Lmin + 1e-9
               for f in self.flow_settings_Lmin):
            raise ValueError(f"cold plate: flow settings {self.flow_settings_Lmin} must lie in "
                             f"[{self.flow_min_Lmin:g}, {self.flow_max_Lmin:g}] L/min, Eq. (47)")
        if self.coolant_min_C > self.coolant_max_C:
            raise ValueError("cold plate: coolant_min_C exceeds coolant_max_C")

    def conductance_W_per_C(self) -> np.ndarray:
        """Per plate and flow setting: min(1/R, water heat capacity rate), W per degC."""
        flow = np.asarray(self.flow_settings_Lmin, dtype=float)
        water = flow / 60.0 * self.water_density_kg_per_L * self.c_pw_J_per_kgC   # Eq. (53)
        return np.minimum(1.0 / np.asarray(self.R_C_per_W, dtype=float), water)

    def pump_W(self) -> np.ndarray:
        """Per plate and flow setting, Eq. (48)."""
        flow = np.asarray(self.flow_settings_Lmin, dtype=float)
        return self.pump_ref_W * (flow / self.pump_ref_flow_Lmin) ** 3

    def level_of(self, flow_Lmin: float) -> int:
        """Index of a flow setting; raises if the flow is not one of the settings."""
        for k, f in enumerate(self.flow_settings_Lmin):
            if abs(f - flow_Lmin) < 1e-9:
                return k
        raise ValueError(f"cold-plate flow {flow_Lmin:g} L/min is not one of the settings "
                         f"{self.flow_settings_Lmin}")


@dataclass(frozen=True)
class ColdPlateOps:
    """Cold-plate operating decisions, shape (S, T). NaN for servers without cold plates."""
    flow_Lmin: np.ndarray
    coolant_C: np.ndarray


@dataclass(frozen=True)
class Scenario:
    name: str
    T: int
    dt_seconds: float
    workloads: tuple[Workload, ...]   # sorted by i; workloads[i - 1].i == i
    servers: tuple[Server, ...]       # sorted by s; servers[s - 1].s == s
    n_zones: int
    CI_g_per_kWh: np.ndarray          # shape (T,)
    constants: Constants
    cold_plate: ColdPlateSpec = field(default_factory=ColdPlateSpec)
    AV: np.ndarray | None = None      # (S, T) 0/1 availability, Eq. (11); None: all 1

    @property
    def I(self) -> int:
        return len(self.workloads)

    @property
    def S(self) -> int:
        return len(self.servers)

    @property
    def Z(self) -> int:
        return self.n_zones

    @property
    def dt_hours(self) -> float:
        return self.dt_seconds / 3600.0

    def servers_in_zone(self, z: int) -> tuple[Server, ...]:
        return tuple(sv for sv in self.servers if sv.zone == z)

    def availability(self) -> np.ndarray:
        """AV[s, t] as a (S, T) 0/1 int array, Eq. (11)."""
        if self.AV is None:
            return np.ones((self.S, self.T), dtype=int)
        return np.asarray(self.AV, dtype=int)


@dataclass
class RunSettings:
    """Every switch of a run. Written verbatim to summary.json."""
    scenario: str = "small"
    run_name: str = "default"
    # baseline
    baseline_rule: str = "least_loaded"
    seed: int = 42
    baseline_supply_temp_C: float = 18.0
    # optimized model
    reassignment: str = "free"
    supply_temp_mode: str = "optimize"
    fixed_supply_temp_C: float | None = None   # None: use baseline_supply_temp_C
    H_min_C: float | None = None               # None: use the scenario constant
    time_limit_s: float = 600.0
    mip_rel_gap: float = 1e-3
    # cold plates (off: Q_cp and pump power are zero everywhere)
    cold_plates: bool = False
    cold_plate_count: int | None = None        # plates on every server; None: servers.csv
    baseline_cp_flow_Lmin: float = 0.30        # baseline runs its cold plates fixed at these
    baseline_cp_coolant_C: float = 20.0
    cp_mode: str = "optimize"                  # "fixed": the optimized run holds them too
    fixed_cp_flow_Lmin: float | None = None    # None: use baseline_cp_flow_Lmin
    fixed_cp_coolant_C: float | None = None    # None: use baseline_cp_coolant_C
    # priority, Eq. (56)-(57)
    priority: bool = True
    # scenario overrides (applied by io.apply_overrides)
    zone_map: tuple[int, ...] | None = None    # zone per server, replaces servers.csv
    carbon_file: str | None = None             # CSV with t, CI_t_g_per_kWh
    ci_flat: bool = False                      # CI_t held at the scenario's mean
    # slot that is not defined yet
    fan_power: bool = False

    def validate(self) -> None:
        if self.baseline_rule not in BASELINE_RULES:
            raise ValueError(f"baseline_rule must be one of {BASELINE_RULES}")
        if self.reassignment not in REASSIGNMENT_MODES:
            raise ValueError(f"reassignment must be one of {REASSIGNMENT_MODES}")
        if self.supply_temp_mode not in SUPPLY_TEMP_MODES:
            raise ValueError(f"supply_temp_mode must be one of {SUPPLY_TEMP_MODES}")
        if self.cp_mode not in COLD_PLATE_MODES:
            raise ValueError(f"cp_mode must be one of {COLD_PLATE_MODES}")
        if self.carbon_file is not None and self.ci_flat:
            raise ValueError("carbon_file and ci_flat cannot both be set")
        if self.zone_map is not None and any(int(z) < 1 for z in self.zone_map):
            raise ValueError("zone_map: zones are numbered from 1")
        if self.cold_plate_count is not None:
            if not self.cold_plates:
                raise ValueError("cold_plate_count is set but cold_plates is off")
            if self.cold_plate_count < 0:
                raise ValueError("cold_plate_count must be 0 or more")
        if self.fan_power:
            raise NotDefinedYet(
                "fan_power: not defined yet. The model document has no fan-power "
                "equation; airflow is not modelled.")

    def h_min(self, scenario: Scenario) -> float:
        return scenario.constants.H_min_C if self.H_min_C is None else self.H_min_C

    def thermal_cap_C(self, scenario: Scenario) -> float:
        """Highest allowed server temperature: T_max - H_min, Eq. (31)-(33)."""
        return scenario.constants.T_max_server_C - self.h_min(scenario)

    def optimized_fixed_supply_temp_C(self) -> float:
        if self.fixed_supply_temp_C is None:
            return self.baseline_supply_temp_C
        return self.fixed_supply_temp_C

    def optimized_fixed_cp(self) -> tuple[float, float]:
        """(flow L/min, coolant degC) held by the optimized run when cp_mode is fixed."""
        flow = self.baseline_cp_flow_Lmin if self.fixed_cp_flow_Lmin is None else self.fixed_cp_flow_Lmin
        cool = (self.baseline_cp_coolant_C if self.fixed_cp_coolant_C is None
                else self.fixed_cp_coolant_C)
        return float(flow), float(cool)

    def as_dict(self) -> dict:
        return asdict(self)


def cold_plate_counts(scenario: Scenario, settings: RunSettings) -> np.ndarray:
    """Cold plates per server, shape (S,): zero everywhere when cold plates are off."""
    if not settings.cold_plates:
        return np.zeros(scenario.S, dtype=int)
    if settings.cold_plate_count is not None:
        return np.full(scenario.S, int(settings.cold_plate_count))
    return np.array([sv.plates for sv in scenario.servers], dtype=int)


def fixed_cold_plate_ops(scenario: Scenario, plates: np.ndarray, flow_Lmin: float,
                         coolant_C: float) -> ColdPlateOps:
    """One flow setting and coolant temperature for every cold plate and period."""
    scenario.cold_plate.level_of(flow_Lmin)
    has = (np.asarray(plates) > 0)[:, None]
    shape = (scenario.S, scenario.T)
    return ColdPlateOps(flow_Lmin=np.where(has, float(flow_Lmin), np.nan) * np.ones(shape),
                        coolant_C=np.where(has, float(coolant_C), np.nan) * np.ones(shape))


def priority_pairs(scenario: Scenario) -> list[tuple[int, int]]:
    """Ordered pairs (i, j) with O_ij = 1: i goes before j.

    Eq. (56): a time-sensitive i goes before a delay-tolerant j. Eq. (57): between two
    workloads of the same type and deadline, the earlier arrival goes first.

    O_ij is linked to the schedule as follows (the thesis prints no link, see
    docs/PROJECT_CONTEXT.md A6): if i and j run on the same server and j starts while i is
    waiting (A_i <= ST_j), then ST_i <= ST_j. Workloads on different servers do not
    compete, and j may start before i arrives.
    """
    out = []
    for wi in scenario.workloads:
        for wj in scenario.workloads:
            if wi.i == wj.i:
                continue
            if (wi.RT == 1 and wj.RT == 0) or (wi.RT == wj.RT and wi.D == wj.D and wi.A < wj.A):
                out.append((wi.i, wj.i))
    return out


def cop(constants: Constants, T_supply_C):
    """CRAC coefficient of performance at a supply temperature in degC (section 4.1)."""
    c0, c1, c2 = constants.COP_coeffs
    return c0 * T_supply_C ** 2 + c1 * T_supply_C + c2


@dataclass(frozen=True)
class Placement:
    """One workload's place in a schedule. ``server`` is None if it was not placed."""
    i: int
    server: int | None
    start: int | None
    end: int | None       # exclusive: start + periods run

    @property
    def periods_run(self) -> int | None:
        return None if self.start is None else self.end - self.start


@dataclass(frozen=True)
class Schedule:
    placements: tuple[Placement, ...]   # sorted by i; placements[i - 1].i == i

    def server_of(self) -> dict[int, int | None]:
        return {p.i: p.server for p in self.placements}


def constant_supply_temp(scenario: Scenario, value_C: float) -> np.ndarray:
    """Supply temperature array of shape (Z, T) at one fixed value."""
    return np.full((scenario.Z, scenario.T), float(value_C))


@dataclass
class EvalResult:
    """Everything the shared evaluator recomputes from a schedule and supply temperatures.

    Shapes: (S, T) per server per period, (Z, T) per zone per period, (T,) per period,
    (I,) per workload. Unplaced workloads have NaN in the per-workload float arrays.
    """
    # per server per period
    cpu_used_cores: np.ndarray      # sum of C_req running, Eq. (12) left side
    mem_used_MB: np.ndarray         # Eq. (13) left side
    util: np.ndarray                # Eq. (16)
    P_IT_W: np.ndarray              # Eq. (20); equals Q_heat, Eq. (21)
    Q_cp_W: np.ndarray              # cold-plate heat removed (all plates of the server)
    P_cp_W: np.ndarray              # cold-plate pump power, Eq. (48)
    T_C: np.ndarray                 # temperature recursion
    headroom_C: np.ndarray          # Eq. (31)
    DPF: np.ndarray                 # Eq. (37), all three branches
    PC_eff_MHz: np.ndarray          # Eq. (38): N_cores * f_s * DPF
    # per zone per period
    T_supply_C: np.ndarray          # as given
    Q_crac_kW: np.ndarray           # heat removed = heat balance at equality
    COP: np.ndarray
    P_crac_kW: np.ndarray           # Q_crac / COP
    P_fan_kW: np.ndarray            # zero slot; airflow not modelled
    # per period
    E_IT_kWh: np.ndarray            # Eq. (3) summed over servers
    E_crac_kWh: np.ndarray          # Eq. (4) summed over zones
    E_cp_kWh: np.ndarray            # Eq. (5) summed over servers
    E_fan_kWh: np.ndarray           # zero slot
    E_total_kWh: np.ndarray         # Eq. (2)
    carbon_g: np.ndarray            # Eq. (6)
    # per workload
    wait_periods: np.ndarray        # start - A, Eq. (24)
    work_delivered_MHz_s: np.ndarray
    work_required_MHz_s: np.ndarray
    placed: np.ndarray              # bool
    starts_on_or_after_arrival: np.ndarray   # bool, Eq. (23)
    wait_within_allowed: np.ndarray          # bool, Eq. (28)
    on_time: np.ndarray                      # bool, Eq. (27): end <= D
    work_complete: np.ndarray                # bool, Eq. (39)
    compliant: np.ndarray                    # bool: all of the above
    min_DPF_run: np.ndarray                  # lowest DPF over the periods it runs, NaN if unplaced
    # totals
    total_carbon_g: float           # Eq. (7), (8)
    totals: dict = field(default_factory=dict)       # scalar totals, see evaluate()
    compliance: dict = field(default_factory=dict)   # every check, see evaluate()
    airflow_modelled: bool = False
    n_cold_plates: np.ndarray | None = None     # (S,)
    cp_flow_Lmin: np.ndarray | None = None      # (S, T), NaN without cold plates
    cp_coolant_C: np.ndarray | None = None      # (S, T), NaN without cold plates

    @property
    def valid(self) -> bool:
        """True when the schedule breaks no limit of the model."""
        return bool(self.compliance["all_ok"])


@dataclass
class SolveResult:
    """What the optimizer returns. Numbers here are the solver's own; the comparison
    uses the evaluator's numbers for the same schedule."""
    status: str                         # one of the STATUS_* values
    schedule: Schedule | None           # None when no schedule was found
    supply_temp_C: np.ndarray | None    # shape (Z, T)
    objective_g: float | None           # solver objective, grams CO2
    best_bound_g: float | None
    mip_gap: float | None               # relative gap reported by the solver
    solve_time_s: float
    n_variables: int
    n_binaries: int
    n_constraints: int
    relaxed: tuple[str, ...] = ()       # constraint families relaxed in this solve
    solver_status_text: str = ""        # raw HiGHS model status
    notes: list[str] = field(default_factory=list)
    model_values: dict = field(default_factory=dict)   # optional solver-side T, DPF, ...
    cold_plate_ops: ColdPlateOps | None = None          # None when cold plates are off

    @property
    def has_schedule(self) -> bool:
        return self.schedule is not None


In [ ]:
%%writefile {LIB}/scheduler/io.py
"""Scenario loader: reads scenarios/<name>/*.csv and params.json into the shared types.

``availability.csv`` (columns s, t, AV) is optional: server-periods it does not list are
available, Eq. (11).
"""

from __future__ import annotations

import csv
import dataclasses
import json
from pathlib import Path

import numpy as np

from scheduler.types import (
    ColdPlateSpec, Constants, Placement, RunSettings, Scenario, Schedule, Server, Workload,
)


def _read_rows(path: Path) -> list[dict[str, str]]:
    if not path.is_file():
        raise ValueError(f"missing scenario file: {path}")
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def _blank(v: str | None) -> bool:
    return v is None or v.strip() == ""


def _int(v: str) -> int:
    return int(round(float(v)))


def _opt_int(v: str | None) -> int | None:
    return None if _blank(v) else _int(v)


def _check_contiguous(values: list[int], what: str, path: Path) -> None:
    expected = list(range(1, len(values) + 1))
    if values != expected:
        raise ValueError(f"{path}: column '{what}' must be 1..{len(values)} in order, "
                         f"got {values[:5]}...")


def _scenario_dir(name: str, scenarios_dir: str | Path) -> Path:
    d = Path(scenarios_dir) / name
    if not d.is_dir():
        raise ValueError(f"scenario directory not found: {d}")
    return d


def load_scenario(name: str, scenarios_dir: str | Path = "scenarios") -> Scenario:
    d = _scenario_dir(name, scenarios_dir)
    with open(d / "params.json") as f:
        params = json.load(f)

    T = int(params["time"]["T"])
    dt = float(params["time"]["dt_seconds"])

    # servers
    srv_path = d / "servers.csv"
    srv_rows = _read_rows(srv_path)
    if not srv_rows:
        raise ValueError(f"{srv_path}: no servers")
    _check_contiguous([_int(r["s"]) for r in srv_rows], "s", srv_path)
    servers = tuple(
        Server(
            s=_int(r["s"]), name=r["server"], zone=_int(r["zone"]),
            f_MHz=float(r["f_s_MHz"]), C_cores=_int(r["C_s_cores"]), M_MB=float(r["M_s_MB"]),
            P_idle_W=float(r["P_idle_W"]), P_max_W=float(r["P_max_W"]), T0_C=float(r["T0_C"]),
            CT_cold_plate=_int(r["CT_s_cold_plate"]), th_a_C=float(r["th_a_C"]),
            th_R_C_per_W=float(r["th_R_C_per_W"]), th_alpha=float(r["th_alpha_5min"]),
            n_cold_plates=_opt_int(r.get("n_cold_plates")),
        )
        for r in srv_rows
    )
    for sv in servers:
        if sv.plates < 0:
            raise ValueError(f"{srv_path}: server {sv.s} has {sv.plates} cold plates")
    n_zones = max(sv.zone for sv in servers)
    for sv in servers:
        if not 1 <= sv.zone <= n_zones:
            raise ValueError(f"{srv_path}: server {sv.s} has zone {sv.zone}, "
                             f"outside 1..{n_zones}")

    # workloads
    wl_path = d / "workloads.csv"
    wl_rows = _read_rows(wl_path)
    _check_contiguous([_int(r["i"]) for r in wl_rows], "i", wl_path)
    workloads = tuple(
        Workload(
            i=_int(r["i"]), workload_id=r["workload_id"], flavor=r["flavor"],
            A=_int(r["A_i"]), D=_int(r["D_i"]), AD=_int(r["AD_i"]),
            ET_base=_int(r["ET_base_i"]), RT=_int(r["RT_i"]), B=_int(r["B_i"]),
            X0=_opt_int(r["X0_s"]), C_req=_int(r["C_req_i"]),
            M_req_MB=float(r["M_req_i_MB"]), W_MHz_s=float(r["W_i_MHz_s"]),
            usage_imputed=_int(r["usage_imputed"]) if not _blank(r.get("usage_imputed")) else 0,
        )
        for r in wl_rows
    )

    # carbon
    ci = _read_carbon(d / "carbon.csv", T)

    # availability, Eq. (11)
    av_path = d / "availability.csv"
    AV = _read_availability(av_path, len(servers), T) if av_path.is_file() else None

    # constants
    c = params["constants_as_printed"]
    constants = Constants(
        R_dvfs=c["R_dvfs"], F_throttle=c["F_throttle"], T_max_server_C=c["T_max_server_C"],
        H_min_C=c["H_min_C"], T_dvfs_C=c["T_dvfs_C"], T_throttle_C=c["T_throttle_C"],
        Q_crac_kW=c["Q_crac_kW"], COP_coeffs=tuple(float(x) for x in c["COP_crac_coeffs"]),
        T_crac_min_C=c["T_crac_min_C"], T_crac_max_C=c["T_crac_max_C"],
        T_max_zone_C=c["T_max_zone_C"],
    )
    if len(constants.COP_coeffs) != 3:
        raise ValueError(f"{d / 'params.json'}: COP_crac_coeffs must have 3 entries")

    cold_plate = _cold_plate_spec(c, params.get("cold_plate", {}))
    cold_plate.validate()

    sc = Scenario(name=name, T=T, dt_seconds=dt, workloads=workloads, servers=servers,
                  n_zones=n_zones, CI_g_per_kWh=ci, constants=constants, cold_plate=cold_plate,
                  AV=AV)
    settings_dt = params.get("settings", {}).get("dt_seconds")
    if settings_dt is not None and float(settings_dt) != dt:
        raise ValueError(f"{d / 'params.json'}: time.dt_seconds={dt} does not match "
                         f"settings.dt_seconds={settings_dt}")
    return sc


def _read_carbon(path: Path, T: int) -> np.ndarray:
    rows = _read_rows(path)
    if len(rows) != T:
        raise ValueError(f"{path}: has {len(rows)} rows, the scenario has T={T}")
    _check_contiguous([_int(r["t"]) for r in rows], "t", path)
    return np.array([float(r["CI_t_g_per_kWh"]) for r in rows], dtype=float)


def _read_availability(path: Path, S: int, T: int) -> np.ndarray:
    AV = np.ones((S, T), dtype=int)
    for r in _read_rows(path):
        s, t, v = _int(r["s"]), _int(r["t"]), _int(r["AV"])
        if not (1 <= s <= S and 1 <= t <= T) or v not in (0, 1):
            raise ValueError(f"{path}: row s={s}, t={t}, AV={v} is outside 1..{S}, 1..{T}, {{0, 1}}")
        AV[s - 1, t - 1] = v
    return AV


def apply_overrides(scenario: Scenario, settings: RunSettings) -> Scenario:
    """Scenario switches of a run: zone map (S1 T1) and carbon intensity (S4 T3)."""
    sc = scenario
    if settings.zone_map is not None:
        zones = [int(z) for z in settings.zone_map]
        if len(zones) != sc.S:
            raise ValueError(f"zone_map has {len(zones)} entries, the scenario has {sc.S} servers")
        n_zones = max(zones)
        if sorted(set(zones)) != list(range(1, n_zones + 1)):
            raise ValueError(f"zone_map {zones}: every zone 1..{n_zones} must have a server")
        servers = tuple(dataclasses.replace(sv, zone=z) for sv, z in zip(sc.servers, zones))
        sc = dataclasses.replace(sc, servers=servers, n_zones=n_zones)
    if settings.carbon_file is not None:
        sc = dataclasses.replace(sc, CI_g_per_kWh=_read_carbon(Path(settings.carbon_file), sc.T))
    elif settings.ci_flat:
        sc = dataclasses.replace(sc, CI_g_per_kWh=np.full(sc.T, float(np.mean(sc.CI_g_per_kWh))))
    return sc


def _cold_plate_spec(c: dict, cp: dict) -> ColdPlateSpec:
    """Printed constants (kelvin and mW converted) plus the params.json "cold_plate" block.
    Anything missing keeps the ColdPlateSpec default."""
    kw: dict = {}
    if "Q_cp_kW" in c:
        kw["Q_max_per_plate_W"] = 1000.0 * float(c["Q_cp_kW"])
    if "T_cp_min_K" in c:
        kw["coolant_min_C"] = round(float(c["T_cp_min_K"]) - 273.15, 6)
    if "T_cp_max_K" in c:
        kw["coolant_max_C"] = round(float(c["T_cp_max_K"]) - 273.15, 6)
    if "FR_cp_min_Lmin" in c:
        kw["flow_min_Lmin"] = float(c["FR_cp_min_Lmin"])
    if "FR_cp_max_Lmin" in c:
        kw["flow_max_Lmin"] = float(c["FR_cp_max_Lmin"])
    if "P_cp_rated_as_printed_mW" in c:
        kw["pump_ref_W"] = float(c["P_cp_rated_as_printed_mW"]) / 1000.0
    if "FR_cp_rated_Lmin" in c:
        kw["pump_ref_flow_Lmin"] = float(c["FR_cp_rated_Lmin"])
    if "c_pw" in c:
        kw["c_pw_J_per_kgC"] = float(c["c_pw"])
    for key in ("flow_settings_Lmin", "R_C_per_W"):
        if key in cp:
            kw[key] = tuple(float(x) for x in cp[key])
    if "water_density_kg_per_L" in cp:
        kw["water_density_kg_per_L"] = float(cp["water_density_kg_per_L"])
    return ColdPlateSpec(**kw)


def load_reference_baseline(name: str, scenarios_dir: str | Path = "scenarios") -> Schedule:
    """Read scenarios/<name>/baseline_schedule.csv (columns i, s, start, end, late)."""
    path = _scenario_dir(name, scenarios_dir) / "baseline_schedule.csv"
    rows = _read_rows(path)
    _check_contiguous([_int(r["i"]) for r in rows], "i", path)
    placements = tuple(
        Placement(i=_int(r["i"]), server=_opt_int(r["s"]), start=_opt_int(r["start"]),
                  end=_opt_int(r["end"]))
        for r in rows
    )
    return Schedule(placements)


In [ ]:
%%writefile {LIB}/scheduler/baseline.py
"""Greedy, thermally blind baseline scheduler.

Workloads are taken in arrival order (the order of ``scenario.workloads``). Each one starts
at the earliest period at or after its arrival where some server has CPU room for the whole
run and is available (AV = 1, Eq. 11) throughout. Only CPU cores and availability are
checked: no memory, temperature or deadline. A workload that fits nowhere gets
``Placement(i, None, None, None)``.
"""

from __future__ import annotations

import math
import random

from scheduler.types import BASELINE_RULES, Placement, RunSettings, Scenario, Schedule


def periods_needed(workload, server, dt_seconds: float) -> int:
    """Run length of ``workload`` on ``server`` in periods: ceil(W / (C_req * f * dt)), at least 1."""
    return max(1, math.ceil(workload.W_MHz_s / (workload.C_req * server.f_MHz * dt_seconds)))


def build_baseline(scenario: Scenario, settings: RunSettings) -> Schedule:
    rule = settings.baseline_rule
    if rule not in BASELINE_RULES:
        raise ValueError(f"baseline_rule must be one of {BASELINE_RULES}, got {rule!r}")

    T, dt = scenario.T, scenario.dt_seconds
    servers = scenario.servers
    # used[s - 1][t] = cores in use in period t (index 0 unused, periods 1..T)
    used = [[0] * (T + 2) for _ in servers]
    AV = scenario.availability()
    rng = random.Random(settings.seed)
    by_speed = sorted(servers, key=lambda sv: (-sv.f_MHz, sv.s))

    def has_room(sv, w, start: int) -> int | None:
        """Return the end (exclusive) of the run if ``w`` fits on ``sv`` from ``start``."""
        end = start + periods_needed(w, sv, dt)
        if end - 1 > T:
            return None
        u, av = used[sv.s - 1], AV[sv.s - 1]
        if all(u[t] + w.C_req <= sv.C_cores and av[t - 1] for t in range(start, end)):
            return end
        return None

    placements: list[Placement] = []
    for w in scenario.workloads:
        chosen = None  # (server, start, end)
        for start in range(max(1, w.A), T + 1):
            if rule == "least_loaded":
                order = sorted(servers, key=lambda sv: (used[sv.s - 1][start], sv.s))
            elif rule == "fastest_first":
                order = by_speed
            else:  # random: pick uniformly among servers with room
                order = servers
            fits = []
            for sv in order:
                end = has_room(sv, w, start)
                if end is not None:
                    fits.append((sv, end))
                    if rule != "random":
                        break
            if fits:
                sv, end = rng.choice(fits) if rule == "random" else fits[0]
                chosen = (sv, start, end)
                break
        if chosen is None:
            placements.append(Placement(w.i, None, None, None))
            continue
        sv, start, end = chosen
        u = used[sv.s - 1]
        for t in range(start, end):
            u[t] += w.C_req
        placements.append(Placement(w.i, sv.s, start, end))
    return Schedule(tuple(placements))


In [ ]:
%%writefile {LIB}/scheduler/build.py
"""Scenario builder: turns the cleaned tables in data/clean/ into a scenario folder.

``build_scenario`` writes servers.csv, workloads.csv, carbon.csv, baseline_schedule.csv and
params.json from a dict of scenario settings, the thesis constants and the cold-plate table.
``notebooks/data_prep.ipynb`` calls it for the scenarios of its Settings cell.
``derive_scenario`` builds a new scenario from an existing one's params.json with some values
changed, for ``notebooks/solver.ipynb``. It reads only data/clean/, not the raw datasets.

Both notebooks hold a copy of this file, so that each runs on its own (on Colab, say);
after editing it, run ``uv run python tools/sync_notebooks.py``.
"""

from __future__ import annotations

import csv
import hashlib
import json
import math
import random
from pathlib import Path


def read_csv_rows(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def write_csv(path, records, fields=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = fields or list(records[0].keys())
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(records)


def pick_servers(sc, clean_dir):
    pool = [s for s in read_csv_rows(Path(clean_dir) / "servers.csv") if s["eligible"] == "1"]
    for s in pool:
        for k, v in s.items():
            if k != "server":
                s[k] = float(v)
    pool.sort(key=lambda s: s["P_max_W"] - s["P_idle_W"])
    n = sc["n_servers"]
    if n > len(pool):
        raise ValueError(f"n_servers = {n}, but only {len(pool)} servers in the cleaned table are eligible")
    chosen = [pool[int((i + 0.5) * len(pool) / n)] for i in range(n)]   # bin midpoints, no extremes

    freqs = [f for f, k in zip(sc["freq_mhz"], sc["freq_split"]) for _ in range(k)]
    top_f = max(sc["freq_mhz"])
    out = []
    for rank, (s, f) in enumerate(zip(sorted(chosen, key=lambda s: s["P_max_W"]), freqs)):
        out.append({
            "s": rank + 1, "server": s["server"], "zone": rank % sc["n_zones"] + 1,
            "f_s_MHz": f, "N_cores": int(s["N_cores"]), "C_s_cores": int(s["C_s_cores"]),
            "M_s_MB": round(s["M_s_MB"]), "PC_s_MHz": s["N_cores"] * f,
            "P_idle_W": round(s["P_idle_W"], 2), "P_max_W": round(s["P_max_W"], 2),
            "P_fit_full_load_W": round(s["P_fit_full_load_W"], 2),
            "T0_C": round(s["T0_C"], 2), "CT_s_cold_plate": int(f == top_f),
            "n_cold_plates": int(f == top_f),   # editable: plates on the server (0 = none)
            "th_a_C": round(s["th_a_C"], 3), "th_R_C_per_W": round(s["th_R_C_per_W"], 5),
            "th_alpha_5min": round(s["th_alpha_5min"], 5),
            # dynamic watts drawn per MHz of work delivered: lower is more efficient
            "W_per_MHz": round((s["P_max_W"] - s["P_idle_W"]) / (s["N_cores"] * f), 6),
        })
    return out


def is_time_sensitive(vm_id, seed, share):
    # Seeded hash of the VM id, uniform on [0, 1): depends only on this workload, so the
    # flag does not change when other workloads enter or leave the catalogue. Keyed on
    # the UFCG id because workload_id is a sort position in the catalogue and shifts with it.
    h = hashlib.sha256(f"{seed}:{vm_id}".encode()).digest()
    return int(int.from_bytes(h[:8], "big") / 2 ** 64 < share)


def pick_workloads(sc, fleet_cores, clean_dir):
    T, rng = sc["periods"], random.Random(sc["seed"])
    per_day = 86400 // sc["dt_seconds"]
    start = sc["start_hour_local"] * 3600 // sc["dt_seconds"]
    cat = [w for w in read_csv_rows(Path(clean_dir) / "workloads.csv")
           if sc["min_et_periods"] <= int(w["ET_base_periods"]) <= sc["max_et_periods"]]
    rng.shuffle(cat)

    demand, picked = [0] * (T + 2), []
    for w in cat:
        et, vcpu = int(w["ET_base_periods"]), int(w["C_req_vcpu"])
        a = (int(w["arrival_slot_of_day"]) - start) % per_day + 1
        rt = is_time_sensitive(w["ufcg_server_id"], sc["seed"], sc["rt_share"])
        slack = max(1, math.ceil(sc["rt_slack_frac"] * et)) if rt else \
            min(sc["dt_slack_max"], max(sc["dt_slack_min"], math.ceil(sc["dt_slack_mult"] * et)))
        d = min(a + et + slack, T + 1)          # D_i: first period by which it must be done
        if d - a - et < 1:
            continue                            # cannot finish inside the horizon
        for t in range(a, a + et):
            demand[t] += vcpu
        picked.append({
            "workload_id": w["workload_id"], "flavor": w["flavor_name"],
            "A_i": a, "D_i": d, "AD_i": d - a - et, "ET_base_i": et, "RT_i": rt, "B_i": 0,
            "C_req_i": vcpu, "M_req_i_MB": int(w["M_req_MB"]), "W_i_MHz_s": float(w["W_MHz_s"]),
            "usage_imputed": int(w["usage_imputed"]),
        })
        if max(demand) >= sc["target_peak_load"] * fleet_cores:
            break
    if not picked:
        raise ValueError("no workload fits: none can finish inside the horizon with these settings")
    picked.sort(key=lambda w: (w["A_i"], w["workload_id"]))
    for i, w in enumerate(picked, 1):
        w["i"] = i
    return picked, max(demand), len(cat)


def carbon_profile(sc):
    # Annual mean with a synthetic daily shape: a midday dip and an evening peak.
    T, dt = sc["periods"], sc["dt_seconds"]
    hours = [(sc["start_hour_local"] + (t + 0.5) * dt / 3600) % 24 for t in range(T)]
    bump = lambda h, mu, sd: math.exp(-0.5 * (min(abs(h - mu), 24 - abs(h - mu)) / sd) ** 2)
    day = [bump(h / 12, 19, 2.0) - bump(h / 12, 12, 3.0) for h in range(288)]
    mean, span = sum(day) / 288, max(abs(x - sum(day) / 288) for x in day)
    shape = lambda h: (bump(h, 19, 2.0) - bump(h, 12, 3.0) - mean) / span
    return [{"t": t + 1, "hour_local": round(h, 3),
             "CI_t_g_per_kWh": round(sc["ci_mean_g_per_kwh"] * (1 + sc["ci_amplitude"] * shape(h)), 2)}
            for t, h in enumerate(hours)]


def baseline_schedule(servers, workloads, sc):
    # Thermally unaware reference: each workload starts at arrival on the least-loaded
    # server that has room.
    T, dt = sc["periods"], sc["dt_seconds"]
    used = {s["s"]: [0] * (T + 2) for s in servers}
    sched = []
    for w in workloads:
        placed = None
        for start in range(w["A_i"], T + 1):
            for s in sorted(servers, key=lambda s: (used[s["s"]][start], s["s"])):
                et = math.ceil(w["W_i_MHz_s"] / (w["C_req_i"] * s["f_s_MHz"] * dt))
                span = range(start, start + max(1, et))
                if span[-1] <= T and all(used[s["s"]][t] + w["C_req_i"] <= s["C_s_cores"] for t in span):
                    placed = (s, span)
                    break
            if placed:
                break
        if not placed:
            sched.append({"i": w["i"], "s": "", "start": "", "end": "", "late": 1})
            continue
        s, span = placed
        for t in span:
            used[s["s"]][t] += w["C_req_i"]
        w["X0_s"] = s["s"]
        sched.append({"i": w["i"], "s": s["s"], "start": span[0], "end": span[-1] + 1,
                      "late": int(span[-1] + 1 > w["D_i"])})
    return sched, used


def check_settings(sc):
    """The scenario settings must describe a fleet the builder can make."""
    if len(sc["freq_mhz"]) != len(sc["freq_split"]):
        raise ValueError(f"freq_mhz has {len(sc['freq_mhz'])} entries but freq_split has {len(sc['freq_split'])}")
    if sum(sc["freq_split"]) != sc["n_servers"]:
        raise ValueError(f"freq_split {sc['freq_split']} adds up to {sum(sc['freq_split'])}, "
                         f"not n_servers = {sc['n_servers']}")
    if not 1 <= sc["n_zones"] <= sc["n_servers"]:
        raise ValueError(f"n_zones = {sc['n_zones']} must be between 1 and n_servers = {sc['n_servers']}")
    if sc["periods"] < 1 or sc["min_et_periods"] > sc["max_et_periods"]:
        raise ValueError("periods must be at least 1 and min_et_periods at most max_et_periods")


def build_scenario(name, sc, const, cold_plate, *, clean_dir, scenarios_dir, f_ref_mhz,
                   slot_seconds=300, extra=None):
    """Write scenarios_dir/name/ from scenario settings ``sc``, the constants and the
    cold-plate table. ``extra`` is added to params.json. Returns (servers, diagnostics)."""
    assert sc["dt_seconds"] == slot_seconds, "period length must match the UFCG slot"
    check_settings(sc)
    out = Path(scenarios_dir) / name
    T, dt = sc["periods"], sc["dt_seconds"]

    servers = pick_servers(sc, clean_dir)
    fleet = sum(s["C_s_cores"] for s in servers)
    workloads, peak, n_cat = pick_workloads(sc, fleet, clean_dir)
    carbon = carbon_profile(sc)
    sched, used = baseline_schedule(servers, workloads, sc)

    # Baseline IT energy and carbon by Eq. (16), (20), (3), (6); cooling is not included.
    ci = [c["CI_t_g_per_kWh"] for c in carbon]
    kwh = [sum(s["P_idle_W"] + (s["P_max_W"] - s["P_idle_W"]) * used[s["s"]][t] / s["C_s_cores"]
               for s in servers) * dt / 3.6e6 for t in range(1, T + 1)]
    cap_T = const["T_max_server_C"] - const["H_min_C"]

    def u_ceiling(s, inlet):
        # Highest steady utilization that keeps the server at or under T_max - H_min.
        p = (cap_T - inlet - s["th_a_C"]) / s["th_R_C_per_W"]
        return round(max(0.0, min(1.0, (p - s["P_idle_W"]) / (s["P_max_W"] - s["P_idle_W"]))), 3)

    for s in servers:
        s["U_ceiling_inlet_min"] = u_ceiling(s, const["T_crac_min_C"])
        s["U_ceiling_inlet_max"] = u_ceiling(s, const["T_crac_max_C"])

    write_csv(out / "servers.csv", servers)
    write_csv(out / "workloads.csv", workloads, ["i", "workload_id", "flavor", "A_i", "D_i", "AD_i", "ET_base_i",
                                                 "RT_i", "B_i", "X0_s", "C_req_i", "M_req_i_MB", "W_i_MHz_s",
                                                 "usage_imputed"])
    write_csv(out / "carbon.csv", carbon)
    write_csv(out / "baseline_schedule.csv", sched)

    top_f = max(sc["freq_mhz"])
    diag = {
        "I_workloads": len(workloads), "S_servers": len(servers), "Z_zones": sc["n_zones"], "T_periods": T,
        # Y_ist can only be 1 between arrival and deadline, so this is the real count
        "Y_ist_binaries_inside_windows": sum(w["D_i"] - w["A_i"] for w in workloads) * len(servers),
        "catalogue_size_within_ET_limits": n_cat,
        "fleet_cores": fleet,
        "peak_requested_vcpus": peak,
        "peak_load_share_of_fleet": round(peak / fleet, 3),
        "peak_load_share_of_fastest_tier": round(peak / sum(s["C_s_cores"] for s in servers if s["f_s_MHz"] == top_f), 3),
        "time_sensitive_workloads": sum(w["RT_i"] for w in workloads),
        "workloads_with_imputed_usage": sum(w["usage_imputed"] for w in workloads),
        "median_slack_periods_delay_tolerant": sorted(w["AD_i"] for w in workloads if not w["RT_i"])[
            sum(1 for w in workloads if not w["RT_i"]) // 2] if any(not w["RT_i"] for w in workloads) else None,
        "W_per_MHz_spread_best_to_worst": round(max(s["W_per_MHz"] for s in servers) / min(s["W_per_MHz"] for s in servers), 2),
        "servers_thermally_capped_at_coldest_inlet": sum(s["U_ceiling_inlet_min"] < 1 for s in servers),
        "servers_thermally_capped_at_warmest_inlet": sum(s["U_ceiling_inlet_max"] < 1 for s in servers),
        "CI_min_max_g_per_kWh": [min(ci), max(ci)],
        "baseline_IT_energy_kWh": round(sum(kwh), 2),
        # idle draw is paid whatever the schedule is: the model has no server on/off variable
        "baseline_idle_share_of_IT_energy": round(sum(s["P_idle_W"] for s in servers) * T * dt / 3.6e6 / sum(kwh), 3),
        "baseline_IT_carbon_kg": round(sum(k * c for k, c in zip(kwh, ci)) / 1000, 2),
        "baseline_unplaced_or_late": sum(x["late"] for x in sched),
    }
    params = {
        "scenario": name, "settings": sc, "constants_as_printed": const,
        "cold_plate": cold_plate,
        "time": {"dt_seconds": dt, "T": T, "period_1_local_hour": sc["start_hour_local"],
                 "convention": "A_i = first period it may run; D_i = period by which it must be finished "
                               "(CT_i = ST_i + ET_i <= D_i); AD_i = D_i - A_i - ET_base_i, Eq. (22) as an equality"},
        "work_rule": "ET_is = W_i / (C_req_i * f_s * DPF * dt): a workload is served by its own requested "
                     "cores, not the whole server. ET_base_i uses f_ref.",
        "f_ref_MHz": f_ref_mhz,
        "thermal_fit": "steady state T_cpu = T_inlet + th_a_C + th_R_C_per_W * P_IT; "
                       "per period T[t] = alpha*T[t-1] + (1-alpha)*steady, alpha = th_alpha_5min. "
                       "Fitted from UniMelb; not part of the thesis formulation.",
        "assumed_not_from_data": ["RT_i", "D_i", "zone", "CT_s_cold_plate", "n_cold_plates",
                                  "f_s_MHz assignment",
                                  "daily shape of CI_t", "AV_st = 1 for all s, t", "B_i = 0"],
        "diagnostics": diag,
        **(extra or {}),
    }
    with open(out / "params.json", "w") as f:
        json.dump(params, f, indent=2)
    return servers, diag


# --- Only notebooks/solver.ipynb uses what follows: new scenarios from existing ones.
# (tools/sync_notebooks.py copies the code above this line into notebooks/data_prep.ipynb.)

# Written by data_prep.ipynb from its Settings cell. derive_scenario never overwrites them.
SHIPPED_SCENARIOS = ("base", "thesis10", "small", "small_running")


def _changed(base: dict, changes: dict | None, what: str) -> dict:
    changes = dict(changes or {})
    unknown = sorted(set(changes) - set(base))
    if unknown:
        raise ValueError(f"unknown {what}: {unknown}. Known: {sorted(base)}")
    return {k: v for k, v in changes.items() if v != base[k]}


def derive_scenario(name, start_from, settings=None, constants=None, cold_plate=None, *,
                    scenarios_dir="scenarios", clean_dir="data/clean"):
    """Build scenario ``name`` from scenario ``start_from``: its settings, constants and
    cold-plate table, with the given values changed. Each dict holds only the keys to change.

    Refuses to write over the shipped scenarios or over a folder this function did not
    write. Returns a dict with the starting and new params and the values that changed.
    """
    scenarios_dir = Path(scenarios_dir)
    if name in SHIPPED_SCENARIOS or name == start_from:
        raise ValueError(f"'{name}' is a shipped scenario or the starting one; choose a new name")
    out = scenarios_dir / name
    if out.exists() and not _built_here(out):
        raise ValueError(f"{out} exists and was not built by derive_scenario; choose a new name")
    start_path = scenarios_dir / start_from / "params.json"
    if not start_path.is_file():
        raise ValueError(f"scenario to start from not found: {start_path}")
    with open(start_path) as f:
        start = json.load(f)
    if "derived_from" in start:
        raise ValueError(f"'{start_from}' was edited by hand ({start['derived_from']['scenario']} plus "
                         f"running workloads or downtime), which the builder does not reproduce; "
                         f"start from '{start['derived_from']['scenario']}'")

    changed = {
        "settings": _changed(start["settings"], settings, "scenario setting"),
        "constants": _changed(start["constants_as_printed"], constants, "constant"),
        "cold_plate": _changed(start["cold_plate"], cold_plate, "cold-plate value"),
    }
    if "dt_seconds" in changed["settings"]:
        raise ValueError("dt_seconds is the slot length of the cleaned data and cannot be changed here")
    sc = start["settings"] | changed["settings"]
    const = start["constants_as_printed"] | changed["constants"]
    cp = start["cold_plate"] | changed["cold_plate"]
    built_from = {"scenario": start_from, "changed": {k: v for k, v in changed.items() if v}}

    build_scenario(name, sc, const, cp, clean_dir=clean_dir, scenarios_dir=scenarios_dir,
                   f_ref_mhz=start["f_ref_MHz"], slot_seconds=start["settings"]["dt_seconds"],
                   extra={"built_from": built_from})
    with open(out / "params.json") as f:
        new = json.load(f)
    return {"start": start, "params": new, "changed": built_from["changed"]}


def _built_here(d: Path) -> bool:
    try:
        with open(d / "params.json") as f:
            return "built_from" in json.load(f)
    except (OSError, ValueError):
        return False


In [ ]:
%%writefile {LIB}/scheduler/optimize.py
"""MILP optimizer (HiGHS) for thermal-aware workload scheduling.

Equation numbers refer to docs/PROJECT_CONTEXT.md section 2. This module builds the
constraints itself and shares no code with evaluate.py (the independent referee) or
baseline.py.

Model in one paragraph
----------------------
Each workload i runs as one unbroken block of periods on one server. ``Y[i,s,t]`` says it
runs on s in t. ``U[i,s,t]`` ("started by t") is a monotone 0/1 step that replaces the
start indicators ``S[i,s,t] = U[i,s,t] - U[i,s,t-1]``; ``X[i,s]`` is the last step of U.
Server power, the temperature recursion, a linear upper bound on the performance factor
DPF, and the work credit ``w`` follow the approved thesis equations. The supply
temperature of every zone and period is chosen from whole degrees by a one-hot choice so
the COP is a lookup. With cold plates on, each cold-plate server picks one flow setting per
period (one-hot, so resistance and pump power are lookups) and a coolant temperature; the
heat removed is linear in the temperature difference split across the settings. Priority
(56)-(57) orders the start times ``ST[i]`` of workloads that share a server, and servers
take no work in periods where AV = 0 (11). All limits are hard.

Units: P_IT in W, Q_crac / P_crac / P_fan in kW, dt in seconds, energy in kWh, CI in
g/kWh, carbon in g.
"""

from __future__ import annotations

import dataclasses
import math
import os
import tempfile
import time
from collections import defaultdict

import numpy as np

import highspy

from .types import (
    RELAX_COOLING, RELAX_CAPACITY, RELAX_DEADLINES, RELAX_FAMILIES, RELAX_PRIORITY, RELAX_THERMAL,
    STATUS_INFEASIBLE, STATUS_NO_SOLUTION, STATUS_OPTIMAL, STATUS_TIME_LIMIT,
    ColdPlateOps, Placement, RunSettings, Scenario, Schedule, SolveResult, cold_plate_counts, cop,
    priority_pairs,
)

_INF = highspy.kHighsInf
_EPS = 1e-9
_IPM_BINARIES = 30_000   # above this many binaries the MIP's LP relaxations use interior point


# ---------------------------------------------------------------------------
# Small LP container filled in bulk and handed to HiGHS in one call
# ---------------------------------------------------------------------------

class _LP:
    def __init__(self) -> None:
        self.lb: list[float] = []
        self.ub: list[float] = []
        self.cost: list[float] = []
        self.integer: list[bool] = []
        self.r_lo: list[float] = []
        self.r_hi: list[float] = []
        self.r_len: list[int] = []
        self.r_idx: list[int] = []
        self.r_val: list[float] = []

    def col(self, lb: float, ub: float, cost: float = 0.0, integer: bool = False) -> int:
        self.lb.append(lb)
        self.ub.append(ub)
        self.cost.append(cost)
        self.integer.append(integer)
        return len(self.lb) - 1

    def row(self, terms, lo: float, hi: float) -> None:
        """Add ``lo <= sum coef*col <= hi``; duplicate columns are merged."""
        acc: dict[int, float] = {}
        for c, v in terms:
            if c is not None:
                acc[c] = acc.get(c, 0.0) + v
        items = [(c, v) for c, v in acc.items() if abs(v) > 1e-13]
        self.r_lo.append(lo)
        self.r_hi.append(hi)
        self.r_len.append(len(items))
        for c, v in items:
            self.r_idx.append(c)
            self.r_val.append(v)

    @property
    def n_cols(self) -> int:
        return len(self.lb)

    @property
    def n_rows(self) -> int:
        return len(self.r_lo)

    def to_highs(self, log: bool = False) -> highspy.Highs:
        h = highspy.Highs()
        h.setOptionValue("log_to_console", bool(log))   # before passModel: no banner
        lp = highspy.HighsLp()
        lp.num_col_ = self.n_cols
        lp.num_row_ = self.n_rows
        lp.col_cost_ = np.asarray(self.cost, dtype=np.float64)
        lp.col_lower_ = np.asarray(self.lb, dtype=np.float64)
        lp.col_upper_ = np.asarray(self.ub, dtype=np.float64)
        lp.row_lower_ = np.asarray(self.r_lo, dtype=np.float64)
        lp.row_upper_ = np.asarray(self.r_hi, dtype=np.float64)
        lp.a_matrix_.format_ = highspy.MatrixFormat.kRowwise
        start = np.zeros(self.n_rows + 1, dtype=np.int32)
        np.cumsum(np.asarray(self.r_len, dtype=np.int64), out=start[1:])
        lp.a_matrix_.start_ = start
        lp.a_matrix_.index_ = np.asarray(self.r_idx, dtype=np.int32)
        lp.a_matrix_.value_ = np.asarray(self.r_val, dtype=np.float64)
        integ = np.asarray(self.integer, dtype=bool)
        lp.integrality_ = [highspy.HighsVarType.kInteger if b else highspy.HighsVarType.kContinuous
                           for b in integ]
        h.passModel(lp)
        return h


# ---------------------------------------------------------------------------
# Cold plates. The temperature, heat-balance and objective code reference the columns
# Q_cp[s,t] (W heat removed by all plates of s) and P_cp[s,t] (W pump power).
# ---------------------------------------------------------------------------

class _ColdPlateCols:
    """Per cold-plate server k and period t: one-hot flow choice b[k][t][j], the temperature
    difference split d[k][t][j] across the settings, and the coolant temperature cool[k][t].
    With the settings fixed (cp_mode "fixed") b and cool are fixed columns, and neg[k][t]
    (coolant above T_prev) with the switch on[k][t] lets the drive fall to zero."""

    def __init__(self):
        self.b: dict[int, list[list[int]]] = {}
        self.d: dict[int, list[list[int]]] = {}
        self.cool: dict[int, list[int]] = {}
        self.neg: dict[int, list[int]] = {}
        self.on: dict[int, list[int]] = {}
        self.fixed: tuple[int, float] | None = None    # (flow level, coolant degC)


def _add_cold_plate_terms(lp: _LP, scenario: Scenario, plates: np.ndarray, d_max: float,
                          ci_w_to_g: np.ndarray, fixed: tuple[int, float] | None = None):
    """Q_cp[s,t] and P_cp[s,t] columns, plus the choice columns of servers with cold plates.

    Servers without cold plates get Q_cp = P_cp = 0. P_cp enters the objective with the same
    weight as IT power (W -> g per period). ``fixed`` = (flow level, coolant) holds every
    plate at one setting. The rows are added by _add_cold_plate_rows.
    """
    S, T = scenario.S, scenario.T
    cp = scenario.cold_plate
    pump = cp.pump_W()
    nlev = len(cp.flow_settings_Lmin)
    cols = _ColdPlateCols()
    cols.fixed = fixed
    Q_cp, P_cp = [], []
    for k in range(S):
        n = int(plates[k])
        Q_cp.append([lp.col(0.0, n * cp.Q_max_per_plate_W) for _ in range(T)])
        P_cp.append([lp.col(n * pump.min(), n * pump.max(), cost=float(ci_w_to_g[t]))
                     for t in range(T)])
        if n == 0:
            continue
        if fixed is None:
            cols.b[k] = [[lp.col(0.0, 1.0, integer=True) for _ in range(nlev)] for _ in range(T)]
            cols.cool[k] = [lp.col(cp.coolant_min_C, cp.coolant_max_C) for _ in range(T)]
        else:
            lv, cool = fixed
            cols.b[k] = [[lp.col(float(j == lv), float(j == lv)) for j in range(nlev)]
                         for _ in range(T)]
            cols.cool[k] = [lp.col(cool, cool) for _ in range(T)]
            cols.on[k] = [lp.col(0.0, 1.0, integer=True) for _ in range(T)]
            cols.neg[k] = [lp.col(0.0, _INF) for _ in range(T)]
        cols.d[k] = [[lp.col(0.0, d_max) for _ in range(nlev)] for _ in range(T)]
    return Q_cp, P_cp, cols


def _add_cold_plate_rows(lp: _LP, scenario: Scenario, plates: np.ndarray, d_max: float,
                         cols: _ColdPlateCols, Q_cp, P_cp, P_IT, Tcol,
                         neg_max: dict[int, float] | None = None) -> None:
    """For server k with n plates in period t, with T_prev = T[k,t-1] (T0 in period 1):

        sum_j b_j = 1                              one flow setting
        sum_j d_j = T_prev - coolant               d_j <= d_max * b_j, d_j >= 0
        Q_cp <= n * sum_j G_j * d_j                heat through the plates, "<=" as agreed
        Q_cp <= P_IT                               no more than the server's heat
        Q_cp <= n * Q_max                          column bound, Eq. (55)
        P_cp  = n * sum_j pump_j * b_j             Eq. (48) at the chosen setting

    With the settings fixed the coolant cannot follow T_prev down, so the drive is
    max(0, T_prev - coolant): sum_j d_j - neg = T_prev - coolant, sum_j d_j <= d_max * on,
    neg <= neg_max[k] * (1 - on).
    """
    cp = scenario.cold_plate
    G = cp.conductance_W_per_C()
    pump = cp.pump_W()
    for k, bk in cols.b.items():
        n = int(plates[k])
        sv = scenario.servers[k]
        for t in range(scenario.T):
            b, d, cool = bk[t], cols.d[k][t], cols.cool[k][t]
            lp.row([(c_, 1.0) for c_ in b], 1.0, 1.0)
            terms = [(c_, 1.0) for c_ in d] + [(cool, 1.0)]
            if cols.fixed is not None:
                on, neg = cols.on[k][t], cols.neg[k][t]
                terms.append((neg, -1.0))
                lp.row([(c_, 1.0) for c_ in d] + [(on, -d_max)], -_INF, 0.0)
                lp.row([(neg, 1.0), (on, neg_max[k])], -_INF, neg_max[k])
            if t == 0:
                lp.row(terms, sv.T0_C, sv.T0_C)
            else:
                lp.row(terms + [(Tcol[k][t - 1], -1.0)], 0.0, 0.0)
            for dj, bj in zip(d, b):
                lp.row([(dj, 1.0), (bj, -d_max)], -_INF, 0.0)
            lp.row([(Q_cp[k][t], 1.0)] + [(dj, -n * float(g)) for dj, g in zip(d, G)], -_INF, 0.0)
            lp.row([(Q_cp[k][t], 1.0), (P_IT[k][t], -1.0)], -_INF, 0.0)
            lp.row([(P_cp[k][t], 1.0)] + [(bj, -n * float(pw)) for bj, pw in zip(b, pump)], 0.0, 0.0)


def _add_fan_terms(lp: _LP, scenario: Scenario, ci_kw_to_g: np.ndarray):
    """P_fan[z,t] (kW): fixed at zero; enters the objective (kW -> g per period)."""
    Z, T = scenario.Z, scenario.T
    return [[lp.col(0.0, 0.0, cost=float(ci_kw_to_g[t])) for t in range(T)] for _ in range(Z)]


# ---------------------------------------------------------------------------
# Model construction
# ---------------------------------------------------------------------------

class _Window:
    __slots__ = ("a", "b", "ya", "yb", "n_min", "n_max")

    def __init__(self, a, b, ya, yb, n_min, n_max):
        self.a, self.b, self.ya, self.yb, self.n_min, self.n_max = a, b, ya, yb, n_min, n_max


class _Built:
    """The assembled model plus the index maps needed to read a solution."""


def _levels(scenario: Scenario) -> list[int]:
    c = scenario.constants
    lo = math.ceil(c.T_crac_min_C - _EPS)
    hi = math.floor(c.T_crac_max_C + _EPS)
    levels = [k for k in range(lo, hi + 1) if k <= c.T_max_zone_C + _EPS]
    if not levels:
        raise ValueError("no whole-degree supply temperature lies in the CRAC range and "
                         "below the zone limit")
    return levels


def _allowed_servers(scenario: Scenario, original_server: dict[int, int],
                     settings: RunSettings) -> dict[int, list[int]]:
    zone_of = {sv.s: sv.zone for sv in scenario.servers}
    out: dict[int, list[int]] = {}
    for w in scenario.workloads:
        if w.i not in original_server:
            raise ValueError(f"original_server has no entry for workload {w.i}")
        o = original_server[w.i]
        if o not in zone_of:
            raise ValueError(f"original_server[{w.i}] = {o} is not a server of the scenario")
        if w.B == 1 or settings.reassignment == "none":
            out[w.i] = [o]
        elif settings.reassignment == "same_zone":
            out[w.i] = [sv.s for sv in scenario.servers if sv.zone == zone_of[o]]
        else:
            out[w.i] = [sv.s for sv in scenario.servers]
    return out


def _build(scenario: Scenario, original_server: dict[int, int], settings: RunSettings,
           relax: tuple[str, ...], feasibility_only: bool):
    settings.validate()
    for fam in relax:
        if fam not in RELAX_FAMILIES:
            raise ValueError(f"unknown relaxation family {fam!r}")
    c = scenario.constants
    S, Z, T, I = scenario.S, scenario.Z, scenario.T, scenario.I
    dt = scenario.dt_seconds
    dt_h = scenario.dt_hours
    cap = settings.thermal_cap_C(scenario)
    if cap > c.T_throttle_C + _EPS:
        raise ValueError(
            f"thermal cap {cap:g} degC (T_max - H_min) exceeds T_throttle {c.T_throttle_C:g} degC: "
            "the linear DPF bound is only valid below T_throttle")

    relax_dead = RELAX_DEADLINES in relax
    relax_therm = RELAX_THERMAL in relax
    relax_cap = RELAX_CAPACITY in relax
    relax_cool = RELAX_COOLING in relax
    notes: list[str] = []

    fixed_mode = settings.supply_temp_mode == "fixed"
    if feasibility_only and not fixed_mode:
        # Feasibility does not depend on the supply schedule: the lowest level gives the lowest
        # temperatures, the same Q_crac cap, and trivially satisfies the ramps and zone limit.
        fixed_mode = True
        feas_supply = float(_levels(scenario)[0])
        settings = dataclasses.replace(settings, supply_temp_mode="fixed",
                                       fixed_supply_temp_C=feas_supply)
    if fixed_mode:
        v = float(settings.optimized_fixed_supply_temp_C())
        if not (c.T_crac_min_C - _EPS <= v <= c.T_crac_max_C + _EPS):
            raise ValueError(f"fixed supply temperature {v:g} degC is outside the CRAC range "
                             f"[{c.T_crac_min_C:g}, {c.T_crac_max_C:g}]")
        if v > c.T_max_zone_C + _EPS:
            raise ValueError(f"fixed supply temperature {v:g} degC exceeds the zone limit "
                             f"{c.T_max_zone_C:g}")
        fixed_value = v
        levels: list[int] = []
    else:
        fixed_value = None
        levels = _levels(scenario)
    cop_levels = [float(cop(c, k)) for k in levels]

    # Highest temperature the T columns may take; the linear DPF bound holds up to T_throttle.
    cap_eff = c.T_throttle_C if relax_therm else cap
    derating = cap_eff > c.T_dvfs_C + _EPS     # DPF can fall below 1 somewhere
    slope = c.R_dvfs / (c.T_throttle_C - c.T_dvfs_C)
    dpf_cap = min(1.0, 1.0 - slope * (cap_eff - c.T_dvfs_C)) if derating else 1.0

    servers = scenario.servers
    zone_of = np.array([sv.zone for sv in servers], dtype=int)
    allowed = _allowed_servers(scenario, original_server, settings)
    CI = np.asarray(scenario.CI_g_per_kWh, dtype=float)
    if CI.shape != (T,):
        raise ValueError("CI_g_per_kWh must have shape (T,)")

    # ---- windows per (i, s) ------------------------------------------------
    win: dict[tuple[int, int], _Window] = {}
    empty_workloads: list[int] = []
    for w in scenario.workloads:
        D = T + 1 if relax_dead else min(w.D, T + 1)
        last_start_by_wait = T + 1 if relax_dead else w.A + w.AD
        for s in allowed[w.i]:
            sv = servers[s - 1]
            per_full = w.C_req * sv.f_MHz * dt          # work of one full-speed period
            if w.W_MHz_s <= 0:
                n_min = n_max = 1
            else:
                n_min = max(1, math.ceil(w.W_MHz_s / per_full - _EPS))
                n_max = (T if dpf_cap <= 1e-6 else
                         max(n_min, math.ceil(w.W_MHz_s / (per_full * dpf_cap) - _EPS)))
            a = max(w.A, 1)
            b = min(last_start_by_wait, D - n_min)
            if b < a:
                continue
            ya = a
            yb = min(D - 1, b + n_max - 1)
            win[(w.i, s)] = _Window(a, b, ya, yb, n_min, n_max)
        if not any((w.i, s) in win for s in allowed[w.i]):
            empty_workloads.append(w.i)

    built = _Built()
    built.empty_workloads = empty_workloads
    built.notes = notes
    built.win = win
    built.levels = levels
    built.fixed_value = fixed_value
    built.derating = derating
    built.relax = relax
    built.feasibility_only = feasibility_only
    if empty_workloads:
        return built

    lp = _LP()
    ci_w_to_g = CI * dt_h / 1000.0      # g per W held for one period
    ci_kw_to_g = CI * dt_h              # g per kW held for one period
    if feasibility_only:
        ci_w_to_g = np.zeros(T)
        ci_kw_to_g = np.zeros(T)

    # ---- scheduling columns ------------------------------------------------
    # X[i,s]: assigned; U[i,s,t]: started by t (t in [a, b-1]; U at b is X);
    # E[i,s,t]: tail period (beyond the n_min core periods) is on.
    X: dict[tuple[int, int], int] = {}
    U: dict[tuple[int, int], dict[int, int]] = {}
    E: dict[tuple[int, int], dict[int, int]] = {}
    for w in scenario.workloads:
        for s in allowed[w.i]:
            wd = win.get((w.i, s))
            if wd is None:
                continue
            X[(w.i, s)] = lp.col(0.0, 1.0, integer=True)
            U[(w.i, s)] = {t: lp.col(0.0, 1.0, integer=True) for t in range(wd.a, wd.b)}
            E[(w.i, s)] = {}
            if wd.n_max > wd.n_min:
                for t in range(wd.a + wd.n_min, wd.yb + 1):
                    E[(w.i, s)][t] = lp.col(0.0, 1.0, integer=True)
    built.X, built.U, built.E = X, U, E

    def ustep(i: int, s: int, t: int):
        """Column of 'started by t' for (i, s), or None when it is the constant 0."""
        wd = win[(i, s)]
        if t < wd.a:
            return None
        if t >= wd.b:
            return X[(i, s)]
        return U[(i, s)][t]

    # Occupancy of (i,s) in period t = core + tail = U[t] - U[t-n_min] + E[t]
    by_st: dict[tuple[int, int], list[tuple[int, list]]] = defaultdict(list)
    for (i, s), wd in win.items():
        emap = E[(i, s)]
        for t in range(wd.a, wd.yb + 1):
            terms = []
            if t <= wd.b + wd.n_min - 1:
                terms += [(ustep(i, s, t), 1.0), (ustep(i, s, t - wd.n_min), -1.0)]
            if t in emap:
                terms.append((emap[t], 1.0))
            by_st[(s, t)].append((i, terms))

    # Upper bound of IT power per server: all workloads that could ever run there.
    max_cores: dict[int, float] = defaultdict(float)
    for (i, s) in win:
        max_cores[s] += scenario.workloads[i - 1].C_req
    P_ub = []
    for sv in servers:
        if relax_cap:
            ub = sv.P_idle_W + (sv.P_max_W - sv.P_idle_W) * max_cores[sv.s] / sv.C_cores
        else:
            ub = sv.P_idle_W + (sv.P_max_W - sv.P_idle_W) * min(1.0, max_cores[sv.s] / sv.C_cores)
        P_ub.append(ub)

    # ---- system columns ----------------------------------------------------
    P_IT = [[lp.col(sv.P_idle_W, P_ub[sv.s - 1], cost=float(ci_w_to_g[t])) for t in range(T)]
            for sv in servers]
    plates = cold_plate_counts(scenario, settings)
    cp_d_max = max([cap_eff] + [sv.T0_C for sv in servers]) - scenario.cold_plate.coolant_min_C
    for sv in servers:
        if plates[sv.s - 1] > 0 and sv.T0_C < scenario.cold_plate.coolant_min_C:
            raise ValueError(f"server {sv.s}: T0 {sv.T0_C:g} C is below the lowest coolant "
                             f"temperature {scenario.cold_plate.coolant_min_C:g} C")
    cp_fixed = None
    cp_neg_max: dict[int, float] = {}
    if settings.cp_mode == "fixed" and plates.any():
        flow, cool = settings.optimized_fixed_cp()
        cps = scenario.cold_plate
        if not cps.coolant_min_C - _EPS <= cool <= cps.coolant_max_C + _EPS:
            raise ValueError(f"fixed coolant temperature {cool:g} degC is outside "
                             f"[{cps.coolant_min_C:g}, {cps.coolant_max_C:g}], Eq. (49)")
        cp_fixed = (cps.level_of(flow), cool)
        # Lowest temperature a server can reach: the steady state with all heat removed.
        sup_min = fixed_value if fixed_mode else float(levels[0])
        for sv in servers:
            cp_neg_max[sv.s - 1] = max(1.0, cool - min(sv.T0_C, sup_min + sv.th_a_C) + 1.0)
    Q_cp, P_cp, cp_cols = _add_cold_plate_terms(lp, scenario, plates, cp_d_max, ci_w_to_g,
                                                cp_fixed)
    P_fan = _add_fan_terms(lp, scenario, ci_kw_to_g)
    T_ub = cap_eff
    Tcol = [[lp.col(-_INF, T_ub) for t in range(T)] for _ in servers]
    # Loss L = 1 - DPF (only where some workload has tail periods, i.e. where DPF matters),
    # and its running sum CL, per server and period.
    Lmax = min(1.0, 1.0 - dpf_cap)
    tail_servers = {s for (i, s), wd in win.items() if wd.n_max > wd.n_min}
    Lcol: dict[int, list[int]] = {}
    CLcol: dict[int, list[int]] = {}
    if derating:
        for s in tail_servers:
            Lcol[s] = [lp.col(0.0, Lmax) for t in range(T)]
            CLcol[s] = [lp.col(0.0, T * Lmax) for t in range(T)]

    # Q_crac upper bound: Eq. (54) as a bound (also the big-M of the level split)
    sum_pub_kw = [sum(P_ub[sv.s - 1] for sv in servers if sv.zone == z) / 1000.0
                  for z in range(1, Z + 1)]
    Qmax = [sum_pub_kw[z - 1] if relax_cool else min(c.Q_crac_kW, sum_pub_kw[z - 1])
            for z in range(1, Z + 1)]
    Qcrac = [[None] * T for _ in range(Z)]
    Tsup = [[None] * T for _ in range(Z)]
    choice: dict[tuple[int, int], list[int]] = {}
    qcols: dict[tuple[int, int], list[int]] = {}
    for z in range(1, Z + 1):
        for t in range(T):
            if fixed_mode:
                Tsup[z - 1][t] = lp.col(fixed_value, fixed_value)
                cost = float(ci_kw_to_g[t] / cop(c, fixed_value))
                Qcrac[z - 1][t] = lp.col(0.0, Qmax[z - 1], cost=cost)
            else:
                Tsup[z - 1][t] = lp.col(levels[0], levels[-1])
                Qcrac[z - 1][t] = lp.col(0.0, Qmax[z - 1])
                choice[(z, t)] = [lp.col(0.0, 1.0, integer=True) for _ in levels]
                qcols[(z, t)] = [lp.col(0.0, Qmax[z - 1], cost=float(ci_kw_to_g[t] / cp))
                                 for cp in cop_levels]

    # ---- workload rows -----------------------------------------------------
    for w in scenario.workloads:
        i = w.i
        lp.row([(X[(i, s)], 1.0) for s in allowed[i] if (i, s) in X], 1.0, 1.0)          # (9)
        work_terms = []
        for s in allowed[i]:
            wd = win.get((i, s))
            if wd is None:
                continue
            Ucols = U[(i, s)]
            for t in range(wd.a + 1, wd.b):                      # monotone started-by
                lp.row([(Ucols[t], 1.0), (Ucols[t - 1], -1.0)], 0.0, _INF)
            if wd.b > wd.a:
                lp.row([(X[(i, s)], 1.0), (Ucols[wd.b - 1], -1.0)], 0.0, _INF)
            emap = E[(i, s)]
            for t, e in emap.items():
                # tail period t is on only inside the tail region of the actual start
                lp.row([(e, 1.0), (ustep(i, s, t - wd.n_min), -1.0), (ustep(i, s, t - wd.n_max), 1.0)],
                       -_INF, 0.0)
                # ... and only as an unbroken extension of the core block
                lp.row([(e, 1.0), (emap.get(t - 1), -1.0), (ustep(i, s, t - wd.n_min), -1.0),
                        (ustep(i, s, t - wd.n_min - 1), 1.0)], -_INF, 0.0)
            if w.W_MHz_s <= 0:
                continue
            coef = w.C_req * servers[s - 1].f_MHz * dt / w.W_MHz_s
            if wd.n_max == wd.n_min:
                # n_max == n_min: any core block delivers at least W even at the worst DPF
                work_terms.append((X[(i, s)], 1.0))
                continue
            work_terms.append((X[(i, s)], coef * wd.n_min))
            if not derating:
                work_terms += [(e, coef) for e in emap.values()]
                continue
            # derating: core credit n_min - (loss over the core block of the chosen start)
            M = wd.n_min * Lmax
            cl = lp.col(0.0, M)
            work_terms.append((cl, -coef))
            CL = CLcol[s]
            for tau in range(wd.a, wd.b + 1):
                lp.row([(cl, 1.0), (CL[tau + wd.n_min - 2], -1.0),
                        (CL[tau - 2] if tau >= 2 else None, 1.0),
                        (ustep(i, s, tau), -M), (ustep(i, s, tau - 1), M)], -M, _INF)
            # tail credit: wt <= E, wt <= 1 - L
            for t, e in emap.items():
                wt = lp.col(0.0, 1.0)
                lp.row([(wt, 1.0), (e, -1.0)], -_INF, 0.0)
                lp.row([(wt, 1.0), (Lcol[s][t - 1], 1.0)], -_INF, 1.0)
                work_terms.append((wt, coef))
        if w.W_MHz_s > 0:
            lp.row(work_terms, 1.0, _INF)                                               # (39)
    if derating:
        for s in tail_servers:
            sv = servers[s - 1]
            for t in range(1, T + 1):
                # L >= slope * (T - T_dvfs): the linear piece of Eq. (37)
                lp.row([(Lcol[s][t - 1], 1.0), (Tcol[s - 1][t - 1], -slope)], -slope * c.T_dvfs_C, _INF)
                lp.row([(CLcol[s][t - 1], 1.0), (CLcol[s][t - 2] if t >= 2 else None, -1.0),
                        (Lcol[s][t - 1], -1.0)], 0.0, 0.0)

    # ---- priority, Eq. (56)-(57) -------------------------------------------
    # ST[i] = sum_s (b * X[i,s] - sum_{t<b} U[i,s,t]), the start period. For a pair (i, j)
    # with O_ij = 1 and a shared server s:
    #   ST_i - ST_j <= M * (2 - X_is - X_js) + M * started_j(A_i - 1)
    # so i starts no later than j when both run on s and j had not started when i arrived.
    ST: dict[int, int] = {}

    def st_col(i: int) -> int:
        if i not in ST:
            spans = [win[(i, s)] for s in allowed[i] if (i, s) in win]
            col = lp.col(float(min(wd.a for wd in spans)), float(max(wd.b for wd in spans)))
            terms = [(col, 1.0)]
            for s in allowed[i]:
                wd = win.get((i, s))
                if wd is None:
                    continue
                terms.append((X[(i, s)], -float(wd.b)))
                terms += [(uc, 1.0) for uc in U[(i, s)].values()]
            lp.row(terms, 0.0, 0.0)
            ST[i] = col
        return ST[i]

    n_priority_rows = 0
    if settings.priority and RELAX_PRIORITY not in relax:
        for i, j in priority_pairs(scenario):
            A_i = scenario.workloads[i - 1].A
            shared = [s for s in allowed[i] if (i, s) in win and (j, s) in win]
            if not shared:
                continue
            M = (max(win[(i, s)].b for s in allowed[i] if (i, s) in win)
                 - min(win[(j, s)].a for s in allowed[j] if (j, s) in win))
            if M <= 0:
                continue                      # i can never start after j
            started_j = [(ustep(j, s, A_i - 1), -float(M)) for s in allowed[j] if (j, s) in win]
            for s in shared:
                wi, wj = win[(i, s)], win[(j, s)]
                if wj.b < A_i or wi.b <= max(wj.a, A_i):
                    continue                  # no inversion possible on s
                lp.row([(st_col(i), 1.0), (st_col(j), -1.0), (X[(i, s)], float(M)),
                        (X[(j, s)], float(M))] + started_j, -_INF, 2.0 * M)
                n_priority_rows += 1

    # ---- per server, per period -------------------------------------------
    AV = scenario.availability()
    for sv in servers:
        k = sv.s - 1
        dP = sv.P_max_W - sv.P_idle_W
        alpha, R, a_s = sv.th_alpha, sv.th_R_C_per_W, sv.th_a_C
        z = sv.zone
        for t in range(1, T + 1):
            items = by_st.get((sv.s, t), [])
            if not AV[k, t - 1]:
                for _, occ in items:                                                   # (11)
                    lp.row(occ, 0.0, 0.0)
            # (16), (20), (21): P_IT = P_idle + dP * sum C_req * occupancy / C_s
            terms = [(P_IT[k][t - 1], 1.0)]
            cpu: list = []
            mem: list = []
            cpu_sum = mem_sum = 0.0
            for i, occ in items:
                wl = scenario.workloads[i - 1]
                terms += [(col, -dP * wl.C_req / sv.C_cores * v) for col, v in occ]
                cpu += [(col, wl.C_req / sv.C_cores * v) for col, v in occ]
                mem += [(col, wl.M_req_MB / sv.M_MB * v) for col, v in occ]
                cpu_sum += wl.C_req / sv.C_cores
                mem_sum += wl.M_req_MB / sv.M_MB
            lp.row(terms, sv.P_idle_W, sv.P_idle_W)
            if not relax_cap:
                if cpu_sum > 1.0 + 1e-12:                                              # (12)
                    lp.row(cpu, -_INF, 1.0)
                if mem_sum > 1.0 + 1e-12:                                              # (13)
                    lp.row(mem, -_INF, 1.0)
            # temperature recursion (approved)
            terms = [(Tcol[k][t - 1], 1.0), (Tsup[z - 1][t - 1], -(1 - alpha)),
                     (P_IT[k][t - 1], -(1 - alpha) * R), (Q_cp[k][t - 1], (1 - alpha) * R)]
            rhs = (1 - alpha) * a_s
            if t == 1:
                rhs += alpha * sv.T0_C
            else:
                terms.append((Tcol[k][t - 2], -alpha))
            lp.row(terms, rhs, rhs)

    _add_cold_plate_rows(lp, scenario, plates, cp_d_max, cp_cols, Q_cp, P_cp, P_IT, Tcol,
                         cp_neg_max)

    # ---- cooling zones -----------------------------------------------------
    members = {z: [sv for sv in servers if sv.zone == z] for z in range(1, Z + 1)}
    for z in range(1, Z + 1):
        for t in range(1, T + 1):
            # heat balance: Q_crac >= sum (P_IT - Q_cp) / 1000   (A3)
            terms = [(Qcrac[z - 1][t - 1], 1.0)]
            for sv in members[z]:
                terms.append((P_IT[sv.s - 1][t - 1], -1e-3))
                terms.append((Q_cp[sv.s - 1][t - 1], 1e-3))
            lp.row(terms, 0.0, _INF)
            if not fixed_mode:
                ch = choice[(z, t - 1)]
                lp.row([(col, 1.0) for col in ch], 1.0, 1.0)
                lp.row([(Tsup[z - 1][t - 1], 1.0)] + [(col, -float(lv)) for col, lv in zip(ch, levels)],
                       0.0, 0.0)
                qs = qcols[(z, t - 1)]
                lp.row([(Qcrac[z - 1][t - 1], 1.0)] + [(q, -1.0) for q in qs], 0.0, 0.0)
                for q, col in zip(qs, ch):
                    lp.row([(q, 1.0), (col, -Qmax[z - 1])], -_INF, 0.0)
        if not fixed_mode:
            for lag, lim in ((c.ramp_short_lag, c.ramp_short_C), (c.ramp_long_lag, c.ramp_long_C)):
                for t in range(lag + 1, T + 1):                                         # (35), (36)
                    lp.row([(Tsup[z - 1][t - 1], 1.0), (Tsup[z - 1][t - 1 - lag], -1.0)], -lim, lim)

    # ---- objective: P_IT, P_cp, P_fan and CRAC columns already carry CI * energy factors
    built.lp = lp
    built.P_IT, built.Q_cp, built.P_cp, built.P_fan = P_IT, Q_cp, P_cp, P_fan
    built.Tcol, built.Qcrac, built.Tsup = Tcol, Qcrac, Tsup
    built.slope = slope
    built.choice = choice
    built.dpf_cap = dpf_cap
    built.plates, built.cp_cols = plates, cp_cols
    built.n_priority_rows = n_priority_rows
    built.n_unavailable = int((AV == 0).sum())
    return built


def _model_notes(scenario: Scenario, settings: RunSettings, built: _Built) -> list[str]:
    n = [
        "Y[i,s,t] is eliminated: a workload occupies n_min core periods after its start, "
        "U[t] - U[t-n_min], plus optional tail binaries E[i,s,t] only where n_max > n_min, because "
        "the Y-based model made the root LP of 'base' unsolvable (500k rows).",
        "Start indicators S are replaced by cumulative 'started by t' steps U[i,s,t] (S = U[t] - "
        "U[t-1], X[i,s] is the last step), so one unbroken block needs no window sums.",
        "The tail E[t] <= U[t-n_min] - U[t-n_max] and E[t] <= E[t-1] + U[t-n_min] - U[t-n_min-1] keep "
        "the run unbroken and within n_min..n_max periods; Y <= X (10) is then implied and Eq. (9) is kept.",
        "n_min = ceil(W/(C_req*f_s*dt)) is the shortest run since DPF <= 1, and starts with t + n_min > D "
        "are dropped.",
        "n_max = ceil(W/(C_req*f_s*dt*DPF_cap)) caps the run: valid for the optimum because trimming the "
        "tail of a run never raises power, heat or temperature, so every other limit stays met.",
        "DPF enters only through the loss L = 1 - DPF >= slope*(T - T_dvfs), L >= 0 (the linear piece of "
        "Eq. 37, valid below T_throttle, so a cap above T_throttle is rejected), and only on servers where "
        "some pair has n_max > n_min; higher DPF never hurts, so no binaries are needed.",
        "Work (39) uses core credit n_min*X - coreLoss, where coreLoss >= CL[tau+n_min-1] - CL[tau-1] "
        "- M(1 - sigma_tau) over the running sum CL of L, plus tail credits wt <= E, wt <= 1 - L; for a pair "
        "with n_max == n_min any core block delivers at least W, so its term is exactly X.",
        "The work row is divided by W_i (right side 1), and with the cap at or below T_dvfs there are no "
        "loss, tail or work-credit columns at all.",
        "Server utilisation and heat are substituted into P_IT[s,t] = P_idle + (P_max-P_idle)*"
        "sum C_req*occupancy/C_s (Eq. 16, 20, 21); P_IT is bounded by P_max (or by the cores of the "
        "workloads that can ever run there) as a column bound.",
        "Eq. (54) Q_crac <= 116 kW and the big-M of the supply-level split are one column bound "
        "Qmax = min(116, sum of the zone's maximum IT power in kW); the heat balance sets Q_crac >= "
        "heat and the cost increases in Q_crac, so it sits at equality in any optimum.",
        "CRAC power is Q_crac / COP (divide), COP looked up per whole-degree level; the printed "
        "Eq. (46) multiplies.",
        "Fan power P_fan is a named column fixed at zero (_add_fan_terms) that already appears in "
        "the objective.",
        "Idle energy is in the objective through the P_IT columns (lower bound P_idle), so "
        "objective_g equals total carbon with no separate offset.",
        "The wait limit (28) is applied to all workloads, which includes (25). Workloads with "
        "B_i = 1 keep their original server (29).",
    ]
    if settings.priority and RELAX_PRIORITY not in built.relax:
        n.append(
            f"Priority (56)-(57): O_ij = 1 for a time-sensitive i and a delay-tolerant j, and for the "
            f"earlier arrival of two same-type, same-deadline workloads. It is linked to the start "
            f"times as: on a shared server, j may not start while i waits (A_i <= ST_j implies "
            f"ST_i <= ST_j), through ST_i = sum_s (b X - sum U) and big-M rows "
            f"({getattr(built, 'n_priority_rows', 0)} pair-server rows after dropping those that "
            f"cannot bind). EDF within a type is not in the equations and is not modelled.")
    else:
        n.append("Priority (56)-(57) is off for this run: start order is not constrained.")
    n_off = getattr(built, "n_unavailable", 0)
    n.append(f"Availability (11): {n_off} server-period(s) with AV = 0 take no work; their "
             "servers stay on at idle power." if n_off else
             "Availability (11): every server is available in every period (AV = 1).")
    plates = getattr(built, "plates", cold_plate_counts(scenario, settings))
    if plates.any():
        cp = scenario.cold_plate
        n.append(
            f"Cold plates: {int(plates.sum())} on {int((plates > 0).sum())} server(s). "
            f"Each picks one of the flow settings {list(cp.flow_settings_Lmin)} L/min per period "
            "(one-hot), so its resistance and the cubic pump power of Eq. (48) are lookups. "
            "T_prev - coolant is split into d_j <= d_max*b_j, so Q_cp <= n*sum G_j*d_j is linear "
            "(G_j = min(1/R_j, flow_j*c_pw)); Q_cp <= P_IT and Q_cp <= n*0.3 kW. T_prev is the "
            "previous period's temperature, and d_j >= 0 keeps the coolant at or below it.")
        if settings.cp_mode == "fixed":
            flow, cool = settings.optimized_fixed_cp()
            n.append(f"Cold plates fixed (cp_mode fixed): every plate runs at {flow:g} L/min and "
                     f"{cool:g} C coolant in every period. The drive is max(0, T_prev - coolant), "
                     "with one on/off binary per server and period.")
    else:
        n.append("Cold plates are off: Q_cp and P_cp are columns fixed at zero.")
    if built.feasibility_only:
        n.append("Feasibility-only solve (diagnosis): zero objective and the supply fixed at the lowest "
                 "whole degree, which gives the same feasibility as optimising it (lowest temperatures, "
                 "same Q_crac cap, no ramp).")
    if RELAX_THERMAL in built.relax:
        n.append("RELAX_THERMAL: the cap T <= T_max - H_min is replaced by T <= T_throttle, the limit "
                 "of validity of the linear DPF bound, with DPF >= 0.")
    if RELAX_CAPACITY in built.relax:
        n.append("RELAX_CAPACITY: CPU and memory rows are dropped; P_IT is bounded by the cores of the "
                 "workloads that can run on the server.")
    if RELAX_COOLING in built.relax:
        n.append("RELAX_COOLING: Q_crac <= 116 dropped; the column bound is the zone's maximum IT power.")
    if RELAX_DEADLINES in built.relax:
        n.append("RELAX_DEADLINES: starts and runs may extend over A_i..T (arrival still enforced).")
    return n


# ---------------------------------------------------------------------------
# Solving
# ---------------------------------------------------------------------------

def _infeasible_result(built: _Built, relax, t0: float, text: str) -> SolveResult:
    return SolveResult(
        status=STATUS_INFEASIBLE, schedule=None, supply_temp_C=None, objective_g=None,
        best_bound_g=None, mip_gap=None, solve_time_s=time.perf_counter() - t0,
        n_variables=0, n_binaries=0, n_constraints=0, relaxed=tuple(relax),
        solver_status_text=text, notes=list(built.notes))


def _warm_start_values(scenario: Scenario, built: _Built, schedule: Schedule,
                       supply: np.ndarray | None, settings: RunSettings) -> dict[int, float]:
    vals: dict[int, float] = {}
    for p in schedule.placements:
        if p.server is None or p.start is None or p.end is None:
            continue
        wd = built.win.get((p.i, p.server))
        if wd is None or not (wd.a <= p.start <= wd.b):
            continue
        m = min(p.end - p.start, wd.n_max)       # a longer run is cut to n_max (never needed)
        if m < wd.n_min or p.start + m - 1 > wd.yb:
            continue
        for s in range(1, scenario.S + 1):
            xc = built.X.get((p.i, s))
            if xc is None:
                continue
            on = s == p.server
            vals[xc] = 1.0 if on else 0.0
            for t, uc in built.U[(p.i, s)].items():
                vals[uc] = 1.0 if (on and t >= p.start) else 0.0
            for t, ec in built.E[(p.i, s)].items():
                vals[ec] = 1.0 if (on and p.start + wd.n_min <= t < p.start + m) else 0.0
    if built.cp_cols.b:
        lv = (built.cp_cols.fixed[0] if built.cp_cols.fixed is not None
              else scenario.cold_plate.level_of(settings.baseline_cp_flow_Lmin))
        for bk in built.cp_cols.b.values():
            for b in bk:
                for j, col in enumerate(b):
                    vals[col] = 1.0 if j == lv else 0.0
    if supply is not None and built.fixed_value is None:
        supply = np.asarray(supply, dtype=float)
        if supply.shape == (scenario.Z, scenario.T):
            for (z, t), cols in built.choice.items():
                v = supply[z - 1, t]
                k = int(round(v)) if abs(v - round(v)) < 1e-6 else None
                if k in built.levels:
                    for col, lv in zip(cols, built.levels):
                        vals[col] = 1.0 if lv == k else 0.0
    return vals


def _solve_impl(scenario: Scenario, original_server: dict[int, int], settings: RunSettings, *,
                relax: tuple[str, ...], warm_start: Schedule | None,
                warm_start_supply_temp_C: np.ndarray | None, time_limit_s: float | None,
                log: bool, feasibility_only: bool) -> SolveResult:
    relax = tuple(relax)
    t_build0 = time.perf_counter()
    built = _build(scenario, original_server, settings, relax, feasibility_only)
    notes = _model_notes(scenario, settings, built)
    built.notes[:] = notes
    if built.empty_workloads:
        res = _infeasible_result(built, relax, t_build0,
                                 "no feasible server/start window for workload(s) "
                                 + ", ".join(map(str, built.empty_workloads[:10])))
        res.notes.append("Detected before building the model: some workload has no server and start "
                         "period that meets arrival, wait limit, deadline and minimum run length.")
        return res
    lp = built.lp
    h = lp.to_highs(log)
    build_time = time.perf_counter() - t_build0

    limit = settings.time_limit_s if time_limit_s is None else time_limit_s
    h.setOptionValue("time_limit", float(limit))
    h.setOptionValue("mip_rel_gap", 1.0 if feasibility_only else float(settings.mip_rel_gap))
    lp_solver = "choose"
    if sum(lp.integer) > _IPM_BINARIES:
        lp_solver = "ipm"
        if h.setOptionValue("mip_lp_solver", "ipm") != highspy.HighsStatus.kOk:
            raise RuntimeError("HiGHS rejected mip_lp_solver = ipm")
        notes.append(f"More than {_IPM_BINARIES} binaries: the MIP's LP relaxations use interior point "
                     "(mip_lp_solver = ipm) because dual and primal simplex do not finish the root LP "
                     "on large models.")
    h.setOptionValue("log_to_console", bool(log))
    log_path = None
    n_ws_set = 0
    if warm_start is not None and not feasibility_only:
        vals = _warm_start_values(scenario, built, warm_start, warm_start_supply_temp_C, settings)
        if vals:
            idx = np.array(sorted(vals), dtype=np.int32)
            h.setSolution(len(idx), idx, np.array([vals[k] for k in idx], dtype=np.float64))
            n_ws_set = len(idx)
            fd, log_path = tempfile.mkstemp(suffix=".log")
            os.close(fd)
            h.setOptionValue("log_file", log_path)

    t0 = time.perf_counter()
    h.run()
    solve_time = time.perf_counter() - t0
    ws_text = ""
    if log_path is not None:
        try:
            with open(log_path) as f:
                ws_text = f.read()
        finally:
            os.unlink(log_path)

    info = h.getInfo()
    ms = h.getModelStatus()
    text = h.modelStatusToString(ms)
    MS = highspy.HighsModelStatus
    has_sol = info.primal_solution_status == 2
    if ms == MS.kInfeasible or ms == MS.kUnboundedOrInfeasible:
        status = STATUS_INFEASIBLE
    elif ms == MS.kOptimal:
        status = STATUS_OPTIMAL
    elif has_sol:
        status = STATUS_TIME_LIMIT
    else:
        status = STATUS_NO_SOLUTION

    nv, nr = lp.n_cols, lp.n_rows
    nb = int(sum(lp.integer))
    out = SolveResult(status=status, schedule=None, supply_temp_C=None, objective_g=None,
                      best_bound_g=None, mip_gap=None, solve_time_s=solve_time,
                      n_variables=nv, n_binaries=nb, n_constraints=nr, relaxed=relax,
                      solver_status_text=text, notes=notes)
    out.model_values["build_time_s"] = build_time
    out.model_values["mip_lp_solver"] = lp_solver
    if n_ws_set:
        out.model_values["warm_start_columns_set"] = n_ws_set
        out.model_values["warm_start_log"] = "\n".join(
            ln for ln in ws_text.splitlines() if "MIP start" in ln or "solution" in ln.lower())[:2000]
    if status in (STATUS_OPTIMAL, STATUS_TIME_LIMIT) and has_sol:
        x = np.asarray(h.getSolution().col_value)
        out.objective_g = None if feasibility_only else float(info.objective_function_value)
        out.best_bound_g = None if feasibility_only else float(info.mip_dual_bound)
        out.mip_gap = None if feasibility_only else float(info.mip_gap)
        out.schedule = _extract_schedule(scenario, built, x)
        out.supply_temp_C = _extract_supply(scenario, built, x)
        S, Z, T = scenario.S, scenario.Z, scenario.T
        out.model_values["T_C"] = np.array([[x[c_] for c_ in row] for row in built.Tcol])
        Tm = out.model_values["T_C"]
        c = scenario.constants
        out.model_values["DPF"] = np.clip(1.0 - built.slope * (Tm - c.T_dvfs_C), 0.0, 1.0)
        out.model_values["Q_crac_kW"] = np.array([[x[c_] for c_ in row] for row in built.Qcrac])
        out.model_values["P_IT_W"] = np.array([[x[c_] for c_ in row] for row in built.P_IT])
        out.model_values["Q_cp_W"] = np.array([[x[c_] for c_ in row] for row in built.Q_cp])
        out.cold_plate_ops = _extract_cold_plate_ops(scenario, built, x)
    elif status == STATUS_INFEASIBLE:
        pass
    return out


def _extract_schedule(scenario: Scenario, built: _Built, x: np.ndarray) -> Schedule:
    chosen: dict[int, tuple[int, int, int]] = {}
    for (i, s), xc in built.X.items():
        if x[xc] <= 0.5:
            continue
        wd = built.win[(i, s)]
        # start = first period at which the started-by step is 1 (the last step is X itself)
        steps = sorted(t for t, uc in built.U[(i, s)].items() if x[uc] > 0.5)
        start = steps[0] if steps else wd.b
        tail = sum(1 for ec in built.E[(i, s)].values() if x[ec] > 0.5)
        chosen[i] = (s, start, start + wd.n_min + tail)
    pl = []
    for w in scenario.workloads:
        if w.i in chosen:
            s, st, en = chosen[w.i]
            pl.append(Placement(i=w.i, server=s, start=st, end=en))
        else:
            pl.append(Placement(i=w.i, server=None, start=None, end=None))
    return Schedule(placements=tuple(pl))


def _extract_supply(scenario: Scenario, built: _Built, x: np.ndarray) -> np.ndarray:
    Z, T = scenario.Z, scenario.T
    if built.fixed_value is not None:
        return np.full((Z, T), float(built.fixed_value))
    out = np.zeros((Z, T))
    for (z, t), cols in built.choice.items():
        k = int(np.argmax([x[col] for col in cols]))
        out[z - 1, t] = built.levels[k]
    return out


def _extract_cold_plate_ops(scenario: Scenario, built: _Built, x: np.ndarray) -> ColdPlateOps | None:
    if not built.cp_cols.b:
        return None
    flows = scenario.cold_plate.flow_settings_Lmin
    flow = np.full((scenario.S, scenario.T), np.nan)
    cool = np.full((scenario.S, scenario.T), np.nan)
    for k, bk in built.cp_cols.b.items():
        for t, b in enumerate(bk):
            flow[k, t] = flows[int(np.argmax([x[c_] for c_ in b]))]
            cool[k, t] = x[built.cp_cols.cool[k][t]]
    cp = scenario.cold_plate
    cool = np.clip(cool, cp.coolant_min_C, cp.coolant_max_C)    # solver round-off only
    return ColdPlateOps(flow_Lmin=flow, coolant_C=cool)


def solve(scenario: Scenario, original_server: dict[int, int], settings: RunSettings, *,
          relax: tuple[str, ...] = (), warm_start: Schedule | None = None,
          warm_start_supply_temp_C: np.ndarray | None = None,
          time_limit_s: float | None = None, log: bool = False) -> SolveResult:
    """Build and solve the MILP. See the module docstring for the model."""
    return _solve_impl(scenario, original_server, settings, relax=relax, warm_start=warm_start,
                       warm_start_supply_temp_C=warm_start_supply_temp_C,
                       time_limit_s=time_limit_s, log=log, feasibility_only=False)


def diagnose_infeasibility(scenario: Scenario, original_server: dict[int, int],
                           settings: RunSettings, *,
                           time_limit_s: float = 120.0) -> dict[str, str]:
    """Re-solve with one constraint family relaxed at a time (first feasible point is enough).

    Returns {family: "feasible" | "infeasible" | "unknown (time limit)"} for all families.
    ``time_limit_s`` applies to each family separately.
    """
    settings.validate()
    out: dict[str, str] = {}
    for fam in RELAX_FAMILIES:
        if fam == RELAX_PRIORITY and not settings.priority:
            continue
        res = _solve_impl(scenario, original_server, settings, relax=(fam,), warm_start=None,
                          warm_start_supply_temp_C=None, time_limit_s=time_limit_s, log=False,
                          feasibility_only=True)
        if res.status in (STATUS_OPTIMAL, STATUS_TIME_LIMIT):
            out[fam] = "feasible"
        elif res.status == STATUS_INFEASIBLE:
            out[fam] = "infeasible"
        else:
            out[fam] = "unknown (time limit)"
    return out


In [ ]:
%%writefile {LIB}/scheduler/evaluate.py
"""Shared evaluator: the referee that scores any schedule with the same code.

Given a Scenario, a Schedule, supply temperatures and (with cold plates on) the cold-plate
flow and coolant temperatures, it recomputes, from scratch, utilisation, IT power,
cold-plate heat and pump power, server temperature, DPF, work delivered, CRAC heat /
power, energy, carbon, and every compliance check. A schedule is never rejected; every
limit is measured and reported. Equation numbers refer to docs/PROJECT_CONTEXT.md.

This module must not import from baseline.py or optimize.py.
"""

from __future__ import annotations

import numpy as np

from .types import (
    ColdPlateOps, EvalResult, RunSettings, Scenario, Schedule, cold_plate_counts, cop,
    fixed_cold_plate_ops, priority_pairs,
)

TOL = 1e-6
W_TO_KWH_FACTOR = 1.0 / 3.6e6   # W * seconds -> kWh


# ---------------------------------------------------------------------------
# Slot that is not defined yet
# ---------------------------------------------------------------------------

def fan_power_kW(scenario: Scenario, Q_crac_kW: np.ndarray) -> np.ndarray:
    """Fan slot: returns P_fan[z, t] in kW. Airflow is not modelled, so zero."""
    return np.zeros_like(Q_crac_kW)


# ---------------------------------------------------------------------------
# Model pieces
# ---------------------------------------------------------------------------

def dpf(T_C, constants):
    """Eq. (37), all three branches as printed. Works on scalars or arrays."""
    T = np.asarray(T_C, dtype=float)
    c = constants
    middle = 1.0 - c.R_dvfs * (T - c.T_dvfs_C) / (c.T_throttle_C - c.T_dvfs_C)
    out = np.where(T < c.T_dvfs_C, 1.0, np.where(T < c.T_throttle_C, middle, c.F_throttle))
    return out if out.ndim else float(out)


def temperature_recursion(scenario: Scenario, supply_temp_C: np.ndarray,
                          P_IT_W: np.ndarray, Q_cp_W: np.ndarray) -> np.ndarray:
    """T[s,t] = alpha*T[s,t-1] + (1-alpha)*(Tsup[zone,t] + a + R*(P_IT - Q_cp))."""
    S, T = scenario.S, scenario.T
    temp = np.zeros((S, T))
    for sv in scenario.servers:
        k = sv.s - 1
        prev = sv.T0_C
        for t in range(T):
            steady = (supply_temp_C[sv.zone - 1, t] + sv.th_a_C
                      + sv.th_R_C_per_W * (P_IT_W[k, t] - Q_cp_W[k, t]))
            prev = sv.th_alpha * prev + (1.0 - sv.th_alpha) * steady
            temp[k, t] = prev
    return temp


def thermal_with_cold_plates(scenario: Scenario, supply_temp_C: np.ndarray, P_IT_W: np.ndarray,
                             plates: np.ndarray, ops: ColdPlateOps):
    """Temperature recursion with cold plates. Returns (T, Q_cp, P_cp), each (S, T) in degC / W.

    Per server and period, with n plates at flow setting k and coolant temperature c:
    Q_cp = min(n * Q_max, P_IT, n * G_k * max(0, T[s,t-1] - c)), the most heat physics and
    the limits allow, using the previous period's temperature (T0 in period 1). Pump power
    is n * pump_k. Servers without cold plates have Q_cp = P_cp = 0.
    """
    cp = scenario.cold_plate
    G = cp.conductance_W_per_C()
    pump = cp.pump_W()
    S, T = scenario.S, scenario.T
    temp = np.zeros((S, T))
    Q_cp = np.zeros((S, T))
    P_cp = np.zeros((S, T))
    for sv in scenario.servers:
        k = sv.s - 1
        n = int(plates[k])
        prev = sv.T0_C
        for t in range(T):
            if n > 0:
                lv = cp.level_of(float(ops.flow_Lmin[k, t]))
                drive = max(0.0, prev - float(ops.coolant_C[k, t]))
                Q_cp[k, t] = min(n * cp.Q_max_per_plate_W, P_IT_W[k, t], n * G[lv] * drive)
                P_cp[k, t] = n * pump[lv]
            steady = (supply_temp_C[sv.zone - 1, t] + sv.th_a_C
                      + sv.th_R_C_per_W * (P_IT_W[k, t] - Q_cp[k, t]))
            prev = sv.th_alpha * prev + (1.0 - sv.th_alpha) * steady
            temp[k, t] = prev
    return temp, Q_cp, P_cp


def _placement_arrays(scenario: Scenario, schedule: Schedule):
    """Per workload: placed flag, server (1-based), start, end (ints; 0 if unplaced)."""
    I, S = scenario.I, scenario.S
    placed = np.zeros(I, dtype=bool)
    srv = np.zeros(I, dtype=int)
    start = np.zeros(I, dtype=int)
    end = np.zeros(I, dtype=int)
    by_i = {p.i: p for p in schedule.placements}
    for w in scenario.workloads:
        p = by_i.get(w.i)
        if p is None or p.server is None or p.start is None or p.end is None:
            continue
        if not (1 <= p.server <= S):
            continue    # a server that does not exist is treated as not placed
        placed[w.i - 1] = True
        srv[w.i - 1], start[w.i - 1], end[w.i - 1] = p.server, p.start, p.end
    return placed, srv, start, end


def priority_violations(scenario: Scenario, placed, srv, start) -> list[tuple[int, int]]:
    """Pairs (i, j) of priority_pairs where j started on i's server while i was waiting:
    A_i <= ST_j < ST_i, Eq. (56)-(57)."""
    out = []
    for i, j in priority_pairs(scenario):
        a, b = i - 1, j - 1
        if not (placed[a] and placed[b]) or srv[a] != srv[b]:
            continue
        if scenario.workloads[a].A <= start[b] < start[a]:
            out.append((i, j))
    return out


def _f(x) -> float:
    return float(x)


def _none_if_nan(x):
    x = float(x)
    return None if np.isnan(x) else x


# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------

def evaluate(scenario: Scenario, schedule: Schedule, supply_temp_C: np.ndarray,
             settings: RunSettings, cold_plate_ops: ColdPlateOps | None = None) -> EvalResult:
    """Score a schedule. ``cold_plate_ops`` gives the cold-plate flow and coolant temperature
    per server and period; None runs every cold plate at the baseline's fixed settings."""
    settings.validate()
    S, Z, T, I = scenario.S, scenario.Z, scenario.T, scenario.I
    c = scenario.constants
    dt = scenario.dt_seconds
    supply = np.asarray(supply_temp_C, dtype=float)
    if supply.shape != (Z, T):
        raise ValueError(f"supply_temp_C must have shape (Z, T) = {(Z, T)}, got {supply.shape}")
    cap = settings.thermal_cap_C(scenario)

    placed, srv, start, end = _placement_arrays(scenario, schedule)

    # Which periods does each placed workload occupy inside the horizon (0-based)?
    cpu_used = np.zeros((S, T))
    mem_used = np.zeros((S, T))
    run_idx: list[np.ndarray] = []
    for w in scenario.workloads:
        k = w.i - 1
        if not placed[k]:
            run_idx.append(np.zeros(0, dtype=int))
            continue
        lo = max(start[k], 1)
        hi = min(end[k] - 1, T)           # last occupied period (1-based), clipped
        idx = np.arange(lo - 1, hi) if hi >= lo else np.zeros(0, dtype=int)
        run_idx.append(idx)
        cpu_used[srv[k] - 1, idx] += w.C_req
        mem_used[srv[k] - 1, idx] += w.M_req_MB

    C_cores = np.array([sv.C_cores for sv in scenario.servers], dtype=float)
    M_MB = np.array([sv.M_MB for sv in scenario.servers], dtype=float)
    P_idle = np.array([sv.P_idle_W for sv in scenario.servers])
    P_max = np.array([sv.P_max_W for sv in scenario.servers])
    f_MHz = np.array([sv.f_MHz for sv in scenario.servers])
    zone_of = np.array([sv.zone for sv in scenario.servers], dtype=int)

    util = cpu_used / C_cores[:, None]                                    # Eq. (16)
    P_IT = P_idle[:, None] + (P_max - P_idle)[:, None] * util            # Eq. (20), (21)
    plates = cold_plate_counts(scenario, settings)
    has_cp = plates > 0
    if has_cp.any():
        ops = cold_plate_ops or fixed_cold_plate_ops(
            scenario, plates, settings.baseline_cp_flow_Lmin, settings.baseline_cp_coolant_C)
        for name in ("flow_Lmin", "coolant_C"):
            if np.shape(getattr(ops, name)) != (S, T):
                raise ValueError(f"cold_plate_ops.{name} must have shape (S, T) = {(S, T)}")
        temp, Q_cp, P_cp = thermal_with_cold_plates(scenario, supply, P_IT, plates, ops)
        cp_flow = np.where(has_cp[:, None], ops.flow_Lmin, np.nan)
        cp_coolant = np.where(has_cp[:, None], ops.coolant_C, np.nan)
    else:
        Q_cp, P_cp = np.zeros_like(P_IT), np.zeros_like(P_IT)
        temp = temperature_recursion(scenario, supply, P_IT, Q_cp)
        cp_flow = cp_coolant = np.full((S, T), np.nan)
    headroom = c.T_max_server_C - temp                                   # Eq. (31)
    DPF = dpf(temp, c)                                                   # Eq. (37)
    PC_eff = C_cores[:, None] * f_MHz[:, None] * DPF                     # Eq. (38)

    # Zones: heat balance at equality, divide by COP.
    Q_crac = np.zeros((Z, T))
    for z in range(1, Z + 1):
        members = zone_of == z
        Q_crac[z - 1] = (P_IT[members] - Q_cp[members]).sum(axis=0) / 1000.0
    COP = cop(c, supply)
    P_crac = Q_crac / COP
    P_fan = fan_power_kW(scenario, Q_crac)

    # Energy and carbon per period.
    E_IT = P_IT.sum(axis=0) * dt * W_TO_KWH_FACTOR
    E_cp = P_cp.sum(axis=0) * dt * W_TO_KWH_FACTOR
    E_crac = P_crac.sum(axis=0) * dt / 3600.0
    E_fan = P_fan.sum(axis=0) * dt / 3600.0
    E_total = E_IT + E_crac + E_cp + E_fan
    carbon = scenario.CI_g_per_kWh * E_total
    total_carbon = float(carbon.sum())

    # Per workload.
    nan = float("nan")
    wait = np.full(I, nan)
    delivered = np.full(I, nan)
    required = np.array([w.W_MHz_s for w in scenario.workloads], dtype=float)
    starts_ok = np.zeros(I, dtype=bool)
    wait_ok = np.zeros(I, dtype=bool)
    on_time = np.zeros(I, dtype=bool)
    work_ok = np.zeros(I, dtype=bool)
    within_horizon = np.zeros(I, dtype=bool)
    min_dpf_run = np.full(I, nan)
    for w in scenario.workloads:
        k = w.i - 1
        if not placed[k]:
            continue
        wait[k] = start[k] - w.A
        starts_ok[k] = start[k] >= w.A
        wait_ok[k] = wait[k] <= w.AD + TOL
        on_time[k] = end[k] <= w.D
        idx = run_idx[k]
        delivered[k] = float(w.C_req * f_MHz[srv[k] - 1] * dt * DPF[srv[k] - 1, idx].sum())
        if len(idx):
            min_dpf_run[k] = float(DPF[srv[k] - 1, idx].min())
        work_ok[k] = delivered[k] >= w.W_MHz_s * (1.0 - 1e-6)
        within_horizon[k] = start[k] >= 1 and end[k] <= T + 1 and end[k] > start[k]
    compliant = placed & starts_ok & wait_ok & on_time & work_ok & within_horizon

    # ---- compliance checks ------------------------------------------------
    cpu_ok = bool(np.all(cpu_used <= C_cores[:, None] + TOL))
    mem_ok = bool(np.all(mem_used <= M_MB[:, None] + TOL))
    thermal_cap_ok = bool(np.all(temp <= cap + TOL))
    workloads_ok = bool(np.all(compliant))
    supply_range_ok = bool(np.all(supply >= c.T_crac_min_C - TOL)
                           and np.all(supply <= c.T_crac_max_C + TOL))
    zone_temp_ok = bool(np.all(supply <= c.T_max_zone_C + TOL))

    def max_ramp(lag: int) -> float:
        if T <= lag:
            return 0.0
        return float(np.abs(supply[:, lag:] - supply[:, :-lag]).max())

    ramp_short = max_ramp(c.ramp_short_lag)
    ramp_long = max_ramp(c.ramp_long_lag)
    ramp_ok = bool(ramp_short <= c.ramp_short_C + TOL and ramp_long <= c.ramp_long_C + TOL)
    crac_capacity_ok = bool(np.all(Q_crac <= c.Q_crac_kW + TOL))
    cps = scenario.cold_plate
    cp_coolant_ok = bool(np.all((cp_coolant[has_cp] >= cps.coolant_min_C - TOL)
                                & (cp_coolant[has_cp] <= cps.coolant_max_C + TOL)))
    cp_flow_ok = bool(np.all((cp_flow[has_cp] >= cps.flow_min_Lmin - TOL)
                             & (cp_flow[has_cp] <= cps.flow_max_Lmin + TOL)))
    AV = scenario.availability()
    unavailable_used = (AV == 0) & (cpu_used > 0)
    availability_ok = not bool(unavailable_used.any())                   # Eq. (11)
    prio_bad = priority_violations(scenario, placed, srv, start)
    priority_ok = not prio_bad                                           # Eq. (56)-(57)

    above_dvfs = temp > c.T_dvfs_C
    above_cap = temp > cap + TOL

    servers_detail = []
    for sv in scenario.servers:
        k = sv.s - 1
        servers_detail.append({
            "s": sv.s, "name": sv.name, "zone": sv.zone, "cold_plate": int(sv.CT_cold_plate),
            "n_cold_plates": int(plates[k]),
            "Q_cp_peak_W": _f(Q_cp[k].max()), "Q_cp_capacity_W": _f(plates[k] * cps.Q_max_per_plate_W),
            "cpu_peak_cores": _f(cpu_used[k].max()), "cpu_capacity_cores": _f(sv.C_cores),
            "max_utilization": _f(util[k].max()),
            "mem_peak_MB": _f(mem_used[k].max()), "mem_capacity_MB": _f(sv.M_MB),
            "hottest_T_C": _f(temp[k].max()), "min_headroom_C": _f(headroom[k].min()),
            "periods_above_dvfs": int(above_dvfs[k].sum()),
            "periods_above_cap": int(above_cap[k].sum()),
            "min_DPF": _f(DPF[k].min()), "min_PC_eff_MHz": _f(PC_eff[k].min()),
            "periods_unavailable": int((AV[k] == 0).sum()),
        })
    cold_plates_detail = []
    for sv in scenario.servers:
        k = sv.s - 1
        if not has_cp[k]:
            continue
        cold_plates_detail.append({
            "s": sv.s, "name": sv.name, "n_cold_plates": int(plates[k]),
            "coolant_min_C": _f(cp_coolant[k].min()), "coolant_max_C": _f(cp_coolant[k].max()),
            "coolant_allowed_min_C": _f(cps.coolant_min_C),
            "coolant_allowed_max_C": _f(cps.coolant_max_C),
            "flow_min_Lmin": _f(cp_flow[k].min()), "flow_max_Lmin": _f(cp_flow[k].max()),
            "flow_allowed_min_Lmin": _f(cps.flow_min_Lmin),
            "flow_allowed_max_Lmin": _f(cps.flow_max_Lmin),
            "Q_cp_peak_W": _f(Q_cp[k].max()),
            "Q_cp_capacity_W": _f(plates[k] * cps.Q_max_per_plate_W),
        })
    zones_detail = []
    for z in range(1, Z + 1):
        zones_detail.append({
            "z": z,
            "supply_min_C": _f(supply[z - 1].min()), "supply_max_C": _f(supply[z - 1].max()),
            "supply_allowed_min_C": _f(c.T_crac_min_C), "supply_allowed_max_C": _f(c.T_crac_max_C),
            "zone_limit_C": _f(c.T_max_zone_C),
            "max_ramp_short_C": (float(np.abs(supply[z - 1, c.ramp_short_lag:]
                                              - supply[z - 1, :-c.ramp_short_lag]).max())
                                 if T > c.ramp_short_lag else 0.0),
            "ramp_short_limit_C": _f(c.ramp_short_C),
            "max_ramp_long_C": (float(np.abs(supply[z - 1, c.ramp_long_lag:]
                                             - supply[z - 1, :-c.ramp_long_lag]).max())
                                if T > c.ramp_long_lag else 0.0),
            "ramp_long_limit_C": _f(c.ramp_long_C),
            "Q_crac_peak_kW": _f(Q_crac[z - 1].max()), "Q_crac_capacity_kW": _f(c.Q_crac_kW),
        })

    checks = {"cpu_ok": cpu_ok, "mem_ok": mem_ok, "thermal_cap_ok": thermal_cap_ok,
              "workloads_ok": workloads_ok, "supply_range_ok": supply_range_ok,
              "zone_temp_ok": zone_temp_ok, "ramp_ok": ramp_ok,
              "crac_capacity_ok": crac_capacity_ok, "cold_plate_coolant_ok": cp_coolant_ok,
              "cold_plate_flow_ok": cp_flow_ok, "availability_ok": availability_ok,
              "priority_ok": priority_ok}
    # Priority is measured either way but is only a limit when the run enforces it.
    binding = {k: v for k, v in checks.items() if settings.priority or k != "priority_ok"}

    violations: list[str] = []
    if not cpu_ok:
        bad = [d for d in servers_detail if d["cpu_peak_cores"] > d["cpu_capacity_cores"] + TOL]
        violations.append("CPU capacity exceeded on server(s) "
                          + ", ".join(str(d["s"]) for d in bad) + " (Eq. 12)")
    if not mem_ok:
        bad = [d for d in servers_detail if d["mem_peak_MB"] > d["mem_capacity_MB"] + TOL]
        violations.append("memory capacity exceeded on server(s) "
                          + ", ".join(str(d["s"]) for d in bad) + " (Eq. 13)")
    if not thermal_cap_ok:
        violations.append(f"server temperature above thermal cap {cap:g} C in "
                          f"{int(above_cap.sum())} server-periods (Eq. 31-33), "
                          f"hottest {temp.max():.2f} C")
    if not workloads_ok:
        n_unp = int((~placed).sum())
        parts = []
        if n_unp:
            parts.append(f"{n_unp} unplaced")
        for label, flag in (("starts before arrival", ~starts_ok), ("wait exceeds allowed delay", ~wait_ok),
                            ("finishes after deadline", ~on_time), ("work short", ~work_ok),
                            ("runs outside horizon", ~within_horizon)):
            n = int((placed & flag).sum())
            if n:
                parts.append(f"{n} {label}")
        violations.append(f"{int((~compliant).sum())} non-compliant workload(s): " + "; ".join(parts))
    if not supply_range_ok:
        violations.append(f"supply temperature outside [{c.T_crac_min_C:g}, {c.T_crac_max_C:g}] C (Eq. 42)")
    if not zone_temp_ok:
        violations.append(f"zone temperature above {c.T_max_zone_C:g} C (Eq. 34)")
    if not ramp_ok:
        violations.append(f"supply ramp too steep: short {ramp_short:.2f} C (limit {c.ramp_short_C:g}), "
                          f"long {ramp_long:.2f} C (limit {c.ramp_long_C:g}) (Eq. 35-36)")
    if not crac_capacity_ok:
        violations.append(f"CRAC heat above capacity {c.Q_crac_kW:g} kW, peak {Q_crac.max():.2f} kW (Eq. 54)")
    if not cp_coolant_ok:
        violations.append(f"cold-plate coolant temperature outside [{cps.coolant_min_C:g}, "
                          f"{cps.coolant_max_C:g}] C (Eq. 49)")
    if not cp_flow_ok:
        violations.append(f"cold-plate flow outside [{cps.flow_min_Lmin:g}, {cps.flow_max_Lmin:g}] "
                          f"L/min (Eq. 47)")
    if not availability_ok:
        bad = sorted({int(k) + 1 for k in np.nonzero(unavailable_used)[0]})
        violations.append(f"work runs on unavailable server-periods ({int(unavailable_used.sum())}, "
                          f"server(s) {', '.join(map(str, bad))}) (Eq. 11)")
    if not priority_ok and settings.priority:
        ex = ", ".join(f"{scenario.workloads[j - 1].workload_id} before "
                       f"{scenario.workloads[i - 1].workload_id}" for i, j in prio_bad[:3])
        violations.append(f"{len(prio_bad)} priority inversion(s): a lower-priority workload "
                          f"started on a server while a higher-priority one waited for it, "
                          f"e.g. {ex} (Eq. 56-57)")

    compliance = {
        "all_ok": all(binding.values()), **checks,
        "servers": servers_detail, "zones": zones_detail, "cold_plates": cold_plates_detail,
        "priority_violations": [list(x) for x in prio_bad],
        "thermal_cap_C": _f(cap), "violations": violations,
        "within_horizon": [bool(x) for x in within_horizon],
    }

    # ---- totals -----------------------------------------------------------
    rt = np.array([w.RT for w in scenario.workloads])
    placed_rt = placed & (rt == 1)
    placed_dt = placed & (rt == 0)

    def mean_max(mask):
        if not mask.any():
            return None, None
        return float(wait[mask].mean()), float(wait[mask].max())

    mean_rt, max_rt = mean_max(placed_rt)
    mean_dt, max_dt = mean_max(placed_dt)
    deadline = np.array([w.D for w in scenario.workloads])
    late = (~placed) | (placed & (end > deadline))
    k_hot, t_hot = np.unravel_index(int(np.argmax(temp)), temp.shape)
    totals = {
        "E_IT_kWh": _f(E_IT.sum()), "E_crac_kWh": _f(E_crac.sum()),
        "E_cp_kWh": _f(E_cp.sum()), "E_fan_kWh": _f(E_fan.sum()),
        "Q_cp_kWh": _f(Q_cp.sum() * dt * W_TO_KWH_FACTOR),
        "E_total_kWh": _f(E_total.sum()),
        "carbon_g": total_carbon, "carbon_kg": total_carbon / 1000.0,
        "n_workloads": int(I), "n_unplaced": int((~placed).sum()),
        "n_late": int(late.sum()),
        "n_wait_exceeded": int((placed & ~wait_ok).sum()),
        "n_work_short": int((placed & ~work_ok).sum()),
        "n_noncompliant": int((~compliant).sum()),
        "mean_wait_RT": mean_rt, "max_wait_RT": max_rt,
        "mean_wait_DT": mean_dt, "max_wait_DT": max_dt,
        "hottest_T_C": _f(temp.max()), "hottest_server": int(k_hot + 1),
        "hottest_period": int(t_hot + 1),
        "min_headroom_C": _f(headroom.min()),
        "periods_above_dvfs": int(above_dvfs.sum()),
        "periods_above_cap": int(above_cap.sum()),
        "min_DPF": _f(DPF.min()),
        "n_throttled_servers": int((DPF < 1.0 - 1e-12).any(axis=1).sum()),
        "mean_T_C": _f(temp.mean()),
        "mean_headroom_C": _f(headroom.mean()),
        "n_priority_violations": len(prio_bad),
        "E_crac_by_zone_kWh": [_f(P_crac[z].sum() * dt / 3600.0) for z in range(Z)],
    }

    return EvalResult(
        cpu_used_cores=cpu_used, mem_used_MB=mem_used, util=util, P_IT_W=P_IT,
        Q_cp_W=Q_cp, P_cp_W=P_cp, T_C=temp, headroom_C=headroom, DPF=DPF, PC_eff_MHz=PC_eff,
        T_supply_C=supply.copy(), Q_crac_kW=Q_crac, COP=COP, P_crac_kW=P_crac, P_fan_kW=P_fan,
        E_IT_kWh=E_IT, E_crac_kWh=E_crac, E_cp_kWh=E_cp, E_fan_kWh=E_fan,
        E_total_kWh=E_total, carbon_g=carbon,
        wait_periods=wait, work_delivered_MHz_s=delivered, work_required_MHz_s=required,
        placed=placed, starts_on_or_after_arrival=starts_ok, wait_within_allowed=wait_ok,
        on_time=on_time, work_complete=work_ok, compliant=compliant, min_DPF_run=min_dpf_run,
        total_carbon_g=total_carbon, totals=totals, compliance=compliance,
        airflow_modelled=False,
        n_cold_plates=plates, cp_flow_Lmin=cp_flow, cp_coolant_C=cp_coolant,
    )


In [ ]:
%%writefile {LIB}/scheduler/report.py
"""Result writer: CSV tables, summary.json and the printed comparison table.

Computes no model numbers of its own: everything comes from EvalResult, SolveResult
and the inputs (apart from differences and percentages between the two columns, and
a count of workloads whose server differs from the original one).
"""

from __future__ import annotations

import csv
import json
import math
from pathlib import Path

import numpy as np

from .types import EvalResult, RunSettings, Scenario, Schedule, SolveResult

AIRFLOW_TEXT = "not modelled"
AIRFLOW_NOTE = "Airflow and fan power are not modelled (fan power and airflow are zero)."
NO_COLD_PLATE_NOTE = ("Cold plates are off (cold-plate heat removal and pump power are zero; "
                      "run with --cold-plates to model them).")
# Flagged with the thesis group: explain in the writeup, revisit after the first prototype.
COLD_PLATE_FLAG = ("FLAGGED: cold-plate flow and coolant temperature carry almost no cost (pump "
                   "power is at most about 1 mW per plate, and no equation charges energy for "
                   "cooling the water), so the optimizer runs them wherever they remove the most "
                   "heat. Where the 0.3 kW limit or the server's own heat is what limits removal, "
                   "the reported flow and coolant values are arbitrary.")


def model_note(settings: RunSettings) -> str:
    return AIRFLOW_NOTE + " " + (COLD_PLATE_FLAG if settings.cold_plates else NO_COLD_PLATE_NOTE)


def _cold_plate_block(scenario: Scenario, settings: RunSettings, ev: EvalResult) -> dict:
    cp = scenario.cold_plate
    return to_jsonable({
        "enabled": bool(settings.cold_plates),
        "plates_per_server": ev.n_cold_plates,
        "count_source": ("not modelled" if not settings.cold_plates else
                         "servers.csv" if settings.cold_plate_count is None else
                         f"--cold-plate-count {settings.cold_plate_count}"),
        "flow_settings_Lmin": list(cp.flow_settings_Lmin),
        "R_C_per_W": list(cp.R_C_per_W),
        "conductance_W_per_C": cp.conductance_W_per_C(),
        "pump_W_per_plate": cp.pump_W(),
        "flow_limits_Lmin": [cp.flow_min_Lmin, cp.flow_max_Lmin],
        "coolant_limits_C": [cp.coolant_min_C, cp.coolant_max_C],
        "Q_max_per_plate_W": cp.Q_max_per_plate_W,
        "baseline_flow_Lmin": settings.baseline_cp_flow_Lmin,
        "baseline_coolant_C": settings.baseline_cp_coolant_C,
        "optimized_mode": settings.cp_mode,
        "optimized_fixed_flow_Lmin": settings.optimized_fixed_cp()[0] if settings.cp_mode == "fixed" else None,
        "optimized_fixed_coolant_C": settings.optimized_fixed_cp()[1] if settings.cp_mode == "fixed" else None,
        "flag": COLD_PLATE_FLAG if settings.cold_plates else None,
    })


# ---------------------------------------------------------------------------
# JSON / CSV helpers
# ---------------------------------------------------------------------------

def to_jsonable(x):
    """Convert numpy scalars/arrays and tuples to plain Python; NaN and inf -> None."""
    if isinstance(x, dict):
        return {str(k): to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple, set)):
        return [to_jsonable(v) for v in x]
    if isinstance(x, np.ndarray):
        return to_jsonable(x.tolist())
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, (float, np.floating)):
        v = float(x)
        return None if (math.isnan(v) or math.isinf(v)) else v
    return x


def _cell(v):
    if v is None:
        return ""
    if isinstance(v, (bool, np.bool_)):
        return int(bool(v))
    if isinstance(v, (float, np.floating)):
        return "" if math.isnan(float(v)) else repr(float(v))
    if isinstance(v, np.integer):
        return int(v)
    return v


def _write_csv(path: Path, header: list[str], rows) -> None:
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header)
        for row in rows:
            w.writerow([_cell(v) for v in row])


def _type_name(RT: int) -> str:
    return "time_sensitive" if RT == 1 else "delay_tolerant"


# ---------------------------------------------------------------------------
# Per-schedule tables
# ---------------------------------------------------------------------------

def _count_reassigned(schedule: Schedule, original_server: dict) -> int:
    n = 0
    for p in schedule.placements:
        if p.server is not None and p.server != original_server.get(p.i):
            n += 1
    return n


def _count_cross_zone(scenario: Scenario, schedule: Schedule, original_server: dict) -> int:
    zone = {sv.s: sv.zone for sv in scenario.servers}
    return sum(1 for p in schedule.placements
               if p.server is not None and original_server.get(p.i) is not None
               and zone[p.server] != zone[original_server[p.i]])


def _move_reason(scenario: Scenario, ev: EvalResult, w, orig: int, final: int,
                 start: int, end: int) -> str:
    """Post-hoc reading of one move, from the scored schedule. The solver only minimises
    carbon; these are the local conditions that favoured the new server."""
    so, sf = scenario.servers[orig - 1], scenario.servers[final - 1]
    idx = np.arange(max(start, 1) - 1, min(end - 1, scenario.T))
    out = []
    if len(idx) and (np.any(ev.cpu_used_cores[orig - 1, idx] + w.C_req > so.C_cores + 1e-9)
                     or np.any(ev.mem_used_MB[orig - 1, idx] + w.M_req_MB > so.M_MB + 1e-9)):
        out.append("original server full at the chosen start")
    if sf.f_MHz > so.f_MHz:
        out.append("faster server")

    def it_energy(sv) -> float:   # extra IT energy of this workload on sv, W x periods
        n = max(1, math.ceil(w.W_MHz_s / (w.C_req * sv.f_MHz * scenario.dt_seconds) - 1e-9))
        return (sv.P_max_W - sv.P_idle_W) * w.C_req / sv.C_cores * n
    if it_energy(sf) < it_energy(so) - 1e-9:
        out.append("less IT energy for its work")
    if len(idx) and ev.T_C[final - 1, idx].mean() < ev.T_C[orig - 1, idx].mean() - 1e-9:
        out.append("cooler server")
    if len(idx) and sf.zone != so.zone and \
            ev.COP[sf.zone - 1, idx].mean() > ev.COP[so.zone - 1, idx].mean() + 1e-9:
        out.append("zone with higher CRAC COP")
    return "; ".join(out) or "carbon trade-off across the fleet (no single local cause)"


def _write_reassignment(path: Path, scenario: Scenario, settings: RunSettings,
                        schedule: Schedule, ev: EvalResult, original_server: dict) -> None:
    """One row per workload: original and final server and zone, and why it moved (or not)."""
    by_i = {p.i: p for p in schedule.placements}
    zone = {sv.s: sv.zone for sv in scenario.servers}
    rows = []
    for w in scenario.workloads:
        p = by_i.get(w.i)
        orig = original_server.get(w.i)
        placed = bool(ev.placed[w.i - 1]) and p is not None
        final = p.server if placed else None
        moved = placed and orig is not None and final != orig
        if not placed:
            reason = "not placed"
        elif moved:
            reason = _move_reason(scenario, ev, w, orig, final, p.start, p.end)
        elif w.B == 1:
            reason = "running at the start (B_i = 1): stays on its server, Eq. (29)"
        elif settings.reassignment == "none":
            reason = "reassignment off"
        else:
            reason = "kept"
        rows.append([
            w.i, w.workload_id, _type_name(w.RT), w.B, orig,
            zone.get(orig) if orig is not None else None, final,
            zone.get(final) if final is not None else None, moved,
            bool(moved and zone[orig] != zone[final]), reason])
    _write_csv(path, ["i", "workload_id", "type", "B_running", "original_server", "original_zone",
                      "final_server", "final_zone", "reassigned", "cross_zone", "reason"], rows)


def _write_schedule_tables(folder: Path, scenario: Scenario, schedule: Schedule,
                           ev: EvalResult, original_server: dict) -> None:
    folder.mkdir(parents=True, exist_ok=True)
    by_i = {p.i: p for p in schedule.placements}

    rows = []
    for w in scenario.workloads:
        k = w.i - 1
        p = by_i.get(w.i)
        placed = bool(ev.placed[k])
        orig = original_server.get(w.i)
        server = p.server if (p is not None and placed) else None
        start = p.start if (p is not None and placed) else None
        end = p.end if (p is not None and placed) else None
        rows.append([
            w.i, w.workload_id, _type_name(w.RT), w.A, orig, server,
            (server != orig) if placed else None, start,
            (end - start) if placed else None, end,
            ev.wait_periods[k], w.AD, w.D,
            ev.work_required_MHz_s[k], ev.work_delivered_MHz_s[k],
            ev.compliant[k], ev.placed[k], ev.starts_on_or_after_arrival[k],
            ev.wait_within_allowed[k], ev.on_time[k], ev.work_complete[k], w.B,
        ])
    _write_csv(folder / "schedule.csv",
               ["i", "workload_id", "type", "arrival", "original_server", "server", "reassigned",
                "start", "periods_run", "finish", "wait", "allowed_delay", "deadline",
                "work_required_MHz_s", "work_delivered_MHz_s", "compliant", "placed",
                "starts_on_or_after_arrival", "wait_within_allowed", "on_time", "work_complete",
                "B_running"],
               rows)

    T = scenario.T
    _write_csv(folder / "server_period.csv",
               ["s", "t", "cpu_used_cores", "mem_used_MB", "utilization", "P_IT_W", "Q_cp_W",
                "cp_flow_Lmin", "cp_coolant_C", "P_cp_W",
                "temperature_C", "headroom_C", "DPF", "PC_eff_MHz"],
               ([s + 1, t + 1, ev.cpu_used_cores[s, t], ev.mem_used_MB[s, t], ev.util[s, t],
                 ev.P_IT_W[s, t], ev.Q_cp_W[s, t], ev.cp_flow_Lmin[s, t], ev.cp_coolant_C[s, t],
                 ev.P_cp_W[s, t], ev.T_C[s, t], ev.headroom_C[s, t],
                 ev.DPF[s, t], ev.PC_eff_MHz[s, t]]
                for s in range(scenario.S) for t in range(T)))
    _write_csv(folder / "zone_period.csv",
               ["z", "t", "supply_temp_C", "Q_crac_kW", "COP", "P_crac_kW", "P_fan_kW", "airflow"],
               ([z + 1, t + 1, ev.T_supply_C[z, t], ev.Q_crac_kW[z, t], ev.COP[z, t],
                 ev.P_crac_kW[z, t], ev.P_fan_kW[z, t], AIRFLOW_TEXT]
                for z in range(scenario.Z) for t in range(T)))
    _write_csv(folder / "period.csv",
               ["t", "E_IT_kWh", "E_crac_kWh", "E_cp_kWh", "E_fan_kWh", "E_total_kWh",
                "CI_g_per_kWh", "carbon_g"],
               ([t + 1, ev.E_IT_kWh[t], ev.E_crac_kWh[t], ev.E_cp_kWh[t], ev.E_fan_kWh[t],
                 ev.E_total_kWh[t], scenario.CI_g_per_kWh[t], ev.carbon_g[t]] for t in range(T)))

    comp = ev.compliance
    srv_cols = ["s", "name", "zone", "cold_plate", "n_cold_plates", "Q_cp_peak_W", "Q_cp_capacity_W",
                "cpu_peak_cores", "cpu_capacity_cores", "max_utilization",
                "mem_peak_MB", "mem_capacity_MB", "hottest_T_C", "min_headroom_C",
                "periods_above_dvfs", "periods_above_cap", "min_DPF", "min_PC_eff_MHz",
                "periods_unavailable"]
    _write_csv(folder / "compliance_servers.csv", srv_cols,
               ([d[c] for c in srv_cols] for d in comp["servers"]))
    zone_cols = ["z", "supply_min_C", "supply_max_C", "supply_allowed_min_C",
                 "supply_allowed_max_C", "zone_limit_C", "max_ramp_short_C", "ramp_short_limit_C",
                 "max_ramp_long_C", "ramp_long_limit_C", "Q_crac_peak_kW", "Q_crac_capacity_kW"]
    _write_csv(folder / "compliance_zones.csv", zone_cols,
               ([d[c] for c in zone_cols] for d in comp["zones"]))
    cp_cols = ["s", "name", "n_cold_plates", "coolant_min_C", "coolant_max_C",
               "coolant_allowed_min_C", "coolant_allowed_max_C", "flow_min_Lmin", "flow_max_Lmin",
               "flow_allowed_min_Lmin", "flow_allowed_max_Lmin", "Q_cp_peak_W", "Q_cp_capacity_W"]
    _write_csv(folder / "compliance_cold_plates.csv", cp_cols,
               ([d[c] for c in cp_cols] for d in comp.get("cold_plates", [])))
    horizon = comp.get("within_horizon", [None] * scenario.I)
    _write_csv(folder / "compliance_workloads.csv",
               ["i", "workload_id", "placed", "work_delivered_MHz_s", "work_required_MHz_s",
                "min_DPF", "starts_on_or_after_arrival", "wait_within_allowed", "on_time",
                "work_complete", "within_horizon", "compliant"],
               ([w.i, w.workload_id, ev.placed[w.i - 1], ev.work_delivered_MHz_s[w.i - 1],
                 ev.work_required_MHz_s[w.i - 1], ev.min_DPF_run[w.i - 1],
                 ev.starts_on_or_after_arrival[w.i - 1],
                 ev.wait_within_allowed[w.i - 1], ev.on_time[w.i - 1], ev.work_complete[w.i - 1],
                 horizon[w.i - 1], ev.compliant[w.i - 1]] for w in scenario.workloads))


def _write_comparison_period(path: Path, scenario: Scenario, base: EvalResult,
                             opt: EvalResult | None) -> None:
    comps = ["E_IT_kWh", "E_crac_kWh", "E_cp_kWh", "E_fan_kWh", "E_total_kWh", "carbon_g"]
    header = (["t", "CI_g_per_kWh"] + [f"baseline_{c}" for c in comps]
              + [f"optimized_{c}" for c in comps] + [f"diff_{c}" for c in comps])
    rows = []
    for t in range(scenario.T):
        b = [getattr(base, c)[t] for c in comps]
        if opt is None:
            o = [None] * len(comps)
            d = [None] * len(comps)
        else:
            o = [getattr(opt, c)[t] for c in comps]
            d = [ov - bv for ov, bv in zip(o, b)]
        rows.append([t + 1, scenario.CI_g_per_kWh[t]] + b + o + d)
    _write_csv(path, header, rows)


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

# (key, label, unit) in table order. Values come from EvalResult.totals.
METRICS = [
    ("carbon_kg", "Total carbon", "kg"),
    ("E_IT_kWh", "Energy, IT", "kWh"),
    ("E_crac_kWh", "Energy, CRAC", "kWh"),
    ("E_cp_kWh", "Energy, cold plate", "kWh"),
    ("Q_cp_kWh", "Heat removed by cold plates", "kWh"),
    ("E_fan_kWh", "Energy, fan", "kWh"),
    ("E_total_kWh", "Energy, total", "kWh"),
    ("n_late", "Workloads late", ""),
    ("n_noncompliant", "Non-compliant workloads", ""),
    ("mean_wait_RT", "Mean wait, time-sensitive", "periods"),
    ("max_wait_RT", "Max wait, time-sensitive", "periods"),
    ("mean_wait_DT", "Mean wait, delay-tolerant", "periods"),
    ("max_wait_DT", "Max wait, delay-tolerant", "periods"),
    ("n_priority_violations", "Priority inversions", ""),
    ("hottest_T_C", "Hottest server temperature", "C"),
    ("mean_T_C", "Mean server temperature", "C"),
    ("min_headroom_C", "Minimum headroom", "C"),
    ("mean_headroom_C", "Mean headroom", "C"),
    ("periods_above_dvfs", "Server-periods above DVFS temperature", ""),
    ("periods_above_cap", "Server-periods above thermal cap", ""),
    ("n_throttled_servers", "Servers throttled (DPF < 1)", ""),
    ("n_reassigned", "Workloads reassigned", ""),
    ("n_cross_zone", "Reassigned to another zone", ""),
]
_SCHEDULE_METRICS = ("n_reassigned", "n_cross_zone")


def _metrics_block(scenario: Scenario, ev: EvalResult, schedule: Schedule,
                   original_server: dict) -> dict:
    t = ev.totals
    block = {k: t.get(k) for k, _, _ in METRICS if k not in _SCHEDULE_METRICS}
    block["n_reassigned"] = _count_reassigned(schedule, original_server)
    block["n_cross_zone"] = _count_cross_zone(scenario, schedule, original_server)
    block["hottest_server"] = t.get("hottest_server")
    block["hottest_period"] = t.get("hottest_period")
    block["n_unplaced"] = t.get("n_unplaced")
    block["min_DPF"] = t.get("min_DPF")
    return to_jsonable(block)


def _side(scenario: Scenario, ev: EvalResult, schedule: Schedule, original_server: dict) -> dict:
    return {
        "valid": bool(ev.valid),
        "violations": list(ev.compliance.get("violations", [])),
        "metrics": _metrics_block(scenario, ev, schedule, original_server),
        "totals": to_jsonable(ev.totals),
        "compliance_flags": to_jsonable({k: v for k, v in ev.compliance.items()
                                         if k.endswith("_ok") or k == "all_ok"}),
    }


def _diff(b, o):
    if b is None or o is None:
        return {"baseline": b, "optimized": o, "diff": None, "pct": None}
    d = o - b
    pct = None if b == 0 else 100.0 * d / b
    return {"baseline": b, "optimized": o, "diff": d, "pct": pct}


def _solver_block(sr: SolveResult | None) -> dict | None:
    if sr is None:
        return None
    return to_jsonable({
        "status": sr.status, "solver_status_text": sr.solver_status_text,
        "solve_time_s": sr.solve_time_s, "mip_gap": sr.mip_gap,
        "objective_g": sr.objective_g, "best_bound_g": sr.best_bound_g,
        "n_variables": sr.n_variables, "n_binaries": sr.n_binaries,
        "n_constraints": sr.n_constraints, "relaxed": list(sr.relaxed),
        "has_schedule": sr.has_schedule, "notes": list(sr.notes),
    })


def write_results(out_dir, scenario: Scenario, settings: RunSettings,
                  original_server: dict, baseline_schedule: Schedule, baseline_eval: EvalResult,
                  optimized_schedule: Schedule | None, optimized_eval: EvalResult | None,
                  solve_result: SolveResult | None, infeasibility: dict | None,
                  checks: dict) -> dict:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    _write_schedule_tables(out / "baseline", scenario, baseline_schedule, baseline_eval,
                           original_server)
    have_opt = optimized_schedule is not None and optimized_eval is not None
    if have_opt:
        _write_schedule_tables(out / "optimized", scenario, optimized_schedule, optimized_eval,
                               original_server)
        _write_reassignment(out / "optimized" / "reassignment.csv", scenario, settings,
                            optimized_schedule, optimized_eval, original_server)
    _write_comparison_period(out / "comparison_period.csv", scenario, baseline_eval,
                             optimized_eval if have_opt else None)

    base_side = _side(scenario, baseline_eval, baseline_schedule, original_server)
    opt_side = (_side(scenario, optimized_eval, optimized_schedule, original_server)
                if have_opt else None)
    comparison = {}
    for key, _, _ in METRICS:
        b = base_side["metrics"].get(key)
        o = opt_side["metrics"].get(key) if opt_side else None
        comparison[key] = _diff(b, o)

    summary = {
        "scenario": {"name": scenario.name, "I": scenario.I, "S": scenario.S, "Z": scenario.Z,
                     "T": scenario.T, "dt_seconds": scenario.dt_seconds,
                     "zones": [sv.zone for sv in scenario.servers],
                     "n_running": sum(w.B for w in scenario.workloads),
                     "n_unavailable_server_periods": int((scenario.availability() == 0).sum()),
                     "CI_mean_g_per_kWh": float(np.mean(scenario.CI_g_per_kWh))},
        "settings": settings.as_dict(),
        "model_note": model_note(settings),
        "cold_plates": _cold_plate_block(scenario, settings, baseline_eval),
        "airflow_modelled": bool(baseline_eval.airflow_modelled),
        "thermal_cap_C": baseline_eval.compliance.get("thermal_cap_C"),
        "T_dvfs_C": scenario.constants.T_dvfs_C,
        "baseline": base_side,
        "optimized": opt_side,
        "has_optimized_schedule": have_opt,
        "comparison": comparison,
        "solver": _solver_block(solve_result),
        "infeasibility": infeasibility,
        "checks": checks,
    }
    summary = to_jsonable(summary)
    with open(out / "summary.json", "w") as fh:
        json.dump(summary, fh, indent=2, allow_nan=False)
    return summary


# ---------------------------------------------------------------------------
# Printed table
# ---------------------------------------------------------------------------

def _fmt(v, unit_key: str | None = None) -> str:
    if v is None:
        return "-"
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, int):
        return str(v)
    a = abs(v)
    if a != 0 and a < 0.01:
        return f"{v:.2e}"
    return f"{v:,.2f}"


def _fmt_pct(p) -> str:
    return "-" if p is None else f"{p:+.1f}%"


def _fmt_diff(d) -> str:
    if d is None:
        return "-"
    if isinstance(d, int):
        return f"{d:+d}"
    return f"{d:+,.2f}" if (d == 0 or abs(d) >= 0.01) else f"{d:+.2e}"


def format_summary(summary: dict) -> str:
    sc = summary["scenario"]
    st = summary["settings"]
    lines = []
    lines.append(f"Scenario {sc['name']}: I={sc['I']} workloads, S={sc['S']} servers, "
                 f"Z={sc['Z']} zones, T={sc['T']} periods of {sc['dt_seconds']:g} s")
    lines.append(f"Run '{st.get('run_name')}': baseline={st.get('baseline_rule')} at "
                 f"{st.get('baseline_supply_temp_C')} C, reassignment={st.get('reassignment')}, "
                 f"supply_temp_mode={st.get('supply_temp_mode')}, "
                 f"priority={'on' if st.get('priority', True) else 'off'}, "
                 f"thermal cap {summary.get('thermal_cap_C')} C")
    extra = []
    if st.get("zone_map"):
        extra.append(f"zone map {st['zone_map']}")
    if st.get("carbon_file"):
        extra.append(f"carbon intensity from {st['carbon_file']}")
    if st.get("ci_flat"):
        extra.append(f"flat carbon intensity {sc.get('CI_mean_g_per_kWh', 0):.2f} g/kWh")
    if extra:
        lines.append("Scenario overrides: " + "; ".join(extra))
    cpb = summary.get("cold_plates") or {}
    if cpb.get("enabled"):
        lines.append(f"Cold plates per server {cpb.get('plates_per_server')} "
                     f"({cpb.get('count_source')}); flow settings {cpb.get('flow_settings_Lmin')} L/min; "
                     f"baseline fixed at {cpb.get('baseline_flow_Lmin')} L/min, "
                     f"{cpb.get('baseline_coolant_C')} C coolant"
                     + (f"; optimized fixed at {cpb.get('optimized_fixed_flow_Lmin')} L/min, "
                        f"{cpb.get('optimized_fixed_coolant_C')} C coolant"
                        if cpb.get("optimized_mode") == "fixed" else ""))
    lines.append("")

    have_opt = summary.get("has_optimized_schedule")
    rows = [("Metric", "Baseline", "Optimized", "Diff", "Diff %")]
    comp = summary["comparison"]
    for key, label, unit in METRICS:
        c = comp[key]
        name = f"{label} ({unit})" if unit else label
        if key == "periods_above_dvfs":
            name = f"Server-periods above {summary.get('T_dvfs_C'):g} C"
        b_txt = _fmt(c["baseline"])
        o_txt = _fmt(c["optimized"]) if have_opt else "-"
        if key == "hottest_T_C":
            b_txt += f" (server {summary['baseline']['metrics'].get('hottest_server')})"
            if have_opt:
                o_txt += f" (server {summary['optimized']['metrics'].get('hottest_server')})"
        rows.append((name, b_txt, o_txt,
                     _fmt_diff(c["diff"]) if have_opt else "-",
                     _fmt_pct(c["pct"]) if have_opt else "-"))
    rows.append(("Schedule valid (no limit broken)", str(summary["baseline"]["valid"]),
                 str(summary["optimized"]["valid"]) if have_opt else "-", "", ""))

    widths = [max(len(r[i]) for r in rows) for i in range(5)]
    for n, r in enumerate(rows):
        lines.append("  ".join(r[0].ljust(widths[0]) if i == 0 else r[i].rjust(widths[i])
                               for i in range(5)))
        if n == 0:
            lines.append("  ".join("-" * w for w in widths))

    for side, label in (("baseline", "Baseline"), ("optimized", "Optimized")):
        d = summary.get(side)
        if d and d["violations"]:
            lines.append("")
            lines.append(f"{label} violations:")
            lines.extend(f"  - {v}" for v in d["violations"])

    sv = summary.get("solver")
    lines.append("")
    if sv is None:
        lines.append("Solver: not run.")
    else:
        gap = sv.get("mip_gap")
        lines.append(f"Solver: status={sv['status']} ({sv.get('solver_status_text') or '-'}), "
                     f"time={sv['solve_time_s']:.1f} s, gap={'-' if gap is None else f'{gap:.4%}'}, "
                     f"objective={_fmt(sv.get('objective_g'))} g, bound={_fmt(sv.get('best_bound_g'))} g")
        lines.append(f"        {sv['n_variables']} variables, {sv['n_binaries']} binaries, "
                     f"{sv['n_constraints']} constraints")
        for note in sv.get("notes") or []:
            lines.append(f"        note: {note}")
    if not have_opt:
        lines.append("")
        lines.append("NO VALID SCHEDULE WAS FOUND by the optimizer; only the baseline is shown.")
    inf = summary.get("infeasibility")
    if inf:
        lines.append("Infeasibility diagnosis (one constraint family relaxed at a time):")
        for fam, outcome in inf.items():
            lines.append(f"  {fam}: {outcome if outcome is not None else '-'}")
    lines.append("")
    lines.append("Note: " + summary["model_note"])
    return "\n".join(lines)


In [ ]:
%%writefile {LIB}/scheduler/run.py
"""Command line: build the baseline, solve the optimized schedule, compare the two.

    uv run python -m scheduler.run --scenario small
    uv run python -m scheduler.run --scenario small --baseline-rule fastest_first --baseline-supply-temp 27
    uv run python -m scheduler.run --scenario small --cold-plates
    uv run python -m scheduler.run --scenario small --cold-plate-count 2
    uv run python -m scheduler.run --scenario base --zone-map 1,1,1,2,2,2
    uv run python -m scheduler.run --scenario base --cold-plates --cp-mode fixed
    uv run python -m scheduler.run --scenario base --ci-flat

Both schedules are scored by the shared evaluator; only its numbers are compared.
"""

from __future__ import annotations

import argparse
import sys
from dataclasses import replace
from pathlib import Path

from scheduler.baseline import build_baseline
from scheduler.evaluate import evaluate
from scheduler.io import apply_overrides, load_reference_baseline, load_scenario
from scheduler.optimize import diagnose_infeasibility, solve
from scheduler.report import format_summary, write_results
from scheduler.types import (
    BASELINE_RULES, COLD_PLATE_MODES, REASSIGNMENT_MODES, STATUS_INFEASIBLE, SUPPLY_TEMP_MODES,
    NotDefinedYet, RunSettings, constant_supply_temp,
)

OBJECTIVE_REL_TOL = 1e-6    # optimizer objective against the evaluator's carbon
NO_WORSE_REL_TOL = 1e-9     # optimized carbon against the baseline's
STAGE_SHARE = 0.2           # share of the time limit for the fixed-server first stage


def parse_args(argv=None) -> tuple[RunSettings, argparse.Namespace]:
    d = RunSettings()
    p = argparse.ArgumentParser(prog="scheduler.run", description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--scenario", default=d.scenario)
    p.add_argument("--run-name", default=None,
                   help="folder under results/<scenario>/ (default: built from the switches)")
    p.add_argument("--baseline-rule", choices=BASELINE_RULES, default=d.baseline_rule)
    p.add_argument("--seed", type=int, default=d.seed, help="for the random baseline rule")
    p.add_argument("--baseline-supply-temp", type=float, default=d.baseline_supply_temp_C,
                   help="fixed supply air temperature of the baseline, degC")
    p.add_argument("--reassignment", choices=REASSIGNMENT_MODES, default=d.reassignment)
    p.add_argument("--supply-temp-mode", choices=SUPPLY_TEMP_MODES, default=d.supply_temp_mode)
    p.add_argument("--fixed-supply-temp", type=float, default=None,
                   help="supply temperature when --supply-temp-mode fixed "
                        "(default: the baseline's)")
    p.add_argument("--h-min", type=float, default=None,
                   help="override the minimum thermal headroom H_min, degC")
    p.add_argument("--time-limit", type=float, default=d.time_limit_s, help="seconds")
    p.add_argument("--mip-gap", type=float, default=d.mip_rel_gap, help="relative gap")
    p.add_argument("--cold-plates", action="store_true",
                   help="model cold plates; plates per server from servers.csv (n_cold_plates, "
                        "else CT_s_cold_plate)")
    p.add_argument("--cold-plate-count", type=int, default=None,
                   help="put this many cold plates on every server (0 disables them); "
                        "implies --cold-plates")
    p.add_argument("--baseline-cp-flow", type=float, default=d.baseline_cp_flow_Lmin,
                   help="fixed cold-plate flow of the baseline, L/min (one of the flow settings)")
    p.add_argument("--baseline-cp-coolant", type=float, default=d.baseline_cp_coolant_C,
                   help="fixed cold-plate coolant temperature of the baseline, degC")
    p.add_argument("--cp-mode", choices=COLD_PLATE_MODES, default=d.cp_mode,
                   help="optimize: the optimized run picks flow and coolant per server and period; "
                        "fixed: it holds them at --fixed-cp-flow / --fixed-cp-coolant")
    p.add_argument("--fixed-cp-flow", type=float, default=None,
                   help="cold-plate flow when --cp-mode fixed, L/min (default: the baseline's)")
    p.add_argument("--fixed-cp-coolant", type=float, default=None,
                   help="coolant temperature when --cp-mode fixed, degC (default: the baseline's)")
    p.add_argument("--no-priority", action="store_true",
                   help="drop the priority rules (56)-(57) from the optimized run")
    p.add_argument("--zone-map", default=None,
                   help="zone of each server in order, e.g. 1,1,1,2,2,2 (replaces servers.csv)")
    ci = p.add_mutually_exclusive_group()
    ci.add_argument("--carbon-file", default=None,
                    help="CSV with columns t, CI_t_g_per_kWh replacing the scenario's carbon.csv")
    ci.add_argument("--ci-flat", action="store_true",
                    help="hold carbon intensity at the scenario's mean in every period")
    p.add_argument("--fan-power", action="store_true", help="not defined yet")
    p.add_argument("--baseline-only", action="store_true", help="skip the optimized run")
    p.add_argument("--no-warm-start", action="store_true",
                   help="do not offer the solver a starting schedule")
    p.add_argument("--diagnosis-time-limit", type=float, default=120.0,
                   help="seconds per relaxed re-solve when no valid schedule exists")
    p.add_argument("--solver-log", action="store_true")
    p.add_argument("--scenarios-dir", default="scenarios")
    p.add_argument("--results-dir", default="results")
    a = p.parse_args(argv)
    cold_plates = a.cold_plates or a.cold_plate_count is not None
    zone_map = None
    if a.zone_map:
        try:
            zone_map = tuple(int(z) for z in a.zone_map.split(","))
        except ValueError:
            p.error(f"--zone-map must be comma-separated zone numbers, got {a.zone_map!r}")

    settings = RunSettings(
        scenario=a.scenario, baseline_rule=a.baseline_rule, seed=a.seed,
        baseline_supply_temp_C=a.baseline_supply_temp, reassignment=a.reassignment,
        supply_temp_mode=a.supply_temp_mode, fixed_supply_temp_C=a.fixed_supply_temp,
        H_min_C=a.h_min, time_limit_s=a.time_limit, mip_rel_gap=a.mip_gap,
        cold_plates=cold_plates, cold_plate_count=a.cold_plate_count,
        baseline_cp_flow_Lmin=a.baseline_cp_flow, baseline_cp_coolant_C=a.baseline_cp_coolant,
        cp_mode=a.cp_mode, fixed_cp_flow_Lmin=a.fixed_cp_flow, fixed_cp_coolant_C=a.fixed_cp_coolant,
        priority=not a.no_priority, zone_map=zone_map, carbon_file=a.carbon_file,
        ci_flat=a.ci_flat, fan_power=a.fan_power,
    )
    settings.run_name = a.run_name or default_run_name(settings)
    return settings, a


def default_run_name(settings: RunSettings) -> str:
    """Run folder name built from the switches, e.g. ``least_loaded_base18_free_optimize``."""
    st = settings
    fixed_temp = st.optimized_fixed_supply_temp_C()
    return "_".join([
        st.baseline_rule, f"base{st.baseline_supply_temp_C:g}", st.reassignment,
        st.supply_temp_mode if st.supply_temp_mode == "optimize" else f"fixed{fixed_temp:g}",
    ] + ([] if not st.cold_plates else
         ["cp" if st.cold_plate_count is None else f"cp{st.cold_plate_count}"])
      + (["cpfixed"] if st.cold_plates and st.cp_mode == "fixed" else [])
      + (["noprio"] if not st.priority else [])
      + ([f"zones{''.join(map(str, st.zone_map))}"] if st.zone_map else [])
      + (["ciflat"] if st.ci_flat else [])
      + ([f"ci-{Path(st.carbon_file).stem}"] if st.carbon_file else []))


def solve_staged(scenario, original, settings, base_sched, base_supply, warm_start, log):
    """Solve the optimized model, offering the solver a starting schedule.

    The start is the baseline. When servers may change, the same model is first solved
    with every server fixed (a much smaller problem whose solutions are all valid for
    the full one) and its schedule is offered instead. A start only gives the solver a
    first incumbent; it does not change the model or its optimum.
    """
    if not warm_start:
        return solve(scenario, original, settings, log=log)
    start_sched, start_supply = base_sched, base_supply
    stage_time, stage_note = 0.0, None
    if settings.reassignment != "none":
        stage = solve(scenario, original, replace(settings, reassignment="none"),
                      warm_start=base_sched, warm_start_supply_temp_C=base_supply,
                      time_limit_s=STAGE_SHARE * settings.time_limit_s, log=log)
        stage_time = stage.solve_time_s
        if stage.has_schedule:
            start_sched, start_supply = stage.schedule, stage.supply_temp_C
            stage_note = (
                f"Starting schedule: the same model with reassignment=none, solved first "
                f"({stage.status}, {stage.objective_g:,.2f} g, {stage_time:.1f} s); its time "
                f"is included in the solve time.")
        else:
            stage_note = (f"Starting schedule: the baseline (the reassignment=none stage "
                          f"ended with status {stage.status} after {stage_time:.1f} s).")
    result = solve(scenario, original, settings, warm_start=start_sched,
                   warm_start_supply_temp_C=start_supply,
                   time_limit_s=max(1.0, settings.time_limit_s - stage_time), log=log)
    result.solve_time_s += stage_time
    if stage_note:
        result.notes.append(stage_note)
    return result


def optimized_checks(scenario, settings, original, base_eval, result, opt_eval) -> dict:
    """Checks on an optimized schedule: the solver's objective equals the evaluator's carbon,
    the evaluator finds no broken limit, running workloads stay put, and with nothing to
    optimise but timing the result is no worse than a valid baseline."""
    out: dict = {}
    diff = abs(result.objective_g - opt_eval.total_carbon_g)
    out["optimizer_objective_equals_evaluator_carbon"] = {
        "passed": diff <= OBJECTIVE_REL_TOL * abs(opt_eval.total_carbon_g),
        "optimizer_objective_g": result.objective_g,
        "evaluator_carbon_g": opt_eval.total_carbon_g,
        "relative_difference": diff / abs(opt_eval.total_carbon_g)}
    out["optimized_schedule_valid_under_evaluator"] = {
        "passed": opt_eval.valid, "violations": opt_eval.compliance["violations"]}
    running = [w.i for w in scenario.workloads if w.B == 1]
    if running:
        moved = [i for i in running
                 if result.schedule.placements[i - 1].server != original[i]]
        out["running_workloads_not_reassigned"] = {
            "passed": not moved, "running": len(running), "moved": moved}
    same_temp = (settings.supply_temp_mode == "fixed" and
                 settings.optimized_fixed_supply_temp_C() == settings.baseline_supply_temp_C)
    if settings.reassignment == "none" and same_temp:
        name = "optimized_no_worse_than_valid_baseline"
        if base_eval.valid:
            out[name] = {
                "passed": opt_eval.total_carbon_g <=
                base_eval.total_carbon_g * (1 + NO_WORSE_REL_TOL),
                "baseline_carbon_g": base_eval.total_carbon_g,
                "optimized_carbon_g": opt_eval.total_carbon_g}
        else:
            out[name] = {"passed": None, "not_applicable":
                         "the baseline breaks limits, so it is not a valid schedule"}
    return out


def format_checks(checks: dict) -> str:
    lines = ["Checks:"]
    for name, c in checks.items():
        state = {True: "PASS", False: "FAIL", None: "n/a"}.get(c.get("passed"), "info")
        detail = {k: v for k, v in c.items() if k != "passed"}
        lines.append(f"  [{state}] {name}" + (f"  {detail}" if detail else ""))
    return "\n".join(lines)


def run(settings: RunSettings, *, scenarios_dir="scenarios", results_dir="results",
        baseline_only=False, warm_start=True, diagnosis_time_limit_s=120.0,
        solver_log=False, quiet=False) -> dict:
    """Run one comparison and write its results. Returns the summary dict."""
    settings.validate()
    scenario = apply_overrides(load_scenario(settings.scenario, scenarios_dir), settings)
    checks: dict = {}

    # baseline
    base_sched = build_baseline(scenario, settings)
    if settings.baseline_rule == "least_loaded":
        ref = load_reference_baseline(settings.scenario, scenarios_dir)
        checks["least_loaded_reproduces_baseline_schedule_csv"] = {
            "passed": base_sched == ref}
    base_supply = constant_supply_temp(scenario, settings.baseline_supply_temp_C)
    base_eval = evaluate(scenario, base_sched, base_supply, settings)
    original = base_sched.server_of()

    # optimized
    result = opt_eval = infeasibility = None
    if baseline_only:
        pass
    elif any(sv is None for sv in original.values()):
        unplaced = [i for i, sv in original.items() if sv is None]
        checks["optimized_run_skipped"] = {
            "reason": f"the baseline could not place workloads {unplaced}, so they have no "
                      "original server"}
    else:
        result = solve_staged(scenario, original, settings, base_sched, base_supply,
                              warm_start, solver_log)
        if result.status == STATUS_INFEASIBLE:
            infeasibility = diagnose_infeasibility(scenario, original, settings,
                                                   time_limit_s=diagnosis_time_limit_s)
        if result.has_schedule:
            opt_eval = evaluate(scenario, result.schedule, result.supply_temp_C, settings,
                                result.cold_plate_ops)
            checks.update(optimized_checks(scenario, settings, original, base_eval,
                                           result, opt_eval))

    out_dir = Path(results_dir) / settings.scenario / settings.run_name
    summary = write_results(
        out_dir, scenario, settings, original, base_sched, base_eval,
        result.schedule if result else None, opt_eval, result, infeasibility, checks)
    if not quiet:
        print(format_summary(summary))
        print("\n" + format_checks(checks))
        print(f"\nResults written to {out_dir}/")
    return summary


def main(argv=None) -> int:
    settings, a = parse_args(argv)
    try:
        summary = run(settings, scenarios_dir=a.scenarios_dir, results_dir=a.results_dir,
                      baseline_only=a.baseline_only, warm_start=not a.no_warm_start,
                      diagnosis_time_limit_s=a.diagnosis_time_limit, solver_log=a.solver_log)
    except NotDefinedYet as e:
        print(f"Not defined yet: {e}", file=sys.stderr)
        return 2
    failed = [n for n, c in summary.get("checks", {}).items() if c.get("passed") is False]
    if failed:
        print(f"\nFAILED checks: {failed}", file=sys.stderr)
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile {LIB}/dashboard/data.py
"""Read-only loaders for ThermaSched run outputs.

A run lives in ``results/<scenario>/<run_name>/`` (see ``scheduler/report.py``). This module
turns one into tidy pandas frames: per-variant CSVs are stacked into one long frame with a
``variant`` column ("baseline" / "optimized"), servers carry their names from
``scenarios/<scenario>/servers.csv``, and every per-period frame gets the local clock time
from ``scenarios/<scenario>/carbon.csv``.

There is no Streamlit here so the loaders can be tested directly; the UI caches them.
"""

from __future__ import annotations

import json
import os
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path(__file__).resolve().parents[1]
RESULTS_DIR = Path(os.environ.get("THERMASCHED_RESULTS_DIR", REPO_ROOT / "results"))
SCENARIOS_DIR = Path(os.environ.get("THERMASCHED_SCENARIOS_DIR", REPO_ROOT / "scenarios"))

VARIANTS = ("baseline", "optimized")
VARIANT_LABELS = {"baseline": "Baseline", "optimized": "Optimized"}

# Energy components in stacking order: column -> label.
ENERGY_COMPONENTS = {
    "E_IT_kWh": "IT",
    "E_crac_kWh": "CRAC",
    "E_cp_kWh": "Cold-plate pumps",
    "E_fan_kWh": "Fans",
}

# Workload types as written in schedule.csv.
WORKLOAD_TYPES = {"time_sensitive": "Time-sensitive", "delay_tolerant": "Delay-tolerant"}

# Constraint families relaxed one at a time by scheduler.optimize.diagnose_infeasibility.
RELAX_FAMILY_LABELS = {
    "deadlines_and_waiting": "Deadlines and waiting limits, Eq. (27)-(28)",
    "thermal_cap": "Thermal cap T <= T_max - H_min, Eq. (31)-(33)",
    "capacity": "CPU and memory capacity, Eq. (12)-(13)",
    "cooling_capacity": "CRAC cooling capacity, Eq. (54)",
    "priority": "Priority order, Eq. (56)-(57)",
}

# Monday 2000-01-03 as a dummy date: only the clock time is shown.
CLOCK_ORIGIN = pd.Timestamp("2000-01-03")
ZERO_TOL = 1e-12


# ---------------------------------------------------------------------------
# Run discovery
# ---------------------------------------------------------------------------

@dataclass(frozen=True, order=True)
class RunRef:
    scenario: str
    name: str
    path: Path = field(compare=False)

    @property
    def key(self) -> str:
        return f"{self.scenario}/{self.name}"


def discover_runs(results_dir: str | Path = RESULTS_DIR) -> list[RunRef]:
    """Every ``<results_dir>/<scenario>/<run>/summary.json``, sorted by scenario and run."""
    root = Path(results_dir)
    if not root.is_dir():
        return []
    return sorted(RunRef(p.parent.parent.name, p.parent.name, p.parent)
                  for p in root.glob("*/*/summary.json"))


def load_summary(run_dir: str | Path) -> dict:
    with open(Path(run_dir) / "summary.json") as f:
        return json.load(f)


# ---------------------------------------------------------------------------
# Scenario inputs: servers and clock
# ---------------------------------------------------------------------------

def scenario_file(scenario: str, name: str, scenarios_dir: str | Path = SCENARIOS_DIR,
                  run_dir: str | Path | None = None) -> Path | None:
    """``scenarios/<scenario>/<name>``, else ``<name>`` next to the run folders
    (``results/<scenario>/<name>``, where an unzipped run export puts it), else None."""
    for path in (Path(scenarios_dir) / scenario / name,
                 Path(run_dir).parent / name if run_dir is not None else None):
        if path is not None and path.is_file():
            return path
    return None


def load_servers(scenario: str, run_dir: str | Path | None = None,
                 scenarios_dir: str | Path = SCENARIOS_DIR) -> pd.DataFrame:
    """Columns s, name, zone, label, zone_label; sorted by zone then s (display order).

    Names come from the scenario's ``servers.csv`` (see ``scenario_file``), else from the
    run's ``compliance_servers.csv``, else "server <s>".
    """
    path = scenario_file(scenario, "servers.csv", scenarios_dir, run_dir)
    if path is not None:
        df = pd.read_csv(path, usecols=["s", "server", "zone"]).rename(columns={"server": "name"})
    else:
        df = pd.DataFrame(columns=["s", "name", "zone"])
        if run_dir is not None:
            comp = Path(run_dir) / "baseline" / "compliance_servers.csv"
            if comp.is_file():
                df = pd.read_csv(comp, usecols=["s", "name", "zone"])
    df = df.astype({"s": int, "zone": int})
    df["name"] = df["name"].fillna(df["s"].map(lambda s: f"server {s}")).astype(str)
    df["zone_label"] = "Zone " + df["zone"].astype(str)
    df["label"] = df["name"] + " (Z" + df["zone"].astype(str) + ")"
    return df.sort_values(["zone", "s"]).reset_index(drop=True)


def _with_zones(servers: pd.DataFrame, zones: list[int]) -> pd.DataFrame:
    df = servers.copy()
    df["zone"] = df["s"].map(dict(zip(range(1, len(zones) + 1), zones))).astype(int)
    df["zone_label"] = "Zone " + df["zone"].astype(str)
    df["label"] = df["name"] + " (Z" + df["zone"].astype(str) + ")"
    return df.sort_values(["zone", "s"]).reset_index(drop=True)


def hours_to_clock(hour_local: pd.Series) -> pd.Series:
    """Decimal local hours -> Timestamps on a dummy date, rolling to the next day at midnight.

    ``hour_local`` is printed to 3 decimals (3.6 s), so it is snapped to the nearest 30 s.
    """
    seconds = (np.round(hour_local.to_numpy(float) * 120) * 30).astype(np.int64)
    day = np.concatenate([[0], np.cumsum(np.diff(seconds) < 0)])
    return pd.Series(CLOCK_ORIGIN + pd.to_timedelta(day * 86400 + seconds, unit="s"),
                     index=hour_local.index)


def load_clock(scenario: str, T: int, dt_seconds: float,
               scenarios_dir: str | Path = SCENARIOS_DIR,
               run_dir: str | Path | None = None) -> pd.DataFrame:
    """Columns t, hour_local, time, clock for t = 1..T.

    ``time`` is the local time at the middle of period t (``carbon.csv:hour_local``, found
    by ``scenario_file``); a period covers ``time +/- dt/2``. Without carbon.csv the clock
    starts at 00:00.
    """
    path = scenario_file(scenario, "carbon.csv", scenarios_dir, run_dir)
    t = pd.DataFrame({"t": np.arange(1, T + 1)})
    if path is not None:
        carbon = pd.read_csv(path, usecols=["t", "hour_local"])
        t = t.merge(carbon, on="t", how="left")
    else:
        t["hour_local"] = np.nan
    if t["hour_local"].isna().any():
        # Fill gaps (or a missing file) by extending the known periods at dt per period.
        known = t.dropna(subset=["hour_local"])
        h0 = (known["hour_local"].iloc[0] - (known["t"].iloc[0] - 1) * dt_seconds / 3600
              if len(known) else 0.5 * dt_seconds / 3600)
        t["hour_local"] = t["hour_local"].fillna(((h0 + (t["t"] - 1) * dt_seconds / 3600) % 24))
    t["time"] = hours_to_clock(t["hour_local"])
    t["clock"] = t["time"].dt.strftime("%H:%M")
    return t


def period_bounds(clock: pd.DataFrame, dt_seconds: float):
    """Function t -> Timestamp of the *start* of period t, extrapolated past 1..T.

    Used for schedule bars: a workload starting in period ``start`` with exclusive end
    ``finish`` spans ``bound(start)`` to ``bound(finish)``.
    """
    t0 = clock["time"].iloc[0] - pd.Timedelta(seconds=dt_seconds / 2)

    def bound(t):
        return t0 + pd.to_timedelta((np.asarray(t, dtype=float) - 1) * dt_seconds, unit="s")
    return bound


# ---------------------------------------------------------------------------
# One run
# ---------------------------------------------------------------------------

def _read_variants(run_dir: Path, filename: str, variants: tuple[str, ...]) -> pd.DataFrame:
    frames = []
    for v in variants:
        path = run_dir / v / filename
        if path.is_file():
            df = pd.read_csv(path)
            df.insert(0, "variant", v)
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["variant"])
    return pd.concat(frames, ignore_index=True)


@dataclass
class Run:
    ref: RunRef
    summary: dict
    variants: tuple[str, ...]          # variants with a folder on disk, baseline first
    servers: pd.DataFrame
    clock: pd.DataFrame
    period: pd.DataFrame               # variant, t, energy, CI, carbon_g, time, clock
    comparison_period: pd.DataFrame    # t, CI, baseline_/optimized_/diff_ columns, time, clock
    server_period: pd.DataFrame        # variant, s, t, ..., name, zone, label, time, clock
    zone_period: pd.DataFrame          # variant, z, t, ..., zone_label, time, clock
    schedule: pd.DataFrame             # variant, i, ..., server_name, original_server_name
    compliance_servers: pd.DataFrame
    compliance_zones: pd.DataFrame
    compliance_workloads: pd.DataFrame
    compliance_cold_plates: pd.DataFrame = field(default_factory=pd.DataFrame)
    reassignment: pd.DataFrame = field(default_factory=pd.DataFrame)   # optimized only

    # -- summary shortcuts ------------------------------------------------------------
    @property
    def settings(self) -> dict:
        return self.summary.get("settings", {})

    @property
    def solver(self) -> dict:
        return self.summary.get("solver") or {}

    @property
    def has_optimized(self) -> bool:
        return bool(self.summary.get("has_optimized_schedule")) and "optimized" in self.variants

    @property
    def dt_seconds(self) -> float:
        return float(self.summary.get("scenario", {}).get("dt_seconds", 300.0))

    @property
    def thermal_cap_C(self) -> float:
        return float(self.summary.get("thermal_cap_C", 80.0))

    @property
    def T_dvfs_C(self) -> float:
        return float(self.summary.get("T_dvfs_C", 75.0))

    @property
    def T_max_C(self) -> float | None:
        """Server temperature limit T_max (headroom is measured from it)."""
        sp = self.server_period
        if sp.empty or "headroom_C" not in sp:
            return None
        return float((sp["temperature_C"] + sp["headroom_C"]).median())

    @property
    def H_min_C(self) -> float | None:
        t_max = self.T_max_C
        return None if t_max is None else t_max - self.thermal_cap_C

    def server_name(self, s) -> str:
        if s is None:
            return "–"
        m = self.servers.set_index("s")["name"]
        return str(m.get(int(s), f"server {s}"))


def load_run(run_dir: str | Path, scenarios_dir: str | Path = SCENARIOS_DIR) -> Run:
    run_dir = Path(run_dir)
    ref = RunRef(run_dir.parent.name, run_dir.name, run_dir)
    summary = load_summary(run_dir)
    sc = summary.get("scenario", {})
    scenario = sc.get("name", ref.scenario)
    variants = tuple(v for v in VARIANTS if (run_dir / v).is_dir())

    servers = load_servers(scenario, run_dir, scenarios_dir)
    zones = sc.get("zones")       # the run's own zones (a --zone-map run differs from servers.csv)
    if zones and len(zones) == len(servers):
        servers = _with_zones(servers, zones)
    dt = float(sc.get("dt_seconds", 300.0))
    T = int(sc.get("T") or _max_t(run_dir / "baseline" / "period.csv"))
    clock = load_clock(scenario, T, dt, scenarios_dir, run_dir)
    tcols = clock[["t", "time", "clock"]]

    def with_time(df: pd.DataFrame) -> pd.DataFrame:
        return df.merge(tcols, on="t", how="left") if "t" in df else df

    period = with_time(_read_variants(run_dir, "period.csv", variants))
    cmp_path = run_dir / "comparison_period.csv"
    comparison = with_time(pd.read_csv(cmp_path)) if cmp_path.is_file() else pd.DataFrame()

    server_cols = servers[["s", "name", "zone", "zone_label", "label"]]
    server_period = _read_variants(run_dir, "server_period.csv", variants)
    if not server_period.empty:
        server_period = with_time(server_period.merge(server_cols, on="s", how="left"))

    zone_period = _read_variants(run_dir, "zone_period.csv", variants)
    if not zone_period.empty:
        zone_period["zone_label"] = "Zone " + zone_period["z"].astype(str)
        if "airflow" in zone_period:
            # "not modelled" when airflow is off; keep a numeric column either way.
            zone_period["airflow"] = pd.to_numeric(zone_period["airflow"], errors="coerce")
        zone_period = with_time(zone_period)

    names = servers.set_index("s")["name"]
    schedule = _read_variants(run_dir, "schedule.csv", variants)
    if not schedule.empty:
        for col in ("server", "original_server"):
            schedule[f"{col}_name"] = schedule[col].map(names)
        schedule["type_label"] = schedule["type"].map(WORKLOAD_TYPES).fillna(schedule["type"])

    comp_servers = _read_variants(run_dir, "compliance_servers.csv", variants)
    if not comp_servers.empty and "name" not in comp_servers:
        comp_servers["name"] = comp_servers["s"].map(names)

    return Run(
        ref=ref, summary=summary, variants=variants, servers=servers, clock=clock,
        period=period, comparison_period=comparison, server_period=server_period,
        zone_period=zone_period, schedule=schedule, compliance_servers=comp_servers,
        compliance_zones=_read_variants(run_dir, "compliance_zones.csv", variants),
        compliance_workloads=_read_variants(run_dir, "compliance_workloads.csv", variants),
        compliance_cold_plates=_read_variants(run_dir, "compliance_cold_plates.csv", variants),
        reassignment=_read_reassignment(run_dir / "optimized" / "reassignment.csv", names),
    )


def _read_reassignment(path: Path, names: pd.Series) -> pd.DataFrame:
    """optimized/reassignment.csv with server names; empty for older runs without it."""
    if not path.is_file():
        return pd.DataFrame()
    df = pd.read_csv(path)
    for col in ("original_server", "final_server"):
        df[f"{col}_name"] = df[col].map(names)
    return df


def _max_t(path: Path) -> int:
    return int(pd.read_csv(path, usecols=["t"])["t"].max()) if path.is_file() else 0


# ---------------------------------------------------------------------------
# Derived views
# ---------------------------------------------------------------------------

def is_all_zero(series: pd.Series) -> bool:
    """True when a column has no non-zero number (all zero, blank or non-numeric)."""
    values = pd.to_numeric(series, errors="coerce").to_numpy(float)
    return bool(np.all(np.isnan(values) | (np.abs(values) <= ZERO_TOL)))


def nonzero_columns(df: pd.DataFrame, columns) -> list[str]:
    """The given columns that exist in ``df`` and are not all zero, in the given order."""
    return [c for c in columns if c in df and not is_all_zero(df[c])]


def active_energy_components(run: Run) -> list[str]:
    """Energy columns with any non-zero value in any variant (E_IT and E_crac always)."""
    always = ["E_IT_kWh", "E_crac_kWh"]
    extra = nonzero_columns(run.period, [c for c in ENERGY_COMPONENTS if c not in always])
    return always + extra


# (key, label, unit, better) for the KPI tiles; better is "lower", "higher" or "neutral".
KPIS = [
    ("carbon_kg", "Total carbon", "kg CO₂", "lower"),
    ("E_total_kWh", "Total energy", "kWh", "lower"),
    ("E_IT_kWh", "IT energy", "kWh", "lower"),
    ("E_crac_kWh", "CRAC energy", "kWh", "lower"),
    ("E_cp_kWh", "Cold-plate pump energy", "kWh", "lower"),
    ("E_fan_kWh", "Fan energy", "kWh", "lower"),
    ("hottest_T_C", "Hottest server temperature", "°C", "lower"),
    ("min_headroom_C", "Minimum thermal headroom", "°C", "higher"),
    ("periods_above_dvfs", "Server-periods above DVFS", "", "lower"),
    ("periods_above_cap", "Server-periods above cap", "", "lower"),
    ("n_late", "Late workloads", "", "lower"),
    ("n_noncompliant", "Non-compliant workloads", "", "lower"),
    ("n_priority_violations", "Priority inversions", "", "lower"),
    ("n_reassigned", "Reassigned workloads", "", "neutral"),
    ("n_cross_zone", "Moved to another zone", "", "neutral"),
]
# KPIs older runs do not have; dropped from the tiles when missing on both sides.
OPTIONAL_KPIS = ("n_priority_violations", "n_cross_zone")


def kpi_frame(summary: dict) -> pd.DataFrame:
    """One row per KPI: key, label, unit, better, baseline, optimized, diff, pct.

    ``optimized``/``diff``/``pct`` are NaN without an optimized schedule; ``pct`` is NaN
    when the baseline is 0. All-zero energy components (not modelled) are dropped.
    """
    base = (summary.get("baseline") or {}).get("metrics", {})
    opt = (summary.get("optimized") or {}).get("metrics", {}) \
        if summary.get("has_optimized_schedule") else {}
    rows = []
    for key, label, unit, better in KPIS:
        b, o = base.get(key), opt.get(key)
        if key in ("E_cp_kWh", "E_fan_kWh") and not b and not o:
            continue
        if key in OPTIONAL_KPIS and b is None and o is None:
            continue
        b = np.nan if b is None else float(b)
        o = np.nan if o is None else float(o)
        diff = o - b
        pct = diff / b * 100 if b and not np.isnan(diff) else np.nan
        rows.append(dict(key=key, label=label, unit=unit, better=better,
                         baseline=b, optimized=o, diff=diff, pct=pct))
    return pd.DataFrame(rows)


def checks_frame(summary: dict) -> pd.DataFrame:
    """Self-checks: name, label, passed (True/False/None), detail."""
    rows = []
    for name, c in (summary.get("checks") or {}).items():
        c = c if isinstance(c, dict) else {"passed": c}
        detail = {k: v for k, v in c.items() if k != "passed"}
        rows.append(dict(name=name, label=name.replace("_", " ").capitalize(),
                         passed=c.get("passed"),
                         detail="; ".join(f"{k.replace('_', ' ')}: {_fmt_detail(v)}"
                                          for k, v in detail.items())))
    return pd.DataFrame(rows, columns=["name", "label", "passed", "detail"])


def _fmt_detail(v) -> str:
    if isinstance(v, float):
        return f"{v:,.6g}"
    if isinstance(v, list):
        return "none" if not v else "; ".join(map(str, v))
    return str(v)


def infeasibility_frame(summary: dict) -> pd.DataFrame:
    """family, label, result: one row per constraint family the diagnosis relaxed."""
    diag = summary.get("infeasibility") or {}
    return pd.DataFrame(
        [dict(family=k, label=RELAX_FAMILY_LABELS.get(k, k.replace("_", " ")), result=v)
         for k, v in diag.items()], columns=["family", "label", "result"])


SAVINGS_COLUMNS = ["component", "label", "energy_saved_kWh", "carbon_saved_kg",
                   "energy_effect_kg", "timing_effect_kg"]


def savings_breakdown(run: Run) -> pd.DataFrame:
    """Where the savings came from, one row per energy component.

    Carbon per period is CI * E_total, so carbon saved splits exactly by component:
    ``carbon_saved_kg = sum_t CI_t * (E_base - E_opt)``. Positive means the optimized run
    emitted less. It is split further into an energy effect (the kWh saved at the mean
    carbon intensity) and a timing effect (the rest: energy moved to cleaner or dirtier
    periods), which explains carbon saved on IT even when IT kWh do not change.
    """
    cp = run.comparison_period
    if cp.empty or not run.has_optimized:
        return pd.DataFrame(columns=SAVINGS_COLUMNS)
    ci = cp["CI_g_per_kWh"]
    rows = []
    for col in active_energy_components(run):
        saved = -cp[f"diff_{col}"].fillna(0.0)
        saved = saved.where(saved.abs() > ZERO_TOL, 0.0)
        carbon = _clean(float((saved * ci).sum() / 1000))
        energy_effect = _clean(float(saved.sum() * ci.mean() / 1000))
        rows.append(dict(component=col, label=ENERGY_COMPONENTS[col],
                         energy_saved_kWh=_clean(float(saved.sum())), carbon_saved_kg=carbon,
                         energy_effect_kg=energy_effect,
                         timing_effect_kg=carbon - energy_effect))
    return pd.DataFrame(rows, columns=SAVINGS_COLUMNS)


def _clean(x: float, tol: float = 1e-9) -> float:
    """Round float noise (e.g. -7e-16 kWh) to an exact 0 so it does not print as -0.0000."""
    return 0.0 if abs(x) < tol else x


def cumulative_savings(run: Run) -> pd.DataFrame:
    """t, time, clock, carbon_saved_kg (running total) and per-component running totals."""
    cp = run.comparison_period
    if cp.empty or not run.has_optimized:
        return pd.DataFrame(columns=["t", "time", "clock", "carbon_saved_kg"])
    out = cp[["t", "time", "clock"]].copy()
    out["carbon_saved_kg"] = (-cp["diff_carbon_g"].fillna(0.0)).cumsum() / 1000
    for col in active_energy_components(run):
        out[f"{col}_carbon_saved_kg"] = \
            (-(cp[f"diff_{col}"].fillna(0.0) * cp["CI_g_per_kWh"])).cumsum() / 1000
    return out


def workload_moves(run: Run) -> pd.DataFrame:
    """One row per workload, baseline next to optimized (server, start, finish, wait).

    Without an optimized schedule the optimized columns are NaN.
    """
    sch = run.schedule
    if sch.empty:
        return pd.DataFrame()
    keep = ["i", "workload_id", "type", "type_label", "arrival", "allowed_delay", "deadline"]
    per = ["server_name", "start", "finish", "wait", "compliant"]
    b = sch[sch["variant"] == "baseline"]
    out = b[keep + per].rename(columns={c: f"{c}_baseline" for c in per})
    if run.has_optimized:
        o = sch[sch["variant"] == "optimized"]
        o = o[["i", "original_server_name", "reassigned"] + per].rename(
            columns={c: f"{c}_optimized" for c in per})
        out = out.merge(o, on="i", how="left")
    else:
        out["original_server_name"] = out["server_name_baseline"]
        out["reassigned"] = 0
        for c in per:
            out[f"{c}_optimized"] = np.nan
    ra = run.reassignment
    if not ra.empty:
        out = out.merge(ra[["i", "original_zone", "final_zone", "cross_zone", "reason"]],
                        on="i", how="left")
    out["wait_change"] = out["wait_optimized"] - out["wait_baseline"]
    out["start_shift"] = out["start_optimized"] - out["start_baseline"]
    out["reassigned"] = out["reassigned"].fillna(0).astype(bool)
    return out.reset_index(drop=True)


def gantt_lanes(schedule: pd.DataFrame) -> pd.Series:
    """Lane index per row so workloads running at once on a server get separate rows.

    Greedy interval partitioning per (variant, server) on [start, finish); unplaced
    workloads (no server or start) get lane -1.
    """
    lanes = pd.Series(-1, index=schedule.index, dtype=int)
    ok = schedule["server"].notna() & schedule["start"].notna() & schedule["finish"].notna()
    for _, grp in schedule[ok].groupby(["variant", "server"]):
        ends: list[float] = []
        for idx, row in grp.sort_values(["start", "finish", "i"]).iterrows():
            for k, end in enumerate(ends):
                if end <= row["start"]:
                    ends[k] = row["finish"]
                    lanes[idx] = k
                    break
            else:
                ends.append(row["finish"])
                lanes[idx] = len(ends) - 1
    return lanes


def throttled_servers(run: Run) -> pd.DataFrame:
    """Servers with DPF < 1 anywhere: variant, s, name, periods_throttled, min_DPF."""
    sp = run.server_period
    if sp.empty or "DPF" not in sp:
        return pd.DataFrame(columns=["variant", "s", "name", "periods_throttled", "min_DPF"])
    low = sp[sp["DPF"] < 1 - 1e-9]
    if low.empty:
        return pd.DataFrame(columns=["variant", "s", "name", "periods_throttled", "min_DPF"])
    return (low.groupby(["variant", "s", "name"], as_index=False)
            .agg(periods_throttled=("t", "size"), min_DPF=("DPF", "min")))


def compare_runs(runs: list[RunRef]) -> pd.DataFrame:
    """One row per run (summary.json only): settings, solver and key metrics."""
    rows = []
    for ref in runs:
        try:
            s = load_summary(ref.path)
        except (OSError, json.JSONDecodeError) as e:
            rows.append(dict(scenario=ref.scenario, run=ref.name, status=f"unreadable: {e}"))
            continue
        st, sol = s.get("settings", {}), s.get("solver") or {}
        base = (s.get("baseline") or {}).get("metrics", {})
        has_opt = bool(s.get("has_optimized_schedule"))
        opt = (s.get("optimized") or {}).get("metrics", {}) if has_opt else {}
        supply = (f"fixed {_supply_fixed(st):g} °C" if st.get("supply_temp_mode") == "fixed"
                  else "optimized")
        b_c, o_c = base.get("carbon_kg"), opt.get("carbon_kg")
        b_e, o_e = base.get("E_total_kWh"), opt.get("E_total_kWh")
        rows.append(dict(
            scenario=ref.scenario, run=ref.name,
            baseline_rule=st.get("baseline_rule"),
            baseline_supply_C=st.get("baseline_supply_temp_C"),
            reassignment=st.get("reassignment"), supply_temp=supply,
            H_min_C=st.get("H_min_C"), cold_plates=bool(st.get("cold_plates")),
            cp_mode=st.get("cp_mode", "optimize") if st.get("cold_plates") else None,
            priority=st.get("priority", False),
            zone_map=",".join(map(str, st["zone_map"])) if st.get("zone_map") else None,
            carbon=_carbon_label(st),
            status=sol.get("status"), solve_time_s=sol.get("solve_time_s"),
            mip_gap=sol.get("mip_gap"),
            baseline_valid=(s.get("baseline") or {}).get("valid"),
            optimized_valid=(s.get("optimized") or {}).get("valid") if has_opt else None,
            baseline_carbon_kg=b_c, optimized_carbon_kg=o_c,
            carbon_saved_kg=(b_c - o_c) if has_opt and b_c is not None else None,
            carbon_saved_pct=(b_c - o_c) / b_c * 100 if has_opt and b_c else None,
            baseline_E_total_kWh=b_e, optimized_E_total_kWh=o_e,
            energy_saved_pct=(b_e - o_e) / b_e * 100 if has_opt and b_e else None,
            baseline_hottest_C=base.get("hottest_T_C"), optimized_hottest_C=opt.get("hottest_T_C"),
            n_reassigned=opt.get("n_reassigned"), n_cross_zone=opt.get("n_cross_zone"),
            optimized_n_noncompliant=opt.get("n_noncompliant"),
        ))
    return pd.DataFrame(rows)


def _carbon_label(settings: dict) -> str:
    if settings.get("carbon_file"):
        return Path(settings["carbon_file"]).name
    return "flat (mean)" if settings.get("ci_flat") else "scenario"


def _supply_fixed(settings: dict) -> float:
    v = settings.get("fixed_supply_temp_C")
    return float(settings.get("baseline_supply_temp_C", 18.0) if v is None else v)


def settings_badges(summary: dict) -> list[tuple[str, str]]:
    """(label, value) pairs that describe a run's switches, in display order."""
    st = summary.get("settings", {})
    supply = (f"fixed at {_supply_fixed(st):g} °C" if st.get("supply_temp_mode") == "fixed"
              else "optimized")
    cp = "off"
    if st.get("cold_plates"):
        n = st.get("cold_plate_count")
        cp = "on (servers.csv)" if n is None else f"{n} per server"
        if st.get("cp_mode") == "fixed":
            flow = st.get("fixed_cp_flow_Lmin") or st.get("baseline_cp_flow_Lmin")
            cool = st.get("fixed_cp_coolant_C") or st.get("baseline_cp_coolant_C")
            cp += f", fixed at {flow:g} L/min, {cool:g} °C"
    h_min = st.get("H_min_C")
    return [
        ("Baseline rule", str(st.get("baseline_rule", "?")).replace("_", " ")),
        ("Baseline supply", f"{st.get('baseline_supply_temp_C', '?'):g} °C"
         if isinstance(st.get("baseline_supply_temp_C"), (int, float)) else "?"),
        ("Reassignment", str(st.get("reassignment", "?")).replace("_", " ")),
        ("Supply temperature", supply),
        ("H_min", "scenario default" if h_min is None else f"{h_min:g} °C"),
        ("Cold plates", cp),
        ("Priority (56)-(57)", "on" if st.get("priority") else "off"),
        ("Zones", "servers.csv" if not st.get("zone_map")
         else ",".join(map(str, st["zone_map"]))),
        ("Carbon intensity", _carbon_label(st)),
        ("Fan power", "on" if st.get("fan_power") else "off"),
        ("Time limit", f"{st.get('time_limit_s', 0):g} s"),
        ("MIP gap", f"{st.get('mip_rel_gap', 0):g}"),
    ]


In [ ]:
%%writefile {LIB}/dashboard/theme.py
"""Colours and shared Plotly styling.

Categorical colours come from a validated colour-blind-safe palette with a light and a
dark step per hue. Baseline and optimized keep the same hue everywhere (orange and blue).
Other dimensions (energy component, workload type) use hues distinct from those two.
Thresholds are drawn in neutral ink with a text label, never in a series colour.
"""

from __future__ import annotations

from dataclasses import dataclass, field

import plotly.graph_objects as go


@dataclass(frozen=True)
class Palette:
    mode: str
    baseline: str
    optimized: str
    components: dict[str, str]
    types: dict[str, str]
    ink: str
    ink2: str
    muted: str
    grid: str
    axis: str
    surface: str
    seq: tuple[str, ...]           # sequential ramp, low -> high (low recedes into surface)
    status: dict[str, str] = field(default_factory=lambda: dict(STATUS))

    def variant(self, v: str) -> str:
        return self.baseline if v == "baseline" else self.optimized


STATUS = {"good": "#0ca30c", "warning": "#fab219", "serious": "#ec835a", "critical": "#d03b3b"}

_BLUE_RAMP = ("#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b")

LIGHT = Palette(
    mode="light", baseline="#eb6834", optimized="#2a78d6",
    components={"E_IT_kWh": "#1baf7a", "E_crac_kWh": "#6250d6",
                "E_cp_kWh": "#e87ba4", "E_fan_kWh": "#eda100"},
    types={"time_sensitive": "#e87ba4", "delay_tolerant": "#008300"},
    ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
    surface="#fcfcfb", seq=_BLUE_RAMP,
)
DARK = Palette(
    mode="dark", baseline="#d95926", optimized="#3987e5",
    components={"E_IT_kWh": "#199e70", "E_crac_kWh": "#9085e9",
                "E_cp_kWh": "#d55181", "E_fan_kWh": "#c98500"},
    types={"time_sensitive": "#d55181", "delay_tolerant": "#008300"},
    ink="#f0efec", ink2="#c3c2b7", muted="#898781", grid="#2c2c2a", axis="#383835",
    surface="#1a1a19", seq=tuple(reversed(_BLUE_RAMP)),
)


def palette(mode: str | None) -> Palette:
    return DARK if mode == "dark" else LIGHT


def rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
TIME_FORMAT = "%H:%M"


def style(fig: go.Figure, pal: Palette, *, height: int = 360, legend: bool = True,
          time_axis: bool = True, hovermode: str | bool = "x unified",
          titled: bool = False) -> go.Figure:
    """Recessive grid and axes, transparent background, legend above the plot.

    ``titled``: the figure has subplot titles, so the legend sits one row higher.
    Charts are drawn with Streamlit's theme off (it replaces colour scales), so every
    colour here comes from the palette for the active light or dark mode.
    """
    lift = 30 if titled and legend else 0
    top = (36 if legend else 16) + lift
    plot_h = max(height - top - 60, 100)
    fig.update_layout(
        template="none", height=height, font=dict(family=FONT, color=pal.ink2, size=13),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=8, r=8, t=top, b=8),
        hovermode=hovermode, showlegend=legend,
        legend=dict(orientation="h", yanchor="bottom", y=1.0 + lift / plot_h, xanchor="left",
                    x=0, title_text="", font=dict(color=pal.ink2), bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(font=dict(family=FONT, color=pal.ink), bgcolor=pal.surface,
                        bordercolor=pal.axis),
    )
    fig.update_xaxes(gridcolor=pal.grid, linecolor=pal.axis, zeroline=False, showgrid=False,
                     tickfont=dict(color=pal.muted), title_font=dict(color=pal.ink2),
                     showline=True, ticks="outside", tickcolor=pal.axis, ticklen=4,
                     automargin=True)
    fig.update_yaxes(gridcolor=pal.grid, linecolor=pal.axis, zerolinecolor=pal.axis,
                     showgrid=True, zeroline=False, tickfont=dict(color=pal.muted),
                     title_font=dict(color=pal.ink2), automargin=True)
    if time_axis:
        fig.update_xaxes(tickformat=TIME_FORMAT, hoverformat=TIME_FORMAT,
                         showspikes=True, spikemode="across", spikethickness=1,
                         spikecolor=pal.muted, spikedash="solid")
    for ann in fig.layout.annotations or ():
        ann.font = dict(family=FONT, color=pal.ink2, size=13)
    return fig


def threshold(fig: go.Figure, pal: Palette, y: float, label: str, *, dash: str = "dash",
              row: int | str | None = None, col: int | str | None = None) -> None:
    """A horizontal limit line in neutral ink with its label at the right end."""
    kw = {} if row is None else dict(row=row, col=col)
    fig.add_hline(y=y, line=dict(color=pal.ink2, width=1, dash=dash),
                  annotation=dict(text=label, font=dict(color=pal.ink2, size=11),
                                  bgcolor=rgba(pal.surface, 0.7)),
                  annotation_position="top right", **kw)


PLOTLY_CONFIG = {"displaylogo": False, "modeBarButtonsToRemove": ["lasso2d", "select2d"]}


In [ ]:
%%writefile {LIB}/dashboard/uploads.py
"""Zip import/export of run folders, so results can be shared without touching results/.

A shared zip mirrors ``results/``: ``<scenario>/<run>/summary.json`` plus the run's CSVs,
and optionally the scenario's ``servers.csv`` / ``carbon.csv`` next to the run folders
(``<scenario>/servers.csv``). Import is forgiving about the layout: any folder with a
``summary.json`` is a run, wherever it sits in the zip (zipping a single run folder, a
scenario folder or the whole ``results/`` folder all work).

There is no Streamlit here so the import can be tested directly; the upload page keeps
imported runs in a per-session folder (see ``dashboard/views/upload.py``).
"""

from __future__ import annotations

import io
import json
import re
import shutil
import zipfile
from dataclasses import dataclass, field
from pathlib import Path, PurePosixPath

from dashboard.data import RunRef

# The only scenario inputs the dashboard reads (server names and the clock).
SCENARIO_FILES = ("servers.csv", "carbon.csv")
UPLOADED_SUFFIX = " (uploaded)"

# Guards against a zip that expands far beyond what a run folder needs.
MAX_MEMBERS = 20_000
MAX_UNCOMPRESSED_BYTES = 2 * 1024 ** 3


@dataclass
class ImportResult:
    runs: list[RunRef] = field(default_factory=list)
    scenarios: list[str] = field(default_factory=list)
    problems: list[str] = field(default_factory=list)


def _safe_name(name: str, fallback: str) -> str:
    """A single folder name: no separators, no leading dot, nothing empty."""
    name = re.sub(r"[\\/:\x00]", "_", str(name)).strip().lstrip(".").strip()
    return name or fallback


def _members(zf: zipfile.ZipFile) -> list[tuple[PurePosixPath, zipfile.ZipInfo]]:
    """Regular files with a safe relative path; macOS/Windows clutter is skipped."""
    out = []
    for info in zf.infolist():
        if info.is_dir():
            continue
        p = PurePosixPath(info.filename.replace("\\", "/"))
        if p.is_absolute() or ".." in p.parts or not p.parts:
            continue
        if p.parts[0] == "__MACOSX" or any(part.startswith(".") for part in p.parts):
            continue
        if p.name in ("Thumbs.db", "desktop.ini"):
            continue
        out.append((p, info))
    return out


def _owner(path: PurePosixPath, roots: list[PurePosixPath]) -> PurePosixPath | None:
    """The deepest run root that contains ``path``."""
    best = None
    for r in roots:
        if not r.parts or path.is_relative_to(r):
            if best is None or len(r.parts) > len(best.parts):
                best = r
    return best


def import_zip(source, results_dir: str | Path, scenarios_dir: str | Path, *,
               reserved: set[str] = frozenset(), fallback_name: str = "uploaded_run",
               fallback_scenarios_dir: str | Path | None = None) -> ImportResult:
    """Unpack every run in the zip into ``results_dir/<scenario>/<run>/``.

    ``reserved`` holds run keys that must not be replaced (runs already on disk); an
    upload with such a key is renamed ``<run> (uploaded)``. Any other run with the same
    key is replaced, so uploading a newer zip of a run updates it.

    Scenario files found in the zip go to ``scenarios_dir/<scenario>/``; files missing
    there are filled in from ``fallback_scenarios_dir`` when it has them.
    """
    results_dir, scenarios_dir = Path(results_dir), Path(scenarios_dir)
    result = ImportResult()
    try:
        zf = zipfile.ZipFile(source)
    except zipfile.BadZipFile:
        result.problems.append("This file is not a valid .zip archive.")
        return result

    with zf:
        members = _members(zf)
        if len(members) > MAX_MEMBERS or \
                sum(i.file_size for _, i in members) > MAX_UNCOMPRESSED_BYTES:
            result.problems.append("The zip is too large to be a set of run folders.")
            return result

        summaries = {p.parent: info for p, info in members if p.name == "summary.json"}
        roots = sorted(summaries, key=lambda r: len(r.parts))
        if not roots:
            result.problems.append(
                "No `summary.json` found. Zip the run folder itself (the folder that "
                "contains `summary.json`, `baseline/` and `optimized/`).")
            return result

        by_root: dict[PurePosixPath, list[tuple[PurePosixPath, zipfile.ZipInfo]]] = {}
        loose = []
        for p, info in members:
            owner = _owner(p, roots)
            if owner is None:
                loose.append((p, info))
            else:
                by_root.setdefault(owner, []).append((p, info))

        written: set[str] = set()
        run_scenarios: set[str] = set()
        for root in roots:
            try:
                summary = json.loads(zf.read(summaries[root]))
            except (KeyError, ValueError, UnicodeDecodeError):
                result.problems.append(f"`{root / 'summary.json'}` could not be read; "
                                       "that run was skipped.")
                continue
            sc = summary.get("scenario") if isinstance(summary, dict) else None
            sc_name = sc.get("name") if isinstance(sc, dict) else None
            parent = root.parent.name if len(root.parts) > 1 else ""
            scenario = _safe_name(sc_name or parent, "uploaded")
            name = _safe_name(root.name if root.parts else "", fallback_name)

            if f"{scenario}/{name}" in reserved:
                name += UPLOADED_SUFFIX
            base, n = name, 2
            while f"{scenario}/{name}" in written:      # two runs with one name in a zip
                name, n = f"{base} {n}", n + 1
            key = f"{scenario}/{name}"
            written.add(key)
            run_scenarios.add(scenario)

            dest = results_dir / scenario / name
            if dest.exists():
                shutil.rmtree(dest)
            for p, info in by_root[root]:
                _extract(zf, info, dest / (p.relative_to(root) if root.parts else p))
            result.runs.append(RunRef(scenario, name, dest))

        # Scenario inputs: servers.csv / carbon.csv outside the run folders.
        for p, info in loose:
            if p.name not in SCENARIO_FILES:
                continue
            folder = p.parent.name if len(p.parts) > 1 else ""
            if not folder and len(run_scenarios) == 1:
                folder = next(iter(run_scenarios))
            if not folder:
                continue
            scenario = _safe_name(folder, "uploaded")
            _extract(zf, info, scenarios_dir / scenario / p.name)
            if scenario not in result.scenarios:
                result.scenarios.append(scenario)

    if fallback_scenarios_dir is not None:
        for scenario in result.scenarios:
            for fname in SCENARIO_FILES:
                target = scenarios_dir / scenario / fname
                src = Path(fallback_scenarios_dir) / scenario / fname
                if not target.exists() and src.is_file():
                    shutil.copyfile(src, target)
    return result


def _extract(zf: zipfile.ZipFile, info: zipfile.ZipInfo, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    with zf.open(info) as src, open(target, "wb") as dst:
        shutil.copyfileobj(src, dst)


def runs_zip(runs: list[tuple[RunRef, Path]]) -> bytes:
    """Zip of ``(run, scenarios_dir)`` pairs laid out as ``<scenario>/<run>/...``.

    The scenario's servers.csv and carbon.csv are added as ``<scenario>/<file>`` so the
    zip shows server names and clock times on a machine without that scenario.
    """
    buf = io.BytesIO()
    added: set[str] = set()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
        for ref, scenarios_dir in runs:
            for f in sorted(ref.path.rglob("*")):
                if f.is_file() and not f.name.startswith("."):
                    rel = f.relative_to(ref.path).as_posix()
                    zf.write(f, f"{ref.scenario}/{ref.name}/{rel}")
            for fname in SCENARIO_FILES:
                arc = f"{ref.scenario}/{fname}"
                src = Path(scenarios_dir) / ref.scenario / fname
                if arc not in added and src.is_file():
                    zf.write(src, arc)
                    added.add(arc)
    return buf.getvalue()


## Run settings

The Settings cell's `RUN` becomes the run's `RunSettings`, and the imports for the steps below.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from dashboard.data import RunRef, load_clock
from dashboard.theme import PLOTLY_CONFIG, palette, style, threshold
from dashboard.uploads import runs_zip
from scheduler.baseline import build_baseline
from scheduler.build import derive_scenario
from scheduler.evaluate import evaluate
from scheduler.io import apply_overrides, load_reference_baseline, load_scenario
from scheduler.optimize import diagnose_infeasibility
from scheduler.report import METRICS, format_summary, write_results
from scheduler.run import default_run_name, format_checks, optimized_checks, solve_staged
from scheduler.types import STATUS_INFEASIBLE, RunSettings, constant_supply_temp, cop

settings = RunSettings(**RUN)
settings.validate()
settings.run_name = RUN_NAME or "nb_" + default_run_name(settings)
PAL = palette("light")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


def show(fig, title, height=360, **kw):
    """Dashboard styling, with the title on its own row above the legend."""
    style(fig, PAL, height=height, **kw)
    fig.update_layout(margin_t=fig.layout.margin.t + 28, title=dict(
        text=title, x=0, xanchor="left", xref="paper", y=1, yref="container", yanchor="top",
        pad=dict(t=8), font=dict(color=PAL.ink, size=15)))
    fig.show(config=PLOTLY_CONFIG)


print(f"Run name: {settings.run_name}")

## 1. Scenario

With `BUILD_FROM` set, the first cell builds the scenario and shows what changed against
the starting scenario, then the figures of the two side by side: the size of the problem,
how loaded the fleet is, how many servers the thermal cap limits, and the baseline's
energy and carbon. `baseline_unplaced_or_late` above 0 means the fleet cannot serve the
workloads in time even before optimizing. Then the scenario is loaded.

A scenario folder holds the servers, the workloads, the carbon intensity of each period and
the model constants (`params.json`). The zone map and carbon-intensity switches from the
Settings cell are applied here. One period is `dt_seconds` long; period `t` is shown at the
local time of its middle.

In [ ]:
if BUILD_FROM is None:
    print(f"Solving scenarios/{SCENARIO}/ as it is (BUILD_FROM = None).")
else:
    built = derive_scenario(SCENARIO, BUILD_FROM, SCENARIO_CHANGES, CONSTANT_CHANGES,
                            COLD_PLATE_CHANGES, scenarios_dir=SCENARIOS_DIR, clean_dir=CLEAN_DIR)
    blocks = {"settings": "settings", "constants": "constants_as_printed", "cold_plate": "cold_plate"}
    changes = pd.DataFrame([
        {"block": block, "name": k, BUILD_FROM: str(built["start"][blocks[block]][k]), SCENARIO: str(v)}
        for block, changed in built["changed"].items() for k, v in changed.items()])
    print(f"Built scenarios/{SCENARIO}/ from {BUILD_FROM}"
          + ("" if len(changes) else ", with nothing changed"))
    if (RESULTS_DIR / SCENARIO).is_dir():
        print(f"Note: runs already in results/{SCENARIO}/ were solved on the build before this one.")
    if len(changes):
        display(changes.set_index(["block", "name"]))
    display(pd.DataFrame({
        BUILD_FROM: {k: str(v) for k, v in built["start"]["diagnostics"].items()},
        SCENARIO: {k: str(v) for k, v in built["params"]["diagnostics"].items()},
    }))

In [ ]:
scenario = apply_overrides(load_scenario(settings.scenario, SCENARIOS_DIR), settings)
clock = load_clock(settings.scenario, scenario.T, scenario.dt_seconds, SCENARIOS_DIR)["time"]
c = scenario.constants

print(f"Scenario {scenario.name}: {scenario.I} workloads, {scenario.S} servers, {scenario.Z} zones, "
      f"{scenario.T} periods of {scenario.dt_seconds / 60:g} min "
      f"({clock.iloc[0]:%H:%M} to {clock.iloc[-1]:%H:%M})")
print(f"Thermal cap: T_max {c.T_max_server_C:g} - H_min {settings.h_min(scenario):g} = "
      f"{settings.thermal_cap_C(scenario):g} C; throttling starts at {c.T_dvfs_C:g} C")
print(f"Already running (B_i = 1): {sum(w.B for w in scenario.workloads)} workloads; "
      f"unavailable server-periods (AV = 0): {int((scenario.availability() == 0).sum())}")
print(f"Carbon intensity: mean {scenario.CI_g_per_kWh.mean():.1f} g/kWh, "
      f"range {scenario.CI_g_per_kWh.min():.1f}-{scenario.CI_g_per_kWh.max():.1f}")

In [ ]:
servers_df = pd.DataFrame([{
    "s": sv.s, "server": sv.name, "zone": sv.zone, "f (MHz)": sv.f_MHz, "cores": sv.C_cores,
    "memory (MB)": sv.M_MB, "P_idle (W)": sv.P_idle_W, "P_max (W)": sv.P_max_W, "T0 (C)": sv.T0_C,
    "a_s (C)": sv.th_a_C, "R_s (C/W)": sv.th_R_C_per_W, "alpha_s": sv.th_alpha,
    "cold plates": sv.plates,
} for sv in scenario.servers]).set_index("s")
servers_df

In [ ]:
workloads_df = pd.DataFrame([{
    "i": w.i, "workload": w.workload_id, "flavor": w.flavor,
    "type": "time-sensitive" if w.RT == 1 else "delay-tolerant",
    "arrival A": w.A, "deadline D": w.D, "allowed delay AD": w.AD, "ET_base": w.ET_base,
    "cores": w.C_req, "memory (MB)": w.M_req_MB, "work W (MHz*s)": w.W_MHz_s, "running B": w.B,
} for w in scenario.workloads]).set_index("i")

print("Workloads by type:")
print(workloads_df.groupby("type").agg(
    workloads=("workload", "size"), cores_mean=("cores", "mean"),
    ET_base_median=("ET_base", "median"), allowed_delay_median=("allowed delay AD", "median"),
).round(1).to_string())
workloads_df.head(10)

In [ ]:
fig = go.Figure(go.Scatter(x=clock, y=scenario.CI_g_per_kWh, mode="lines",
                           line=dict(color=PAL.ink2, width=2), name="Carbon intensity"))
fig.update_layout(yaxis_title="g CO2 per kWh", xaxis_title="Local time")
show(fig, "Grid carbon intensity", height=300, legend=False)

## 2. Baseline

The baseline places the workloads in arrival order, each at the earliest period where a
server has enough free cores for the whole run (`least_loaded` picks the server with the
fewest cores in use, `fastest_first` the fastest one). It checks only cores and availability:
not memory, temperature or deadlines. Every zone holds the fixed supply temperature
`baseline_supply_temp_C`.

The evaluator then recomputes, from the schedule alone, power, temperatures, throttling, CRAC
load, energy and carbon, and checks every limit. It never rejects a schedule: a broken limit
is reported, so the baseline can be invalid (for example at a warm supply temperature).

In [ ]:
checks = {}
base_sched = build_baseline(scenario, settings)
if settings.baseline_rule == "least_loaded":
    ref = load_reference_baseline(settings.scenario, SCENARIOS_DIR)
    checks["least_loaded_reproduces_baseline_schedule_csv"] = {"passed": base_sched == ref}
base_supply = constant_supply_temp(scenario, settings.baseline_supply_temp_C)
base_eval = evaluate(scenario, base_sched, base_supply, settings)
original = base_sched.server_of()

t = base_eval.totals
print(f"Carbon {t['carbon_kg']:.3f} kg; energy {t['E_total_kWh']:.2f} kWh "
      f"(IT {t['E_IT_kWh']:.2f}, CRAC {t['E_crac_kWh']:.2f}, cold plates {t['E_cp_kWh']:.4f})")
print(f"Hottest server {t['hottest_T_C']:.1f} C (server {t['hottest_server']}), "
      f"min headroom {t['min_headroom_C']:.1f} C, late {t['n_late']}, unplaced {t['n_unplaced']}")
print("Valid (no limit broken):", base_eval.valid)
for v in base_eval.compliance["violations"]:
    print("  -", v)

## 3. The optimized model

The MILP decides, for the whole horizon at once:

| Decision | Variable in `scheduler/optimize.py` |
|---|---|
| server of each workload | `X[i,s]`, one per allowed server |
| start period | `U[i,s,t]`, a 0/1 step "started by period t"; the start is where it turns on |
| extra run periods when throttled | `E[i,s,t]`, only where throttling can lengthen the run |
| CRAC supply temperature of each zone and period | one-hot choice among whole degrees, so the COP is a lookup |
| cold-plate flow setting and coolant temperature | one-hot flow choice and a continuous coolant temperature (with cold plates on) |

**Objective.** Minimise total carbon:
$\sum_t CI_t \cdot \Delta t \left(\sum_s P^{IT}_{st} + \sum_s P^{cp}_{st} + \sum_z Q^{crac}_{zt} / COP(T^{sup}_{zt})\right)$,
with the powers in kW and $\Delta t$ in hours. Fan power is a column fixed at zero.

**Constraints**, all hard. The equation numbers are those of `docs/PROJECT_CONTEXT.md`:

| Group | Equations | In the model |
|---|---|---|
| Assignment | (9), (10) | each workload runs once, on one allowed server, as one unbroken block |
| Reassignment | (29) | `none` and running workloads (B_i = 1) keep the baseline server; `same_zone` stays in its zone |
| Availability | (11) | no work on a server in a period with AV = 0 |
| Capacity | (12), (13) | cores and memory in use on each server in each period |
| IT power | (16), (20), (21) | $P^{IT} = P^{idle} + (P^{max} - P^{idle}) \cdot$ cores in use / cores |
| Timing | (23)-(28) | start at or after arrival, wait at most AD_i, finish by the deadline D_i |
| Temperature | recursion, (31)-(33) | $T_{st} = \alpha_s T_{s,t-1} + (1-\alpha_s)(T^{sup}_{zt} + a_s + R_s (P^{IT}_{st} - Q^{cp}_{st}))$, at most $T^{max} - H^{min}$ |
| Throttling and work | (37)-(39) | the linear part of the DPF slowdown; work delivered is at least W_i |
| CRAC | (34)-(36), (42), (54) | supply temperature in range and below the zone limit, ramp limits, heat removed at most the CRAC capacity |
| Cold plates | (47)-(49), (55) | flow and coolant limits, heat through the plates, at most 0.3 kW per plate |
| Priority | (56), (57) | on a shared server, a lower-priority workload may not start while a higher-priority one waits |

The constants of this scenario:

In [ ]:
cp = scenario.cold_plate
pd.DataFrame([
    ("T_max (server)", c.T_max_server_C, "C"), ("H_min (this run)", settings.h_min(scenario), "C"),
    ("T_dvfs (throttling starts)", c.T_dvfs_C, "C"), ("T_throttle", c.T_throttle_C, "C"),
    ("R_dvfs (slowdown at T_throttle)", c.R_dvfs, ""),
    ("CRAC supply range", f"{c.T_crac_min_C:g}-{c.T_crac_max_C:g}", "C"),
    ("Zone temperature limit", c.T_max_zone_C, "C"), ("CRAC capacity per zone", c.Q_crac_kW, "kW"),
    (f"Ramp limit over {c.ramp_short_lag} periods", c.ramp_short_C, "C"),
    (f"Ramp limit over {c.ramp_long_lag} periods", c.ramp_long_C, "C"),
    ("COP at 18 / 27 C", f"{cop(c, 18.0):.2f} / {cop(c, 27.0):.2f}", ""),
    ("Cold-plate flow settings", ", ".join(f"{f:g}" for f in cp.flow_settings_Lmin), "L/min"),
    ("Cold-plate coolant range", f"{cp.coolant_min_C:g}-{cp.coolant_max_C:g}", "C"),
    ("Heat per cold plate, max", cp.Q_max_per_plate_W, "W"),
], columns=["constant", "value", "unit"]).set_index("constant")

## 4. Solve

The solver gets a starting schedule, which gives it a first valid solution to improve on.
It does not change the model or its optimum. With `reassignment="free"` or `"same_zone"`, the
same model is first solved with every workload fixed to its baseline server, a much smaller
problem that uses 20% of the time limit. Its schedule is the start for the full model. With
`"none"` the baseline itself is the start.

Status `optimal` means the solver reached `mip_rel_gap`. `time_limit` means it stopped with a
valid schedule and a proven bound: the true optimum lies between the bound and the objective.
`infeasible` means no schedule meets every limit. `no_solution` means none was found in time,
which does not prove there is none.

In [ ]:
unplaced = [i for i, sv in original.items() if sv is None]
result = opt_eval = infeasibility = None
if unplaced:
    checks["optimized_run_skipped"] = {
        "reason": f"the baseline could not place workloads {unplaced}, so they have no "
                  "original server"}
    print(checks["optimized_run_skipped"]["reason"])
else:
    result = solve_staged(scenario, original, settings, base_sched, base_supply,
                          WARM_START, SOLVER_LOG)
    gap = "-" if result.mip_gap is None else f"{result.mip_gap:.4%}"
    print(f"Status {result.status} ({result.solver_status_text}), {result.solve_time_s:.1f} s, gap {gap}")
    if result.has_schedule:
        print(f"Objective {result.objective_g:,.2f} g, bound {result.best_bound_g:,.2f} g")
    print(f"Model: {result.n_variables:,} variables, {result.n_binaries:,} binaries, "
          f"{result.n_constraints:,} constraints")

How the equations were linearised and what this run switched on, as the solver reports it
(the same notes are saved in `summary.json`):

In [ ]:
for note in (result.notes if result else []):
    print("-", note)

### Infeasibility diagnosis

Runs only when the model is infeasible. The model is solved again with one group of limits
relaxed at a time (deadlines, thermal cap, capacity, cooling capacity, priority). A group whose
relaxation makes it `feasible` is one that causes the infeasibility.

In [ ]:
if result is not None and result.status == STATUS_INFEASIBLE:
    infeasibility = diagnose_infeasibility(scenario, original, settings,
                                           time_limit_s=DIAGNOSIS_TIME_LIMIT_S)
    display(pd.Series(infeasibility, name="with this group relaxed").to_frame())
else:
    print("Not needed: the model is not infeasible.")

## 5. Evaluate and check

The optimized schedule is scored by the same evaluator as the baseline, which shares no code
with the optimizer. The checks confirm that:

- the solver's objective equals the evaluator's carbon for the same schedule;
- the evaluator finds no broken limit in the optimized schedule;
- already-running workloads stayed on their server;
- with `reassignment="none"` at the baseline's fixed supply temperature, the optimized
  schedule is no worse than a valid baseline;
- the `least_loaded` baseline matches `baseline_schedule.csv` in the scenario folder.

The command line exits with an error when a check fails. Here, the last cell of the notebook
fails instead.

In [ ]:
if result is not None and result.has_schedule:
    opt_eval = evaluate(scenario, result.schedule, result.supply_temp_C, settings,
                        result.cold_plate_ops)
    checks.update(optimized_checks(scenario, settings, original, base_eval, result, opt_eval))
print(format_checks(checks))

## 6. Results

The comparison table is the one printed by the command line. Without `WRITE_RESULTS` the
files go to a temporary folder that is deleted at the end.

In [ ]:
out_dir = (RESULTS_DIR / settings.scenario / settings.run_name if WRITE_RESULTS
           else Path(tempfile.mkdtemp(prefix="thermasched_")))
summary = write_results(
    out_dir, scenario, settings, original, base_sched, base_eval,
    result.schedule if result else None, opt_eval, result, infeasibility, checks)
print(format_summary(summary))

In [ ]:
rows = []
for key, label, unit in METRICS:
    d = summary["comparison"][key]
    rows.append({"metric": f"{label} ({unit})" if unit else label, "baseline": d["baseline"],
                 "optimized": d["optimized"], "difference": d["diff"], "change %": d["pct"]})
comparison_df = pd.DataFrame(rows).set_index("metric")
comparison_df.style.format(precision=2, na_rep="-")

### Charts

Baseline in orange, optimized in blue, as in the dashboard. Hover over a chart to read the
values. The charts that need an optimized schedule are skipped when there is none.

In [ ]:
variants = [("Baseline", base_eval, PAL.baseline)]
if opt_eval is not None:
    variants.append(("Optimized", opt_eval, PAL.optimized))

fig = go.Figure()
for name, ev, color in variants:
    fig.add_trace(go.Scatter(x=clock, y=ev.carbon_g, mode="lines", name=name,
                             line=dict(color=color, width=2)))
fig.update_layout(yaxis_title="g CO2", xaxis_title="Local time")
show(fig, "Carbon emitted per period")

In [ ]:
fig = go.Figure()
for name, ev, color in variants:
    fig.add_trace(go.Scatter(x=clock, y=ev.T_C.max(axis=0), mode="lines", name=name,
                             line=dict(color=color, width=2)))
threshold(fig, PAL, settings.thermal_cap_C(scenario), "thermal cap")
threshold(fig, PAL, c.T_dvfs_C, "throttling starts", dash="dot")
fig.update_layout(yaxis_title="C", xaxis_title="Local time")
show(fig, "Hottest server temperature in each period")

In [ ]:
fig = make_subplots(rows=scenario.Z, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=[f"Zone {z}" for z in range(1, scenario.Z + 1)])
for z in range(scenario.Z):
    for name, ev, color in variants:
        fig.add_trace(go.Scatter(x=clock, y=ev.T_supply_C[z], mode="lines", name=name,
                                 line=dict(color=color, width=2, shape="hv"),
                                 legendgroup=name, showlegend=z == 0), row=z + 1, col=1)
fig.update_yaxes(title_text="C")
show(fig, "CRAC supply air temperature", height=220 + 180 * scenario.Z, titled=True)

In [ ]:
if opt_eval is None:
    print("No optimized schedule.")
else:
    moves = pd.DataFrame({
        "baseline server": [original[p.i] for p in result.schedule.placements],
        "optimized server": [p.server for p in result.schedule.placements]})
    n_moved = int((moves["baseline server"] != moves["optimized server"]).sum())
    print(f"{n_moved} of {scenario.I} workloads changed server. "
          "Rows: baseline server; columns: optimized server.")
    display(pd.crosstab(moves["baseline server"], moves["optimized server"]))

## 7. Where the results are

With `WRITE_RESULTS = True` the run is in `results/<scenario>/nb_<run name>/`, in the same
layout as a command-line run (see [`scheduler/README.md`](../scheduler/README.md#4-outputs)).
Open it in the dashboard with `uv run streamlit run dashboard/app.py` from the repository root.

A zip of the run is written to `exports/<scenario>__<run name>.zip` (and downloaded, on Colab).
It holds the run folder and the scenario's `servers.csv` and `carbon.csv`, so server names and
clock times show correctly in a dashboard that does not have the scenario. To view it:

- **For a while:** drop the zip on the dashboard's *Upload results* page. It stays until the
  browser tab is closed or reloaded, and only you see it.
- **For good, in the deployed dashboard:** unzip it into the repository's `results/` folder,
  commit and push. Streamlit Community Cloud redeploys the app with the run.

To reproduce this run from the command line, use the same switches, for example
`uv run python -m scheduler.run --scenario small`. A scenario built here is used the same
way, with `--scenario <SCENARIO>`. Section 5 of the solver
guide lists the switches of each thesis scenario. Each one is a field in the Settings cell:
`--zone-map 1,1,1,2,2,2` is `zone_map=(1, 1, 1, 2, 2, 2)`, `--no-priority` is `priority=False`,
and so on.

This cell fails if any check failed, as the command line does.

In [ ]:
if WRITE_RESULTS:
    print(f"Results written to {out_dir}")
    # The run and its scenario's servers.csv and carbon.csv, laid out like results/
    zip_path = EXPORTS_DIR / f"{settings.scenario}__{settings.run_name}.zip"
    EXPORTS_DIR.mkdir(parents=True, exist_ok=True)
    zip_path.write_bytes(runs_zip([(RunRef(settings.scenario, settings.run_name, out_dir),
                                    SCENARIOS_DIR)]))
    print(f"Zip for the dashboard: {zip_path}")
    if IN_COLAB and DOWNLOAD_ZIP:
        from google.colab import files
        files.download(str(zip_path))
else:
    shutil.rmtree(out_dir, ignore_errors=True)
    print("Nothing written (WRITE_RESULTS = False).")
failed = [name for name, ch in checks.items() if ch.get("passed") is False]
assert not failed, f"FAILED checks: {failed}"